# BetaTour v10 — sıkıştırılmış tur onarım pilotu

Bu pilot, v9'daki **aynı sekiz geliştirme örneğini** kullanır; bağımsız doğrulama değildir. Altı POPMUSIC-6/LKH süreci süre bütçesinin yaklaşık %35'inde, tamamlanan ilk uygun deneme sonunda beklerken BetaTour seçicisi veya modelsiz kontrol, o andaki en iyi turun üç ayrı noktasında yaklaşık 256'şar düğümlük pencereleri açar. Korunan tur parçaları zorunlu kenarlara sıkıştırılır; küçük altproblem LKH ile çözülür. Tam ve geçerli tur yalnızca daha kısaysa kabul edilir. Ana LKH süreçleri sonra kendi durumlarıyla müdahalesiz devam eder. Nihai hibrit sonuç, ana LKH ile dış onarımın kısa turudur.

**Çalıştırma:** Colab'da tercihen **A100 veya L4 GPU** ve **High RAM** seçin. T4 için eski tür engeli kaldırıldı, ancak 20k'deki bellek ve süre davranışı doğrulanmadı. Farklı GPU/CPU profillerinin sonuçlarını aynı karşılaştırmada karıştırmayın. Hücreleri sırayla çalıştırın. Tamamlanan her kol `MyDrive/BetaTour/v10_repair_pilot` içine kaydedilir. Oturum kesilirse hücreleri yeniden çalıştırın; tamamlanan kollar doğrulanıp atlanır. Bütün pilot bitince kodu ve özetleri içeren sonuç ZIP'i tarayıcıdan otomatik indirilir. Tam denetim ZIP'i Drive'da kalır.

**Kollar:** kesintisiz POPMUSIC-6; aynı kancalı onarımsız sham; rastgele, uzun-kenar, altı tur uyuşmazlığı ve mevcut TourFold puanıyla pencere seçimi. Tüm onarımlı kollar aynı küçük LKH motorunu ve aynı deneme sayısını kullanır. Bu sürümde TourFold henüz onarım faydasına göre yeniden eğitilmedi. Sekiz örneğin altı kol için salt süre sınırları toplamı yaklaşık **2,8 saat**; kurulum ve Drive aktarımı eklenir. Sonuç kötü gelirse mevcut makale taslağı değiştirilmeyecek.


In [ ]:
EXPERIMENT_NAME = "v10_repair_pilot"
SEEDS = list(range(301, 307))
ARMS = ["native_pop6", "sham", "repair_random", "repair_longest", "repair_disagreement", "repair_betatour"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 2, "uniform", 4107, 120.0],
    ["clu10k", 10000, 2, "clustered", 4208, 120.0],
    ["uni20k", 20000, 2, "uniform", 4309, 300.0],
    ["clu20k", 20000, 2, "clustered", 4410, 300.0],
]
EXPECTED_XY_SHA256 = {
    "augment_uni10k_000": "0d5d680b465faa68d33467c4f70bb5ad1e52b25a2d444066ab922130458efcd6",
    "augment_uni10k_001": "00747175507524f481bd6e34ea55f0a5474d3dcd642a67543d871d9a64b53796",
    "augment_clu10k_000": "072fc96ec668bb9c95b3a52562f23cedb0b2e89494946d159243b638b7b3dc8e",
    "augment_clu10k_001": "965fe4315060118d313a357404f750a23ea89cc11c174cb1138a9f32b312ca22",
    "augment_uni20k_000": "d0dbad4fdd4151029d3bf59cca81c19248129c87030003d7c455e37e32b7d674",
    "augment_uni20k_001": "c35e5c135b527d91c5c9b6258320f35526c2e735a50d581c48e96150e9cb3173",
    "augment_clu20k_000": "abdd6474a4cd0e25ee394c227ef8f5c4beb5c09c8a24abddcad273d0dc39181e",
    "augment_clu20k_001": "ee3edf42dbbb94cd344d1e3bef06d5674e47330e937a8acda4fd069f4faf79ff",
}
MAX_NEW_ARMS = None  # Örn. 3: bu oturumda en fazla üç yeni kol; sonra tekrar çalıştırın.
REPAIR_PARAMS = {"attempts_per_seed": 8, "cut_count": 3, "window_size": 256,
                 "sidecar_timeout_s": 2.0, "sidecar_trials": 100,
                 "repair_fraction_cap": 0.30, "barrier_fraction": 0.35}
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
LKH_TAR_SHA256 = "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33"
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"
assert len(SEEDS) == 6 and len(EXPECTED_XY_SHA256) == 8


In [ ]:
import base64, hashlib, importlib.util, json, math, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request, uuid, zipfile
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
for package in ("scipy", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import numpy as np
import scipy
from scipy.stats import binomtest
import pandas as pd
import torch
from IPython.display import display

assert torch.cuda.is_available(), "Colab GPU runtime seçin (tercihen A100 veya L4)."
torch.set_num_threads(1)
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
if not any(name in GPU_NAME for name in ("A100", "L4")):
    print("UYARI: GPU", GPU_NAME, "— pilot çalışır ama farklı profilleri karıştırmayın.")
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
if CPU_COUNT < 6:
    print("UYARI: Altı eşzamanlı LKH süreci için en az 6 vCPU önerilir; görünen:", CPU_COUNT)
def cpu_model_name():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.lower().startswith("model name"):
            return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

WORK_ROOT = Path("/content/betatour_v10_repair_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
assert hashlib.sha256((REPO / "checkpoints/betatour_v2.pt").read_bytes()).hexdigest() == CHECKPOINT_SHA256
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M
MODEL, _ = M.load_model(str(REPO / "checkpoints/betatour_v2.pt"), DEVICE)
assert MODEL.cfg.get("use_ensemble") is True

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
for sub in ("instances", "code", "summary", "errors", "archives"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
HARDWARE = {"gpu": GPU_NAME, "vcpus": CPU_COUNT, "cpu_model": cpu_model_name(),
            "python": platform.python_version(), "torch": torch.__version__}
print("Hazır:", HARDWARE, "Drive:", OUTPUT_ROOT)


Mounted at /content/drive
Hazır: {'gpu': 'NVIDIA A100-SXM4-80GB', 'vcpus': 12, 'cpu_model': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'python': '3.13.16', 'torch': '2.11.0+cu130'} Drive: /content/drive/MyDrive/BetaTour/v10_repair_pilot


In [ ]:
BUNDLED_FILES = {'livehook/BetaTourHook.h': 'LyogQmV0YVRvdXIgcmVzZWFyY2ggaG9vayBmb3IgTEtIIDMuMC4xNC4gIFRoaXMgZmlsZSBjb250YWlucyBvbmx5IG5ldyBjb2RlLiAqLwojaWZuZGVmIEJFVEFUT1VSX0hPT0tfSAojZGVmaW5lIEJFVEFUT1VSX0hPT0tfSAoKI2luY2x1ZGUgPGVycm5vLmg+CiNpbmNsdWRlIDxsaW1pdHMuaD4KI2luY2x1ZGUgPHN0ZGlvLmg+CiNpbmNsdWRlIDxzdGRsaWIuaD4KI2luY2x1ZGUgPHN0cmluZy5oPgojaW5jbHVkZSA8c3lzL3RpbWUuaD4KI2luY2x1ZGUgPHVuaXN0ZC5oPgoKc3RhdGljIGNvbnN0IGNoYXIgKkJUX0hvb2tEaXJlY3Rvcnkodm9pZCkKewogICAgY29uc3QgY2hhciAqUGF0aCA9IGdldGVudigiQkVUQVRPVVJfSE9PS19ESVIiKTsKICAgIHJldHVybiBSdW4gPT0gMSAmJiBQcm9ibGVtVHlwZSA9PSBUU1AgJiYgRGltZW5zaW9uID09IERpbWVuc2lvblNhdmVkICYmCiAgICAgICAgUGF0aCAmJiAqUGF0aCA/IFBhdGggOiAwOwp9CgpzdGF0aWMgdm9pZCBCVF9QYXRoKGNoYXIgKk91dCwgc2l6ZV90IFNpemUsIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwKICAgICAgICAgICAgICAgICAgICBjb25zdCBjaGFyICpOYW1lKQp7CiAgICBpZiAoc25wcmludGYoT3V0LCBTaXplLCAiJXMvJXMiLCBEaXJlY3RvcnksIE5hbWUpID49IChpbnQpIFNpemUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgcGF0aCBleGNlZWRzIFBBVEhfTUFYXG4iKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKc3RhdGljIGRvdWJsZSBCVF9XYWxsVGltZSh2b2lkKQp7CiAgICBzdHJ1Y3QgdGltZXZhbCBUdjsKICAgIGdldHRpbWVvZmRheSgmVHYsIDApOwogICAgcmV0dXJuIFR2LnR2X3NlYyArIFR2LnR2X3VzZWMgLyAxMDAwMDAwLjA7Cn0KCnN0YXRpYyBGSUxFICpCVF9PcGVuVGVtcG9yYXJ5KGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwgY29uc3QgY2hhciAqTmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hhciAqRmluYWwsIGNoYXIgKlRlbXBvcmFyeSkKewogICAgRklMRSAqRmlsZTsKICAgIEJUX1BhdGgoRmluYWwsIFBBVEhfTUFYLCBEaXJlY3RvcnksIE5hbWUpOwogICAgaWYgKHNucHJpbnRmKFRlbXBvcmFyeSwgUEFUSF9NQVgsICIlcy50bXAuJWxkIiwgRmluYWwsIChsb25nKSBnZXRwaWQoKSkKICAgICAgICA+PSBQQVRIX01BWCkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayB0ZW1wb3JhcnkgcGF0aCBleGNlZWRzIFBBVEhfTUFYXG4iKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIEZpbGUgPSBmb3BlbihUZW1wb3JhcnksICJ3Iik7CiAgICBpZiAoIUZpbGUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IHdyaXRlICVzOiAlc1xuIiwgVGVtcG9yYXJ5LAogICAgICAgICAgICAgICAgc3RyZXJyb3IoZXJybm8pKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIHJldHVybiBGaWxlOwp9CgpzdGF0aWMgdm9pZCBCVF9Db21taXQoRklMRSAqRmlsZSwgY29uc3QgY2hhciAqVGVtcG9yYXJ5LCBjb25zdCBjaGFyICpGaW5hbCkKewogICAgaWYgKGZjbG9zZShGaWxlKSAhPSAwIHx8IHJlbmFtZShUZW1wb3JhcnksIEZpbmFsKSAhPSAwKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIGNhbm5vdCBjb21taXQgJXM6ICVzXG4iLCBGaW5hbCwKICAgICAgICAgICAgICAgIHN0cmVycm9yKGVycm5vKSk7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9Cn0KCi8qIEVhY2ggcm93OiBub2RlIElELCBjb3VudCwgdGhlbiAobmVpZ2hib3IgSUQsIGFscGhhKSBwYWlycy4gKi8Kc3RhdGljIHZvaWQgQlRfV3JpdGVDYW5kaWRhdGVzKGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwgY29uc3QgY2hhciAqTmFtZSkKewogICAgY2hhciBGaW5hbFtQQVRIX01BWF0sIFRlbXBvcmFyeVtQQVRIX01BWF07CiAgICBGSUxFICpGaWxlID0gQlRfT3BlblRlbXBvcmFyeShEaXJlY3RvcnksIE5hbWUsIEZpbmFsLCBUZW1wb3JhcnkpOwogICAgaW50IGk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKSB7CiAgICAgICAgQ2FuZGlkYXRlICpDYW5kaWRhdGVQdHI7CiAgICAgICAgaW50IENvdW50ID0gMDsKICAgICAgICBOb2RlICpGcm9tID0gJk5vZGVTZXRbaV07CiAgICAgICAgZm9yIChDYW5kaWRhdGVQdHIgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICAgICAgICAgICBDYW5kaWRhdGVQdHIgJiYgQ2FuZGlkYXRlUHRyLT5UbzsgQ2FuZGlkYXRlUHRyKyspCiAgICAgICAgICAgIENvdW50Kys7CiAgICAgICAgZnByaW50ZihGaWxlLCAiJWQgJWQiLCBpLCBDb3VudCk7CiAgICAgICAgZm9yIChDYW5kaWRhdGVQdHIgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICAgICAgICAgICBDYW5kaWRhdGVQdHIgJiYgQ2FuZGlkYXRlUHRyLT5UbzsgQ2FuZGlkYXRlUHRyKyspCiAgICAgICAgICAgIGZwcmludGYoRmlsZSwgIiAlZCAlZCIsIENhbmRpZGF0ZVB0ci0+VG8tPklkLAogICAgICAgICAgICAgICAgICAgIENhbmRpZGF0ZVB0ci0+QWxwaGEpOwogICAgICAgIGZwdXRjKCdcbicsIEZpbGUpOwogICAgfQogICAgQlRfQ29tbWl0KEZpbGUsIFRlbXBvcmFyeSwgRmluYWwpOwp9CgpzdGF0aWMgdm9pZCBCVF9Xcml0ZUZpcnN0VG91cihjb25zdCBjaGFyICpEaXJlY3RvcnkpCnsKICAgIGNoYXIgRmluYWxbUEFUSF9NQVhdLCBUZW1wb3JhcnlbUEFUSF9NQVhdOwogICAgRklMRSAqRmlsZSA9IEJUX09wZW5UZW1wb3JhcnkoRGlyZWN0b3J5LCAiZmlyc3QudG91ciIsIEZpbmFsLCBUZW1wb3JhcnkpOwogICAgaW50IGk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKLyogQW4gYXRvbWljIGluY3VtYmVudCBzdXJ2aXZlcyBhbiBleHRlcm5hbCB3YWxsLWNsb2NrIGRlYWRsaW5lL3Rlcm1pbmF0aW9uLiAqLwpzdGF0aWMgdm9pZCBCVF9SZWNvcmRCZXN0KHZvaWQpCnsKICAgIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSA9IEJUX0hvb2tEaXJlY3RvcnkoKTsKICAgIGNoYXIgRmluYWxbUEFUSF9NQVhdLCBUZW1wb3JhcnlbUEFUSF9NQVhdLCBFdmVudHNbUEFUSF9NQVhdOwogICAgRklMRSAqRmlsZTsKICAgIGludCBpOwogICAgaWYgKCFEaXJlY3RvcnkpCiAgICAgICAgcmV0dXJuOwogICAgRmlsZSA9IEJUX09wZW5UZW1wb3JhcnkoRGlyZWN0b3J5LCAiYmVzdC50b3VyIiwgRmluYWwsIFRlbXBvcmFyeSk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKICAgIEJUX1BhdGgoRXZlbnRzLCBzaXplb2YoRXZlbnRzKSwgRGlyZWN0b3J5LCAiYmVzdC5ldmVudHMiKTsKICAgIEZpbGUgPSBmb3BlbihFdmVudHMsICJhIik7CiAgICBpZiAoIUZpbGUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IGFwcGVuZCAlczogJXNcbiIsIEV2ZW50cywKICAgICAgICAgICAgICAgIHN0cmVycm9yKGVycm5vKSk7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmcHJpbnRmKEZpbGUsICIlLjZmICVsbGRcbiIsIEJUX1dhbGxUaW1lKCksIChsb25nIGxvbmcpIEJldHRlckNvc3QpOwogICAgaWYgKGZjbG9zZShGaWxlKSAhPSAwKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIGNhbm5vdCBmbHVzaCAlc1xuIiwgRXZlbnRzKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKc3RhdGljIGludCBCVF9BcHBseU9uZShpbnQgRnJvbUlkLCBpbnQgVG9JZCwgaW50IFNsb3QsIGludCBBbHBoYSwKICAgICAgICAgICAgICAgICAgICAgICBpbnQgKldhc0V4aXN0aW5nLCBpbnQgKldhc01vdmVkKQp7CiAgICBOb2RlICpGcm9tLCAqVG87CiAgICBDYW5kaWRhdGUgKlN0YXJ0LCAqQ2hvc2VuLCBUZW1wOwogICAgaW50IENvdW50ID0gMCwgSW5kZXgsIERlc3RpbmF0aW9uOwoKICAgIGlmIChGcm9tSWQgPCAxIHx8IEZyb21JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPCAxIHx8CiAgICAgICAgVG9JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPT0gRnJvbUlkIHx8IFNsb3QgPCAwIHx8CiAgICAgICAgQWxwaGEgPCAwIHx8IEFscGhhID09IElOVF9NQVgpCiAgICAgICAgcmV0dXJuIC0xOwogICAgRnJvbSA9ICZOb2RlU2V0W0Zyb21JZF07CiAgICBUbyA9ICZOb2RlU2V0W1RvSWRdOwogICAgU3RhcnQgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICBmb3IgKENob3NlbiA9IFN0YXJ0OyBDaG9zZW4gJiYgQ2hvc2VuLT5UbzsgQ2hvc2VuKyspCiAgICAgICAgaWYgKENob3Nlbi0+VG8gPT0gVG8pCiAgICAgICAgICAgICpXYXNFeGlzdGluZyA9IDE7CiAgICAvKiBBIHJhbmstb25seSBhY3Rpb24gbXVzdCByZXRhaW4gdGhlIG5hdGl2ZSBlZGdlJ3MgYWxwaGEgdmFsdWUuICovCiAgICBpZiAoISpXYXNFeGlzdGluZyAmJiAhQWRkQ2FuZGlkYXRlKEZyb20sIFRvLCBDKEZyb20sIFRvKSwgQWxwaGEpKQogICAgICAgIHJldHVybiAtMTsKICAgIFN0YXJ0ID0gRnJvbS0+Q2FuZGlkYXRlU2V0OwogICAgZm9yIChDaG9zZW4gPSBTdGFydDsgQ2hvc2VuICYmIENob3Nlbi0+VG87IENob3NlbisrKQogICAgICAgIENvdW50Kys7CiAgICBmb3IgKEluZGV4ID0gMDsgSW5kZXggPCBDb3VudDsgSW5kZXgrKykKICAgICAgICBpZiAoU3RhcnRbSW5kZXhdLlRvID09IFRvKQogICAgICAgICAgICBicmVhazsKICAgIGlmIChJbmRleCA9PSBDb3VudCkKICAgICAgICByZXR1cm4gLTE7CiAgICBpZiAoU2xvdCA+IDApIHsKICAgICAgICBEZXN0aW5hdGlvbiA9IFNsb3QgLSAxOwogICAgICAgIGlmIChEZXN0aW5hdGlvbiA+PSBDb3VudCkKICAgICAgICAgICAgRGVzdGluYXRpb24gPSBDb3VudCAtIDE7CiAgICAgICAgaWYgKEluZGV4ICE9IERlc3RpbmF0aW9uKSB7CiAgICAgICAgICAgIGludCBqOwogICAgICAgICAgICBUZW1wID0gU3RhcnRbSW5kZXhdOwogICAgICAgICAgICBpZiAoSW5kZXggPiBEZXN0aW5hdGlvbikKICAgICAgICAgICAgICAgIGZvciAoaiA9IEluZGV4OyBqID4gRGVzdGluYXRpb247IGotLSkKICAgICAgICAgICAgICAgICAgICBTdGFydFtqXSA9IFN0YXJ0W2ogLSAxXTsKICAgICAgICAgICAgZWxzZQogICAgICAgICAgICAgICAgZm9yIChqID0gSW5kZXg7IGogPCBEZXN0aW5hdGlvbjsgaisrKQogICAgICAgICAgICAgICAgICAgIFN0YXJ0W2pdID0gU3RhcnRbaiArIDFdOwogICAgICAgICAgICBTdGFydFtEZXN0aW5hdGlvbl0gPSBUZW1wOwogICAgICAgICAgICAqV2FzTW92ZWQgPSAxOwogICAgICAgIH0KICAgIH0KICAgIHJldHVybiAwOwp9CgpzdGF0aWMgdm9pZCBCVF9XYWl0QW5kQXBwbHkoY29uc3QgY2hhciAqRGlyZWN0b3J5KQp7CiAgICBjaGFyIEFjdGlvbltQQVRIX01BWF0sIEZpbmFsW1BBVEhfTUFYXSwgVGVtcG9yYXJ5W1BBVEhfTUFYXTsKICAgIEZJTEUgKkZpbGU7CiAgICBpbnQgTiwgaSwgRnJvbSwgVG8sIFNsb3QsIEFscGhhOwogICAgaW50IEFkZGVkID0gMCwgRXhpc3RpbmcgPSAwLCBSZWplY3RlZCA9IDAsIE1vdmVkID0gMDsKICAgIGRvdWJsZSBTdGFydCA9IEJUX1dhbGxUaW1lKCksIFRpbWVvdXQgPSAzMDAuMDsKICAgIGNvbnN0IGNoYXIgKlRpbWVvdXRUZXh0ID0gZ2V0ZW52KCJCRVRBVE9VUl9IT09LX1RJTUVPVVRfUyIpOwogICAgaWYgKFRpbWVvdXRUZXh0ICYmIGF0b2YoVGltZW91dFRleHQpID4gMCkKICAgICAgICBUaW1lb3V0ID0gYXRvZihUaW1lb3V0VGV4dCk7CiAgICBCVF9QYXRoKEFjdGlvbiwgc2l6ZW9mKEFjdGlvbiksIERpcmVjdG9yeSwgImFjdGlvbi50eHQiKTsKICAgIHdoaWxlICghKEZpbGUgPSBmb3BlbihBY3Rpb24sICJyIikpKSB7CiAgICAgICAgaWYgKEJUX1dhbGxUaW1lKCkgLSBTdGFydCA+IFRpbWVvdXQpIHsKICAgICAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIHRpbWVkIG91dCB3YWl0aW5nIGZvciAlc1xuIiwgQWN0aW9uKTsKICAgICAgICAgICAgZXhpdCg5Nyk7CiAgICAgICAgfQogICAgICAgIHVzbGVlcCgxMDAwMCk7CiAgICB9CiAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCIsICZOKSAhPSAxIHx8IE4gIT0gRGltZW5zaW9uU2F2ZWQpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgYWN0aW9uIGRpbWVuc2lvbiBtaXNtYXRjaFxuIik7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmb3IgKGkgPSAxOyBpIDw9IE47IGkrKykgewogICAgICAgIGludCBXYXNFeGlzdGluZyA9IDAsIFdhc01vdmVkID0gMDsKICAgICAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCAlZCAlZCAlZCIsICZGcm9tLCAmVG8sICZTbG90LCAmQWxwaGEpICE9IDQgfHwKICAgICAgICAgICAgRnJvbSAhPSBpKSB7CiAgICAgICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayBtYWxmb3JtZWQgYWN0aW9uIHJvdyAlZFxuIiwgaSk7CiAgICAgICAgICAgIGV4aXQoOTcpOwogICAgICAgIH0KICAgICAgICBpZiAoVG8gPT0gMCkKICAgICAgICAgICAgY29udGludWU7CiAgICAgICAgaWYgKEJUX0FwcGx5T25lKEZyb20sIFRvLCBTbG90LCBBbHBoYSwgJldhc0V4aXN0aW5nLAogICAgICAgICAgICAgICAgICAgICAgICAmV2FzTW92ZWQpICE9IDApCiAgICAgICAgICAgIFJlamVjdGVkKys7CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIEFkZGVkICs9ICFXYXNFeGlzdGluZzsKICAgICAgICAgICAgRXhpc3RpbmcgKz0gV2FzRXhpc3Rpbmc7CiAgICAgICAgICAgIE1vdmVkICs9IFdhc01vdmVkOwogICAgICAgIH0KICAgIH0KICAgIGZjbG9zZShGaWxlKTsKICAgIGlmIChSZWplY3RlZCkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayByZWplY3RlZCAlZCBjYW5kaWRhdGUgYWN0aW9uc1xuIiwKICAgICAgICAgICAgICAgIFJlamVjdGVkKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIEJUX1dyaXRlQ2FuZGlkYXRlcyhEaXJlY3RvcnksICJwb3N0LmNhbmRpZGF0ZXMiKTsKICAgIEZpbGUgPSBCVF9PcGVuVGVtcG9yYXJ5KERpcmVjdG9yeSwgImFwcGxpZWQuc3RhdHMiLCBGaW5hbCwgVGVtcG9yYXJ5KTsKICAgIGZwcmludGYoRmlsZSwgImFkZGVkICVkXG5leGlzdGluZyAlZFxubW92ZWQgJWRcbnJlamVjdGVkICVkXG53YWl0X3dhbGxfcyAlLjZmXG4iLAogICAgICAgICAgICBBZGRlZCwgRXhpc3RpbmcsIE1vdmVkLCBSZWplY3RlZCwgQlRfV2FsbFRpbWUoKSAtIFN0YXJ0KTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKc3RhdGljIHZvaWQgQlRfRXhwb3J0TmF0aXZlQ2FuZGlkYXRlcyh2b2lkKQp7CiAgICBjb25zdCBjaGFyICpEaXJlY3RvcnkgPSBCVF9Ib29rRGlyZWN0b3J5KCk7CiAgICBpZiAoRGlyZWN0b3J5KQogICAgICAgIEJUX1dyaXRlQ2FuZGlkYXRlcyhEaXJlY3RvcnksICJuYXRpdmUuY2FuZGlkYXRlcyIpOwp9CgpzdGF0aWMgdm9pZCBCVF9BZnRlckZpcnN0VHJpYWwodm9pZCkKewogICAgY29uc3QgY2hhciAqRGlyZWN0b3J5ID0gQlRfSG9va0RpcmVjdG9yeSgpOwogICAgaWYgKERpcmVjdG9yeSAmJiBUcmlhbCA9PSAxKSB7CiAgICAgICAgQlRfV3JpdGVDYW5kaWRhdGVzKERpcmVjdG9yeSwgImxpdmUuY2FuZGlkYXRlcyIpOwogICAgICAgIEJUX1dyaXRlRmlyc3RUb3VyKERpcmVjdG9yeSk7IC8qIHJlYWRpbmVzcyBtYXJrZXI7IHB1Ymxpc2hlZCBsYXN0ICovCiAgICAgICAgQlRfV2FpdEFuZEFwcGx5KERpcmVjdG9yeSk7CiAgICB9Cn0KCiNlbmRpZgo=', 'livehook/build_livehook.py': 'IiIiQnVpbGQgYSBzdGF0ZS1wcmVzZXJ2aW5nIEJldGFUb3VyIGhvb2sgZnJvbSBhbiB1bnBhY2tlZCBMS0gtMy4wLjE0IHRyZWUuCgpUaGUgTEtIIHNvdXJjZSBpcyBmZXRjaGVkIHNlcGFyYXRlbHkgYnkgdGhlIGV4cGVyaW1lbnQgbm90ZWJvb2suIFRoaXMgc2NyaXB0CnNoaXBzIG9ubHkgdGhlIEJldGFUb3VyIHBhdGNoIGFuZCBjaGVja3MgdGhlIHVwc3RyZWFtIGZpbGUgYmVmb3JlIGVkaXRpbmcgaXQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKVVBTVFJFQU1fRklORFRPVVJfU0hBMjU2ID0gKAogICAgIjU2N2U3MmFjYzliNzM5MjQ4OGRkMTU4YmU0ZGYwMjY3NTYwNDI0MjA0YTBlODhkMmQ5YjYyZjkyM2E3YzBlZWUiCikKCgpkZWYgX3JlcGxhY2Vfb25jZShzb3VyY2U6IHN0ciwgb2xkOiBzdHIsIG5ldzogc3RyKSAtPiBzdHI6CiAgICBjb3VudCA9IHNvdXJjZS5jb3VudChvbGQpCiAgICBpZiBjb3VudCAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBvbmUgRmluZFRvdXIgYW5jaG9yLCBmb3VuZCB7Y291bnR9OiB7b2xkWzo4MF0hcn0iKQogICAgcmV0dXJuIHNvdXJjZS5yZXBsYWNlKG9sZCwgbmV3LCAxKQoKCmRlZiBwYXRjaF9maW5kdG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbiNpbmNsdWRlICJCZXRhVG91ckhvb2suaCJcbicsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgIGRvdWJsZSBFbnRyeVRpbWUgPSBHZXRUaW1lKCk7XG5cbiAgICB0ID0gRmlyc3ROb2RlOyIsCiAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4iCiAgICAgICAgIiAgICBCVF9FeHBvcnROYXRpdmVDYW5kaWRhdGVzKCk7XG4gICAgdCA9IEZpcnN0Tm9kZTsiLAogICAgKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiIsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiAgICAgICAgICAgIEJUX1JlY29yZEJlc3QoKTtcbiIsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgICAgICB9XG4gICAgfVxuICAgIGlmIChCYWNrYm9uZVRyaWFscyA+IDAgJiYgQmFja2JvbmVUcmlhbHMgPCBNYXhUcmlhbHMpIHsiLAogICAgICAgICIgICAgICAgIH1cbiAgICAgICAgQlRfQWZ0ZXJGaXJzdFRyaWFsKCk7XG4iCiAgICAgICAgIiAgICB9XG4gICAgaWYgKEJhY2tib25lVHJpYWxzID4gMCAmJiBCYWNrYm9uZVRyaWFscyA8IE1heFRyaWFscykgeyIsCiAgICApCiAgICByZXR1cm4gc291cmNlCgoKZGVmIGJ1aWxkKHNvdXJjZV9kaXI6IFBhdGgsIG91dHB1dF9kaXI6IFBhdGgsIGpvYnM6IGludCA9IDQpIC0+IFBhdGg6CiAgICBzb3VyY2VfZGlyID0gc291cmNlX2Rpci5yZXNvbHZlKCkKICAgIG91dHB1dF9kaXIgPSBvdXRwdXRfZGlyLnJlc29sdmUoKQogICAgdXBzdHJlYW0gPSBzb3VyY2VfZGlyIC8gIlNSQyIgLyAiRmluZFRvdXIuYyIKICAgIGlmIG5vdCB1cHN0cmVhbS5leGlzdHMoKSBvciBub3QgKHNvdXJjZV9kaXIgLyAiTWFrZWZpbGUiKS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkxLSC0zLjAuMTQgc291cmNlIHRyZWUgbWlzc2luZzoge3NvdXJjZV9kaXJ9IikKICAgIG9yaWdpbmFsX2J5dGVzID0gdXBzdHJlYW0ucmVhZF9ieXRlcygpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihvcmlnaW5hbF9ieXRlcykuaGV4ZGlnZXN0KCkKICAgIGlmIGRpZ2VzdCAhPSBVUFNUUkVBTV9GSU5EVE9VUl9TSEEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgTEtIIEZpbmRUb3VyLmMgU0hBMjU2OiB7ZGlnZXN0fSIpCiAgICBpZiBvdXRwdXRfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSB7b3V0cHV0X2Rpcn0iKQogICAgc2h1dGlsLmNvcHl0cmVlKAogICAgICAgIHNvdXJjZV9kaXIsCiAgICAgICAgb3V0cHV0X2RpciwKICAgICAgICBpZ25vcmU9c2h1dGlsLmlnbm9yZV9wYXR0ZXJucygiT0JKIiwgIkxLSCIsICIqLm8iKSwKICAgICkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiT0JKIikubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGNoZWQgPSBwYXRjaF9maW5kdG91cihvcmlnaW5hbF9ieXRlcy5kZWNvZGUoInV0Zi04IikpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIkZpbmRUb3VyLmMiKS53cml0ZV90ZXh0KHBhdGNoZWQpCiAgICBob29rID0gUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJCZXRhVG91ckhvb2suaCIpCiAgICBzaHV0aWwuY29weTIoaG9vaywgb3V0cHV0X2RpciAvICJTUkMiIC8gIkJldGFUb3VySG9vay5oIikKICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgIFsibWFrZSIsICItaiIsIHN0cihqb2JzKV0sCiAgICAgICAgY3dkPW91dHB1dF9kaXIsCiAgICAgICAgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwKICAgICAgICBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgdGV4dD1UcnVlLAogICAgICAgIGNoZWNrPUZhbHNlLAogICAgKQogICAgaWYgcmVzdWx0LnJldHVybmNvZGU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTEtIIGhvb2sgYnVpbGQgZmFpbGVkOlxue3Jlc3VsdC5zdGRvdXRbLTYwMDA6XX0iKQogICAgYmluYXJ5ID0gb3V0cHV0X2RpciAvICJMS0giCiAgICBpZiBub3QgYmluYXJ5LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUGF0Y2hlZCBiaW5hcnkgbWlzc2luZzoge2JpbmFyeX0iKQogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgInVwc3RyZWFtX2ZpbmR0b3VyX3NoYTI1NiI6IGRpZ2VzdCwKICAgICAgICAicGF0Y2hfaGVhZGVyX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGhvb2sucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGJpbmFyeS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgfQogICAgKG91dHB1dF9kaXIgLyAiYmV0YXRvdXJfYnVpbGQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIpICsgIlxuIgogICAgKQogICAgcmV0dXJuIGJpbmFyeQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWpvYnMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHByaW50KGJ1aWxkKGFyZ3Muc291cmNlX2RpciwgYXJncy5vdXRwdXRfZGlyLCBhcmdzLmpvYnMpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK', 'livehook/coordinator.py': 'IiIiV2FsbC1jbG9jayBjb29yZGluYXRvciBmb3Igc2l4IHN0YXRlLXByZXNlcnZpbmcgQmV0YVRvdXIvTEtIIHByb2Nlc3Nlcy4KCk9ubHkgb25lIHByb2JsZW0gaW5zdGFuY2Ugc2hvdWxkIHVzZSBhIGdpdmVuIHdvcmsgZGlyZWN0b3J5LiBUaGUgYWN0aW9uCmNhbGxiYWNrIHJlY2VpdmVzIHRoZSBzaXggZmlyc3QgdG91cnMgYW5kIG5hdGl2ZS9saXZlIGNhbmRpZGF0ZSBzbmFwc2hvdHMuCkl0IHJldHVybnMgb25lIHBlci1ub2RlIGFjdGlvbiBsaXN0IHBlciBzZWVkOyBlYWNoIGFjdGlvbiBpcwpgYChuZWlnaGJvcl9pZF8xX2Jhc2VkX29yXzAsIHNsb3RfMV9iYXNlZF9vcl8wLCBhbHBoYSlgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIFNlcXVlbmNlCgoKQGRhdGFjbGFzcwpjbGFzcyBTbmFwc2hvdDoKICAgIHNlZWQ6IGludAogICAgZmlyc3RfdG91cjogbGlzdFtpbnRdCiAgICBuYXRpdmVfY2FuZGlkYXRlczogbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dCiAgICBsaXZlX2NhbmRpZGF0ZXM6IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludF1dXQoKCmRlZiByZWFkX3RvdXIocGF0aDogUGF0aCkgLT4gbGlzdFtpbnRdOgogICAgd29yZHMgPSBbaW50KHgpIGZvciB4IGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXQoKV0KICAgIGlmIG5vdCB3b3JkcyBvciBsZW4od29yZHMpICE9IHdvcmRzWzBdICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWFsZm9ybWVkIHRvdXI6IHtwYXRofSIpCiAgICBuID0gd29yZHNbMF0KICAgIHRvdXIgPSB3b3Jkc1sxOl0KICAgIGlmIHNvcnRlZCh0b3VyKSAhPSBsaXN0KHJhbmdlKDEsIG4gKyAxKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRvdXIgaXMgbm90IGEgcGVybXV0YXRpb246IHtwYXRofSIpCiAgICByZXR1cm4gdG91cgoKCmRlZiByZWFkX2NhbmRpZGF0ZXMocGF0aDogUGF0aCkgLT4gbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dOgogICAgbGluZXMgPSBwYXRoLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKQogICAgaWYgbm90IGxpbmVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbXB0eSBjYW5kaWRhdGUgc25hcHNob3Q6IHtwYXRofSIpCiAgICBuID0gaW50KGxpbmVzWzBdKQogICAgaWYgbGVuKGxpbmVzKSAhPSBuICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2FuZGlkYXRlIHNuYXBzaG90IGhhcyB7bGVuKGxpbmVzKS0xfSByb3dzLCBuZWVkIHtufSIpCiAgICByb3dzOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnRdXV0gPSBbXQogICAgZm9yIGV4cGVjdGVkX2lkLCBsaW5lIGluIGVudW1lcmF0ZShsaW5lc1sxOl0sIDEpOgogICAgICAgIG51bXMgPSBbaW50KHgpIGZvciB4IGluIGxpbmUuc3BsaXQoKV0KICAgICAgICBpZiBsZW4obnVtcykgPCAyIG9yIG51bXNbMF0gIT0gZXhwZWN0ZWRfaWQgb3IgbGVuKG51bXMpICE9IDIgKyAyICogbnVtc1sxXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1hbGZvcm1lZCBjYW5kaWRhdGUgcm93IHtleHBlY3RlZF9pZH06IHtwYXRofSIpCiAgICAgICAgcm93ID0gWyhudW1zW2ldLCBudW1zW2kgKyAxXSkgZm9yIGkgaW4gcmFuZ2UoMiwgbGVuKG51bXMpLCAyKV0KICAgICAgICBpZiBhbnkodiA8IDEgb3IgdiA+IG4gb3IgdiA9PSBleHBlY3RlZF9pZCBmb3IgdiwgXyBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBuZWlnaGJvciBpbiByb3cge2V4cGVjdGVkX2lkfToge3BhdGh9IikKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcm93cwoKCmRlZiBub29wX2FjdGlvbnMobjogaW50KSAtPiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXToKICAgIHJldHVybiBbKDAsIDAsIDUwMCldICogbgoKCmRlZiB3cml0ZV9hY3Rpb25zKHBhdGg6IFBhdGgsIGFjdGlvbnM6IFNlcXVlbmNlW3R1cGxlW2ludCwgaW50LCBpbnRdXSkgLT4gTm9uZToKICAgIG4gPSBsZW4oYWN0aW9ucykKICAgIGxpbmVzID0gW3N0cihuKV0KICAgIGZvciBzb3VyY2VfaWQsIChuZWlnaGJvciwgc2xvdCwgYWxwaGEpIGluIGVudW1lcmF0ZShhY3Rpb25zLCAxKToKICAgICAgICBpZiBub3QgKDAgPD0gbmVpZ2hib3IgPD0gbiBhbmQgMCA8PSBzbG90IDw9IG4gKyAyIGFuZCAwIDw9IGFscGhhIDwgMioqMzEgLSAxKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgYWN0aW9uIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgaWYgbmVpZ2hib3IgPT0gc291cmNlX2lkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU2VsZi1lZGdlIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYie3NvdXJjZV9pZH0ge25laWdoYm9yfSB7c2xvdH0ge2FscGhhfSIpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX3BhcmFtZXRlcl9maWxlKHBhdGg6IFBhdGgsIHByb2JsZW06IFBhdGgsIHNlZWQ6IGludCwgd2FsbF9idWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgICAgbWF4X3RyaWFsczogaW50LCBleHRyYTogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSkgLT4gTm9uZToKICAgIHBhcmFtczogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSA9IHsKICAgICAgICAiUFJPQkxFTV9GSUxFIjogc3RyKHByb2JsZW0ucmVzb2x2ZSgpKSwKICAgICAgICAiQ0FORElEQVRFX1NFVF9UWVBFIjogIlBPUE1VU0lDIiwKICAgICAgICAiTUFYX0NBTkRJREFURVMiOiA1LAogICAgICAgICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgIlNFRUQiOiBzZWVkLAogICAgICAgICJSVU5TIjogMSwKICAgICAgICAiTUFYX1RSSUFMUyI6IG1heF90cmlhbHMsCiAgICAgICAgIyBMS0gncyBUSU1FX0xJTUlUIG1lYXN1cmVzIHByb2Nlc3MgQ1BVIHRpbWUuIFRoZSBjb29yZGluYXRvciBiZWxvdwogICAgICAgICMgZW5mb3JjZXMgdGhlIGFjdHVhbCBlbmQtdG8tZW5kIHdhbGwgYnVkZ2V0LgogICAgICAgICJUSU1FX0xJTUlUIjogbWF4KDEwLjAsIDEwLjAgKiB3YWxsX2J1ZGdldF9zKSwKICAgICAgICAiVFJBQ0VfTEVWRUwiOiAwLAogICAgICAgICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgfQogICAgcGFyYW1zLnVwZGF0ZShleHRyYSkKICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oZiJ7a2V5fSA9IHt2YWx1ZX0iIGZvciBrZXksIHZhbHVlIGluIHBhcmFtcy5pdGVtcygpKSArICJcbiIpCgoKZGVmIHJ1bl9zaXgoCiAgICBsa2hfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHdvcmtfZGlyOiBQYXRoLAogICAgc2VlZHM6IFNlcXVlbmNlW2ludF0sCiAgICB3YWxsX2J1ZGdldF9zOiBmbG9hdCwKICAgIGFjdGlvbl9mbjogQ2FsbGFibGVbW2xpc3RbU25hcHNob3RdXSwgU2VxdWVuY2VbU2VxdWVuY2VbdHVwbGVbaW50LCBpbnQsIGludF1dXV0sCiAgICAqLAogICAgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICBleHRyYV9wYXJhbXM6IGRpY3Rbc3RyLCBzdHIgfCBpbnQgfCBmbG9hdF0gfCBOb25lID0gTm9uZSwKICAgIGJhcnJpZXJfbmFtZTogc3RyID0gImZpcnN0LnRvdXIiLAogICAgaG9va19hZnRlcl9zOiBmbG9hdCA9IDAuMCwKKSAtPiBkaWN0OgogICAgIiIiUnVuIG9uZSBpbnN0YW5jZSB3aXRoIHNpeCBzZWVkczsgY2hhcmdlIHN0YXJ0dXAsIGJhcnJpZXIgYW5kIHBvbGljeSB0aW1lLgoKICAgIGB3b3JrX2RpcmAgbXVzdCBiZSBuZXcgdG8gYXZvaWQgc3RhbGUgYWN0aW9uIGZpbGVzLiBSdW5zIG1heSBlbmQgYmVmb3JlIHRoZQogICAgZGVhZGxpbmUgd2hlbiBgbWF4X3RyaWFsc2AgaXMgc21hbGwuIFJlc3VsdHMgaW5jbHVkZSBhbGwgaG9vayBmaWxlcyBzbyB0aGUKICAgIGNhbGxlciBjYW4gYXVkaXQgdGhlIGFjdHVhbCBjYW5kaWRhdGUgYXJyYXlzIGFuZCBmaXJzdCB0b3Vycy4KICAgICIiIgogICAgaWYgbGVuKHNlZWRzKSAhPSA2IG9yIGxlbihzZXQoc2VlZHMpKSAhPSA2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0bHkgc2l4IGRpc3RpbmN0IHNlZWRzIGFyZSByZXF1aXJlZCIpCiAgICBpZiB3YWxsX2J1ZGdldF9zIDw9IDAgb3IgbWF4X3RyaWFscyA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9zaXRpdmUgYnVkZ2V0IGFuZCB0cmlhbHMgYXJlIHJlcXVpcmVkIikKICAgIGlmIGJhcnJpZXJfbmFtZSBub3QgaW4gKCJmaXJzdC50b3VyIiwgImJhcnJpZXIudG91ciIpIG9yIGhvb2tfYWZ0ZXJfcyA8IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBiYXJyaWVyIG1hcmtlciBvciBkZWxheSIpCiAgICBpZiB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJVc2UgYSBmcmVzaCBydW4gZGlyZWN0b3J5OiB7d29ya19kaXJ9IikKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHByb2Nlc3NlczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdCiAgICBvcGVuX2xvZ3M6IGxpc3QgPSBbXQogICAgc3RhcnRfbW9ubyA9IHRpbWUubW9ub3RvbmljKCkKICAgIHN0YXJ0X2Vwb2NoID0gdGltZS50aW1lKCkKICAgIGRlYWRsaW5lID0gc3RhcnRfbW9ubyArIHdhbGxfYnVkZ2V0X3MKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgc2VlZF9kaXIgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgICAgIHNlZWRfZGlyLm1rZGlyKCkKICAgICAgICAgICAgcGFyYW1zID0gc2VlZF9kaXIgLyAicnVuLnBhciIKICAgICAgICAgICAgX3BhcmFtZXRlcl9maWxlKHBhcmFtcywgcHJvYmxlbSwgc2VlZCwgd2FsbF9idWRnZXRfcywgbWF4X3RyaWFscywKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV4dHJhX3BhcmFtcyBvciB7fSkKICAgICAgICAgICAgc3Rkb3V0ID0gKHNlZWRfZGlyIC8gInN0ZG91dC50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgc3RkZXJyID0gKHNlZWRfZGlyIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgb3Blbl9sb2dzLmV4dGVuZCgoc3Rkb3V0LCBzdGRlcnIpKQogICAgICAgICAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24pCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19ESVIiXSA9IHN0cihzZWVkX2Rpci5yZXNvbHZlKCkpCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19USU1FT1VUX1MiXSA9IHN0cihtYXgoNS4wLCB3YWxsX2J1ZGdldF9zICsgNS4wKSkKICAgICAgICAgICAgZW52WyJCRVRBVE9VUl9IT09LX0FGVEVSX1MiXSA9IHN0cihob29rX2FmdGVyX3MpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3BlbigKICAgICAgICAgICAgICAgIFtzdHIobGtoX2JpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyYW1zLnJlc29sdmUoKSldLAogICAgICAgICAgICAgICAgY3dkPXNlZWRfZGlyLCBzdGRvdXQ9c3Rkb3V0LCBzdGRlcnI9c3RkZXJyLCBlbnY9ZW52LAogICAgICAgICAgICApKQogICAgICAgIGZpcnN0X3BhdGhzID0gW3dvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyBiYXJyaWVyX25hbWUgZm9yIHNlZWQgaW4gc2VlZHNdCiAgICAgICAgd2hpbGUgbm90IGFsbChwYXRoLmV4aXN0cygpIGZvciBwYXRoIGluIGZpcnN0X3BhdGhzKToKICAgICAgICAgICAgZmFpbHVyZXMgPSBbcC5yZXR1cm5jb2RlIGZvciBwIGluIHByb2Nlc3NlcyBpZiBwLnBvbGwoKSBub3QgaW4gKE5vbmUsIDApXQogICAgICAgICAgICBpZiBmYWlsdXJlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkxLSCBleGl0ZWQgYmVmb3JlIGZpcnN0LXRvdXIgYmFycmllcjoge2ZhaWx1cmVzfSIpCiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICB0aW1lLnNsZWVwKDAuMDIpCiAgICAgICAgc25hcHNob3RzOiBsaXN0W1NuYXBzaG90XSA9IFtdCiAgICAgICAgaWYgYWxsKHBhdGguZXhpc3RzKCkgZm9yIHBhdGggaW4gZmlyc3RfcGF0aHMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgICAgIHNuYXBzaG90cy5hcHBlbmQoU25hcHNob3QoCiAgICAgICAgICAgICAgICAgICAgc2VlZD1zZWVkLAogICAgICAgICAgICAgICAgICAgIGZpcnN0X3RvdXI9cmVhZF90b3VyKGJhc2UgLyBiYXJyaWVyX25hbWUpLAogICAgICAgICAgICAgICAgICAgIG5hdGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gIm5hdGl2ZS5jYW5kaWRhdGVzIiksCiAgICAgICAgICAgICAgICAgICAgbGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLAogICAgICAgICAgICAgICAgKSkKICAgICAgICAgICAgYWN0aW9ucyA9IGFjdGlvbl9mbihzbmFwc2hvdHMpCiAgICAgICAgICAgIGlmIGxlbihhY3Rpb25zKSAhPSBsZW4oc2VlZHMpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9saWN5IG11c3QgcmV0dXJuIG9uZSBhY3Rpb24gbWF0cml4IHBlciBzZWVkIikKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgZm9yIHNlZWQsIHNuYXBzaG90LCBhY3Rpb24gaW4gemlwKHNlZWRzLCBzbmFwc2hvdHMsIGFjdGlvbnMpOgogICAgICAgICAgICAgICAgICAgIGlmIGxlbihhY3Rpb24pICE9IGxlbihzbmFwc2hvdC5maXJzdF90b3VyKToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBvbGljeSBkaW1lbnNpb24gbWlzbWF0Y2ggZm9yIHNlZWQge3NlZWR9IikKICAgICAgICAgICAgICAgICAgICB3cml0ZV9hY3Rpb25zKHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyAiYWN0aW9uLnR4dCIsIGFjdGlvbikKICAgICAgICB3aGlsZSBhbnkocC5wb2xsKCkgaXMgTm9uZSBmb3IgcCBpbiBwcm9jZXNzZXMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4wMikKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgcHJvY2Vzcy50ZXJtaW5hdGUoKQogICAgICAgIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlczoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9MikKICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgICAgICAgICBwcm9jZXNzLmtpbGwoKQogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KCkKICAgICAgICBmb3Igc3RyZWFtIGluIG9wZW5fbG9nczoKICAgICAgICAgICAgc3RyZWFtLmNsb3NlKCkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic2VlZHMiOiBsaXN0KHNlZWRzKSwKICAgICAgICAid2FsbF9idWRnZXRfcyI6IHdhbGxfYnVkZ2V0X3MsCiAgICAgICAgIndhbGxfZWxhcHNlZF9zIjogdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0X21vbm8sCiAgICAgICAgInN0YXJ0X2Vwb2NoX3MiOiBzdGFydF9lcG9jaCwKICAgICAgICAiZGVhZGxpbmVfZXBvY2hfcyI6IHN0YXJ0X2Vwb2NoICsgd2FsbF9idWRnZXRfcywKICAgICAgICAiYmFycmllcl9jb21wbGV0ZWQiOiBsZW4oc25hcHNob3RzKSA9PSA2LAogICAgICAgICJydW5zIjogW10sCiAgICB9CiAgICBmb3Igc2VlZCwgcHJvY2VzcyBpbiB6aXAoc2VlZHMsIHByb2Nlc3Nlcyk6CiAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICBiZXN0ID0gYmFzZSAvICJiZXN0LnRvdXIiCiAgICAgICAgcmVzdWx0WyJydW5zIl0uYXBwZW5kKHsKICAgICAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAicmV0dXJuY29kZSI6IHByb2Nlc3MucmV0dXJuY29kZSwKICAgICAgICAgICAgImZpcnN0X3RvdXIiOiBzdHIoYmFzZSAvIGJhcnJpZXJfbmFtZSkgaWYgKGJhc2UgLyBiYXJyaWVyX25hbWUpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfdG91ciI6IHN0cihiZXN0KSBpZiBiZXN0LmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfZXZlbnRzIjogc3RyKGJhc2UgLyAiYmVzdC5ldmVudHMiKSBpZiAoYmFzZSAvICJiZXN0LmV2ZW50cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgIm5hdGl2ZV9jYW5kaWRhdGVzIjogc3RyKGJhc2UgLyAibmF0aXZlLmNhbmRpZGF0ZXMiKSBpZiAoYmFzZSAvICJuYXRpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImxpdmVfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgInBvc3RfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImFwcGxpZWRfc3RhdHMiOiBzdHIoYmFzZSAvICJhcHBsaWVkLnN0YXRzIikgaWYgKGJhc2UgLyAiYXBwbGllZC5zdGF0cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICB9KQogICAgKHdvcmtfZGlyIC8gInJ1bl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAiXG4iKQogICAgcmV0dXJuIHJlc3VsdAoK', 'policy/adaptive_policy.py': 'IiIiQ29uc2VydmF0aXZlIFBPUE1VU0lDIGNhbmRpZGF0ZSBhY3Rpb25zIGZyb20gYW4gZXhpc3RpbmcgVG91ckZvbGQgcHJlZGljdGlvbi4KClRoZSBmdW5jdGlvbnMgaW4gdGhpcyBmaWxlIGRvIG5vdCBjbGFpbSB0byBlc3RpbWF0ZSBzZWFyY2ggdXRpbGl0eS4gVGhleSB0dXJuClRvdXJGb2xkJ3MgcmVmZXJlbmNlLWVkZ2Ugc2NvcmVzIGludG8gKnRlc3RhYmxlKiBpbnRlcnZlbnRpb25zIGZvciBhIGxpdmUgTEtICmV4cGVyaW1lbnQuIE5vIGFjdGlvbiBpcyB0YWtlbiB3aGVuIGFuIGVkZ2UgY2Fubm90IGJlIHNjb3JlZCByZWxpYWJseS4KCklucHV0cyB1c2UgemVyby1iYXNlZCBub2RlIElEcy4gYGBwb3BgYCBoYXMgc2hhcGUgKHNlZWRzLCBub2Rlcywga19wb3ApLCB3aXRoCmBgLTFgYCBmb3IgYW4gdW5maWxsZWQgbmF0aXZlIFBPUE1VU0lDIHNsb3QuIGBgbmVpZ2hib3Vyc2BgL2BgcHJvYmFiaWxpdGllc2BgCmhhdmUgc2hhcGUgKG5vZGVzLCBrX2dyYXBoKS4gVGhlIHNhbWUgbW9kZWwgcHJlZGljdGlvbiBtYXkgYmUgYXBwbGllZCB0byBlYWNoCnNlZWQncyBkaWZmZXJlbnQgUE9QTVVTSUMgY2FuZGlkYXRlIHNldC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFNwYXJzZUFkZENvbmZpZzoKICAgICIiIkEgcHJlY29tbWl0dGVkIGRlbnNpdHkgY2FwIGFuZCBldmlkZW5jZSBydWxlIGZvciBvbmUgZXh0cmEgZWRnZSBwZXIgbm9kZS4KCiAgICBgYG1heF9mcmFjdGlvbmBgIGxpbWl0cyBpbnRlcnZlbnRpb24gdG8gdGhlIGJlc3Qtc2NvcmVkIGZyYWN0aW9uIG9mIG5vZGVzCiAgICBpbiAqZWFjaCBzZWVkKi4gYGBtaW5fc2NvcmVkX3BvcGBgIHJlcXVpcmVzIGEgbWluaW11bSBudW1iZXIgb2YgdGhlIFBPUAogICAgZWRnZXMgdG8gYmUgcHJlc2VudCBpbiBUb3VyRm9sZCdzIHNjb3JpbmcgZ3JhcGguIGBgbWluX21hcmdpbmBgIGlzIHRoZQogICAgcHJvYmFiaWxpdHkgZGlmZmVyZW5jZSBiZXR3ZWVuIHRoZSBwcm9wb3NlZCBlZGdlIGFuZCB0aGUgd2Vha2VzdCBzY29yZWQKICAgIFBPUCBlZGdlOyB0aGVzZSBwcm9iYWJpbGl0aWVzIGFyZSBub3QgYXNzdW1lZCBjYWxpYnJhdGVkLgogICAgIiIiCgogICAgbWF4X2ZyYWN0aW9uOiBmbG9hdCA9IDAuMjUKICAgIG1pbl9zY29yZWRfcG9wOiBpbnQgPSAyCiAgICBtaW5fbWFyZ2luOiBmbG9hdCA9IDAuMAoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IDAuMCA8PSBzZWxmLm1heF9mcmFjdGlvbiA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF9mcmFjdGlvbiBtdXN0IGJlIGluIFswLCAxXSIpCiAgICAgICAgaWYgc2VsZi5taW5fc2NvcmVkX3BvcCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1pbl9zY29yZWRfcG9wIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzZWxmLm1pbl9tYXJnaW4pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fbWFyZ2luIG11c3QgYmUgZmluaXRlIikKCgpAZGF0YWNsYXNzCmNsYXNzIFNwYXJzZUFkZFJlc3VsdDoKICAgICIiIk9uZSBwZXItbm9kZSBvcHRpb25hbCBhY3Rpb24gZm9yIGV2ZXJ5IGluZGVwZW5kZW50IExLSCBwcm9jZXNzLgoKICAgIGBgZGVzdGluYXRpb24gPT0gLTFgYCBtZWFucyBrZWVwIHRoZSBuYXRpdmUgUE9QTVVTSUMgY2FuZGlkYXRlIHJvdy4KICAgIGBgcHJvcG9zZWRfZGVzdGluYXRpb25gYCByZWNvcmRzIGEgcG9zc2libGUgZ3JhcGgtbm92ZWwgZWRnZSBldmVuIHdoZW4gdGhlCiAgICBnYXRlIGRlY2xpbmVzIGl0LiBgYHNjb3JlYGAgaXMgYSBwcm94eSwgbm90IGFuIG9ic2VydmVkIExLSCBnYWluLgogICAgIiIiCgogICAgZGVzdGluYXRpb246IG5wLm5kYXJyYXkKICAgIHByb3Bvc2VkX2Rlc3RpbmF0aW9uOiBucC5uZGFycmF5CiAgICBwcm9wb3NlZF9wcm9iYWJpbGl0eTogbnAubmRhcnJheQogICAgc2NvcmU6IG5wLm5kYXJyYXkKICAgIHNjb3JlZF9wb3BfY291bnQ6IG5wLm5kYXJyYXkKICAgIGVsaWdpYmxlOiBucC5uZGFycmF5CgoKZGVmIF92YWxpZGF0ZShwb3A6IG5wLm5kYXJyYXksIG5laWdoYm91cnM6IG5wLm5kYXJyYXksIHByb2JhYmlsaXRpZXM6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgcG9wID0gbnAuYXNhcnJheShwb3ApCiAgICBuZWlnaGJvdXJzID0gbnAuYXNhcnJheShuZWlnaGJvdXJzKQogICAgcHJvYmFiaWxpdGllcyA9IG5wLmFzYXJyYXkocHJvYmFiaWxpdGllcywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIHBvcC5uZGltICE9IDMgb3IgbmVpZ2hib3Vycy5uZGltICE9IDIgb3IgcHJvYmFiaWxpdGllcy5zaGFwZSAhPSBuZWlnaGJvdXJzLnNoYXBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV4cGVjdGVkIHBvcD0oc2VlZHMsbm9kZXMsayksIG5laWdoYm91cnM9cHJvYmFiaWxpdGllcz0obm9kZXMsa19ncmFwaCkiKQogICAgc2VlZHMsIG4sIGtfcG9wID0gcG9wLnNoYXBlCiAgICBpZiBzZWVkcyA8IDEgb3IgbiA8IDMgb3Iga19wb3AgPCAxIG9yIG5laWdoYm91cnMuc2hhcGVbMF0gIT0gbiBvciBuZWlnaGJvdXJzLnNoYXBlWzFdIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGFycmF5IGRpbWVuc2lvbnMiKQogICAgaWYga19wb3AgPj0gbiBvciBuZWlnaGJvdXJzLnNoYXBlWzFdID49IG46CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2FuZGlkYXRlIHdpZHRoIG11c3QgYmUgYmVsb3cgbm9kZSBjb3VudCIpCiAgICBpZiBub3QgbnAuaXNzdWJkdHlwZShwb3AuZHR5cGUsIG5wLmludGVnZXIpIG9yIG5vdCBucC5pc3N1YmR0eXBlKG5laWdoYm91cnMuZHR5cGUsIG5wLmludGVnZXIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNhbmRpZGF0ZSBJRHMgbXVzdCBiZSBpbnRlZ2VycyIpCiAgICBpZiBucC5hbnkoKHBvcCA8IC0xKSB8IChwb3AgPj0gbikpIG9yIG5wLmFueSgobmVpZ2hib3VycyA8IDApIHwgKG5laWdoYm91cnMgPj0gbikpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNhbmRpZGF0ZSBJRHMgb3V0c2lkZSBpbnN0YW5jZSIpCiAgICBvcmlnaW4gPSBucC5hcmFuZ2UobikKICAgIGlmIG5wLmFueShwb3AgPT0gb3JpZ2luW05vbmUsIDosIE5vbmVdKSBvciBucC5hbnkobmVpZ2hib3VycyA9PSBvcmlnaW5bOiwgTm9uZV0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNlbGYtZWRnZSBpbiBjYW5kaWRhdGVzIikKICAgICMgTXVsdGlwbGUgLTEgc2xvdHMgYXJlIGxlZ2l0aW1hdGUuIFRyZWF0IHRoZW0gYXMgYSBzZW50aW5lbCBhYm92ZSBhbGwKICAgICMgbm9kZSBJRHMgb25seSBmb3IgZHVwbGljYXRlIGRldGVjdGlvbjsgbmV2ZXIgdHVybiB0aGVtIGludG8gY2FuZGlkYXRlcy4KICAgIHNvcnRlZF9wb3AgPSBucC5zb3J0KG5wLndoZXJlKHBvcCA+PSAwLCBwb3AsIG4pLCBheGlzPTIpCiAgICBpZiBucC5hbnkoKHNvcnRlZF9wb3BbOiwgOiwgMTpdID09IHNvcnRlZF9wb3BbOiwgOiwgOi0xXSkKICAgICAgICAgICAgICAmIChzb3J0ZWRfcG9wWzosIDosIDE6XSAhPSBuKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZHVwbGljYXRlIFBPUCBjYW5kaWRhdGUgaW4gYSByb3ciKQogICAgaWYgbnAuYW55KG5wLnNvcnQobmVpZ2hib3VycywgYXhpcz0xKVs6LCAxOl0gPT0gbnAuc29ydChuZWlnaGJvdXJzLCBheGlzPTEpWzosIDotMV0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImR1cGxpY2F0ZSBtb2RlbCBncmFwaCBuZWlnaGJvdXIgaW4gYSByb3ciKQogICAgaWYgbm90IG5wLmFsbChucC5pc2Zpbml0ZShwcm9iYWJpbGl0aWVzKSkgb3IgbnAuYW55KChwcm9iYWJpbGl0aWVzIDwgMCkgfCAocHJvYmFiaWxpdGllcyA+IDEpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9iYWJpbGl0aWVzIG11c3QgYmUgZmluaXRlIGFuZCBpbiBbMCwgMV0iKQogICAgcmV0dXJuIHBvcC5hc3R5cGUobnAuaW50MzIsIGNvcHk9RmFsc2UpLCBuZWlnaGJvdXJzLmFzdHlwZShucC5pbnQzMiwgY29weT1GYWxzZSksIHByb2JhYmlsaXRpZXMKCgpkZWYgX3BvcF9zY29yZXMocG9wX3JvdzogbnAubmRhcnJheSwgbmVpZ2hib3VyczogbnAubmRhcnJheSwgcHJvYmFiaWxpdGllczogbnAubmRhcnJheSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICBtYXRjaGVzID0gcG9wX3Jvd1s6LCA6LCBOb25lXSA9PSBuZWlnaGJvdXJzWzosIE5vbmUsIDpdCiAgICBwcmVzZW50ID0gbWF0Y2hlcy5hbnkoYXhpcz0yKQogICAgc2NvcmVzID0gbnAud2hlcmUocHJlc2VudCwgKG1hdGNoZXMgKiBwcm9iYWJpbGl0aWVzWzosIE5vbmUsIDpdKS5zdW0oYXhpcz0yKSwgbnAubmFuKQogICAgcmV0dXJuIHNjb3JlcywgcHJlc2VudAoKCmRlZiByZXJhbmtfc2NvcmVkX3BvcChwb3A6IG5wLm5kYXJyYXksIG5laWdoYm91cnM6IG5wLm5kYXJyYXksIHByb2JhYmlsaXRpZXM6IG5wLm5kYXJyYXksCiAgICAgICAgICAgICAgICAgICAgICBtaW5fbWFyZ2luOiBmbG9hdCA9IDAuMCkgLT4gbnAubmRhcnJheToKICAgICIiIlJlb3JkZXIgb25seSBtb2RlbC12aXNpYmxlIFBPUCBlZGdlcywga2VlcGluZyB1bnNlZW4gcG9zaXRpb25zIGZpeGVkLgoKICAgIENhbmRpZGF0ZSBtZW1iZXJzaGlwIGFuZCBlYWNoIHJvdydzIG9yaWdpbmFsIHVuc2Vlbi1lZGdlIHBvc2l0aW9ucyByZW1haW4KICAgIGV4YWN0bHkgdW5jaGFuZ2VkLiBUaGUgY2FsbGVyIGNhbiBjb21wYXJlIHRoaXMgYXJtIHdpdGggYSBzaGFtIGxpdmUgaG9vay4KICAgICIiIgogICAgcG9wLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzID0gX3ZhbGlkYXRlKHBvcCwgbmVpZ2hib3VycywgcHJvYmFiaWxpdGllcykKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShtaW5fbWFyZ2luKSBvciBtaW5fbWFyZ2luIDwgMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fbWFyZ2luIG11c3QgYmUgZmluaXRlIGFuZCBub25uZWdhdGl2ZSIpCiAgICBvdXQgPSBwb3AuY29weSgpCiAgICBmb3Igc2VlZCBpbiByYW5nZShsZW4ocG9wKSk6CiAgICAgICAgc2NvcmVzLCBwcmVzZW50ID0gX3BvcF9zY29yZXMocG9wW3NlZWRdLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzKQogICAgICAgIGZvciBub2RlIGluIHJhbmdlKGxlbihuZWlnaGJvdXJzKSk6CiAgICAgICAgICAgIHNsb3RzID0gbnAuZmxhdG5vbnplcm8ocHJlc2VudFtub2RlXSkKICAgICAgICAgICAgaWYgbGVuKHNsb3RzKSA8IDIgb3IgbnAucHRwKHNjb3Jlc1tub2RlLCBzbG90c10pIDw9IG1pbl9tYXJnaW46CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBvcmRlciA9IG5wLmFyZ3NvcnQoLXNjb3Jlc1tub2RlLCBzbG90c10sIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgICAgIG91dFtzZWVkLCBub2RlLCBzbG90c10gPSBwb3Bbc2VlZCwgbm9kZSwgc2xvdHNbb3JkZXJdXQogICAgcmV0dXJuIG91dAoKCmRlZiBfZWRnZV9rZXlzKG46IGludCwgZGVzdGluYXRpb246IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICBvcmlnaW4gPSBucC5hcmFuZ2UobiwgZHR5cGU9bnAuaW50NjQpWzosIE5vbmVdCiAgICBkZXN0ID0gbnAuYXNhcnJheShkZXN0aW5hdGlvbiwgZHR5cGU9bnAuaW50NjQpCiAgICBrZXlzID0gbnAubWluaW11bShvcmlnaW4sIGRlc3QpICogbnAuaW50NjQobikgKyBucC5tYXhpbXVtKG9yaWdpbiwgZGVzdCkKICAgIHJldHVybiBucC53aGVyZShkZXN0ID49IDAsIGtleXMsIC0xKQoKCmRlZiBwcm9wb3NlX3NwYXJzZV9hZGQocG9wOiBucC5uZGFycmF5LCBuZWlnaGJvdXJzOiBucC5uZGFycmF5LCBwcm9iYWJpbGl0aWVzOiBucC5uZGFycmF5LAogICAgICAgICAgICAgICAgICAgICAgIGNvbmZpZzogU3BhcnNlQWRkQ29uZmlnID0gU3BhcnNlQWRkQ29uZmlnKCkpIC0+IFNwYXJzZUFkZFJlc3VsdDoKICAgICIiIlNlbGVjdCBhdCBtb3N0IG9uZSBnbG9iYWxseSBQT1Atbm92ZWwgZWRnZSBmb3IgYSBzdWJzZXQgb2Ygbm9kZXMuCgogICAgVGhlIG5vLW9wIG9wdGlvbiBpcyBleHBsaWNpdC4gTm92ZWx0eSBpcyBjaGVja2VkIGFnYWluc3QgdGhlIGZ1bGwKICAgICp1bmRpcmVjdGVkKiBQT1AgY2FuZGlkYXRlIGdyYXBoIGZvciBlYWNoIHNlZWQsIG1hdGNoaW5nIHRoZSB2OCBwcm90b2NvbC4KICAgIFNlbGVjdGlvbiBpcyBkZXRlcm1pbmlzdGljOiBwcm9iYWJpbGl0eSByYW5rIGJyZWFrcyB0aWVzIGJ5IG9yaWdpbmFsCiAgICBncmFwaCBvcmRlcjsgbm9kZSBJRCBicmVha3MgZXF1YWwgcHJveHkgc2NvcmVzLgogICAgIiIiCiAgICBwb3AsIG5laWdoYm91cnMsIHByb2JhYmlsaXRpZXMgPSBfdmFsaWRhdGUocG9wLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzKQogICAgc2VlZHMsIG4sIGtfcG9wID0gcG9wLnNoYXBlCiAgICBpZiBjb25maWcubWluX3Njb3JlZF9wb3AgPiBrX3BvcDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fc2NvcmVkX3BvcCBleGNlZWRzIFBPUCB3aWR0aCIpCiAgICBkZXN0aW5hdGlvbiA9IG5wLmZ1bGwoKHNlZWRzLCBuKSwgLTEsIGR0eXBlPW5wLmludDMyKQogICAgcHJvcG9zZWQgPSBucC5mdWxsKChzZWVkcywgbiksIC0xLCBkdHlwZT1ucC5pbnQzMikKICAgIHBpY2tlZF9wcm9iID0gbnAuZnVsbCgoc2VlZHMsIG4pLCBucC5uYW4sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBzY29yZSA9IG5wLmZ1bGwoKHNlZWRzLCBuKSwgbnAubmFuLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY292ZXJlZCA9IG5wLnplcm9zKChzZWVkcywgbiksIGR0eXBlPW5wLmludDE2KQogICAgZWxpZ2libGUgPSBucC56ZXJvcygoc2VlZHMsIG4pLCBkdHlwZT1ib29sKQogICAgcmFua2VkX2dyYXBoID0gbnAuYXJnc29ydCgtcHJvYmFiaWxpdGllcywgYXhpcz0xLCBraW5kPSJzdGFibGUiKQogICAgZ3JhcGhfa2V5cyA9IF9lZGdlX2tleXMobiwgbmVpZ2hib3VycykKICAgIGxpbWl0ID0gaW50KG5wLmZsb29yKGNvbmZpZy5tYXhfZnJhY3Rpb24gKiBuKSkKCiAgICBmb3Igc2VlZCBpbiByYW5nZShzZWVkcyk6CiAgICAgICAgcG9wX2tleXMgPSBucC51bmlxdWUoX2VkZ2Vfa2V5cyhuLCBwb3Bbc2VlZF0pKQogICAgICAgIHBvcF9rZXlzID0gcG9wX2tleXNbcG9wX2tleXMgPj0gMF0KICAgICAgICBub3ZlbCA9IH5ucC5pc2luKGdyYXBoX2tleXMsIHBvcF9rZXlzKQogICAgICAgIHJhbmtlZF9ub3ZlbCA9IG5wLnRha2VfYWxvbmdfYXhpcyhub3ZlbCwgcmFua2VkX2dyYXBoLCBheGlzPTEpCiAgICAgICAgaGFzX25vdmVsID0gcmFua2VkX25vdmVsLmFueShheGlzPTEpCiAgICAgICAgZmlyc3RfcmFuayA9IHJhbmtlZF9ub3ZlbC5hcmdtYXgoYXhpcz0xKQogICAgICAgIGNob3Nlbl9jb2wgPSByYW5rZWRfZ3JhcGhbbnAuYXJhbmdlKG4pLCBmaXJzdF9yYW5rXQogICAgICAgIHByb3Bvc2VkW3NlZWQsIGhhc19ub3ZlbF0gPSBuZWlnaGJvdXJzW25wLmFyYW5nZShuKSwgY2hvc2VuX2NvbF1baGFzX25vdmVsXQogICAgICAgIHBpY2tlZF9wcm9iW3NlZWQsIGhhc19ub3ZlbF0gPSBwcm9iYWJpbGl0aWVzW25wLmFyYW5nZShuKSwgY2hvc2VuX2NvbF1baGFzX25vdmVsXQoKICAgICAgICBwb3BfcHJvYiwgcHJlc2VudCA9IF9wb3Bfc2NvcmVzKHBvcFtzZWVkXSwgbmVpZ2hib3VycywgcHJvYmFiaWxpdGllcykKICAgICAgICBjb3ZlcmVkW3NlZWRdID0gcHJlc2VudC5zdW0oYXhpcz0xKQogICAgICAgIHdlYWtlc3Rfc2NvcmVkID0gbnAubWluKG5wLndoZXJlKHByZXNlbnQsIHBvcF9wcm9iLCBucC5pbmYpLCBheGlzPTEpCiAgICAgICAgc2NvcmVbc2VlZCwgaGFzX25vdmVsXSA9IHBpY2tlZF9wcm9iW3NlZWQsIGhhc19ub3ZlbF0gLSB3ZWFrZXN0X3Njb3JlZFtoYXNfbm92ZWxdCiAgICAgICAgZWxpZ2libGVbc2VlZF0gPSAoaGFzX25vdmVsICYgKGNvdmVyZWRbc2VlZF0gPj0gY29uZmlnLm1pbl9zY29yZWRfcG9wKQogICAgICAgICAgICAgICAgICAgICAgICAgICYgKHNjb3JlW3NlZWRdID4gY29uZmlnLm1pbl9tYXJnaW4pKQogICAgICAgIGlmIGxpbWl0OgogICAgICAgICAgICBjYW5kaWRhdGVzID0gbnAuZmxhdG5vbnplcm8oZWxpZ2libGVbc2VlZF0pCiAgICAgICAgICAgIG9yZGVyID0gbnAuYXJnc29ydCgtc2NvcmVbc2VlZCwgY2FuZGlkYXRlc10sIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgICAgIHNlbGVjdGVkID0gY2FuZGlkYXRlc1tvcmRlcls6bGltaXRdXQogICAgICAgICAgICBkZXN0aW5hdGlvbltzZWVkLCBzZWxlY3RlZF0gPSBwcm9wb3NlZFtzZWVkLCBzZWxlY3RlZF0KCiAgICByZXR1cm4gU3BhcnNlQWRkUmVzdWx0KGRlc3RpbmF0aW9uLCBwcm9wb3NlZCwgcGlja2VkX3Byb2IsIHNjb3JlLCBjb3ZlcmVkLCBlbGlnaWJsZSkK', 'v9_live/live_pilot.py': 'IiIiQmV0YVRvdXIgdjkgbGl2ZS1MS0ggcGlsb3QgaGVscGVycy4KClRoaXMgbW9kdWxlIGRlZmluZXMgaW50ZXJ2ZW50aW9ucyBhbmQgdmFsaWRhdGVzIHRoZWlyIGFjdHVhbCB0b3VyIGVuZHBvaW50cy4KVGhlIGZyb3plbiBUb3VyRm9sZCBjaGVja3BvaW50IHN1cHBsaWVzIHJlZmVyZW5jZS1lZGdlIHNjb3Jlczsgd2hldGhlciB0aGVzZQppbnRlcnZlbnRpb25zIGltcHJvdmUgTEtIIGlzIGFuIGVtcGlyaWNhbCBxdWVzdGlvbiBmb3IgdGhlIHBpbG90LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gYWRhcHRpdmVfcG9saWN5IGltcG9ydCBTcGFyc2VBZGRDb25maWcsIHByb3Bvc2Vfc3BhcnNlX2FkZCwgcmVyYW5rX3Njb3JlZF9wb3AKZnJvbSBjb29yZGluYXRvciBpbXBvcnQgU25hcHNob3QsIG5vb3BfYWN0aW9ucywgcmVhZF90b3VyLCBydW5fc2l4CgoKTElWRV9BUk1TID0gKCJzaGFtIiwgImJldGFfcHJvbW90ZSIsICJiZXRhX3NwYXJzZSIsICJmcmVxdWVuY3lfc2FtZV9nYXRlIikKTkFUSVZFX0FSTVMgPSAoIm5hdGl2ZV9wb3A1IiwgIm5hdGl2ZV9wb3A2IikKCgpkZWYgc2hhMjU2X2ZpbGUocGF0aDogUGF0aCkgLT4gc3RyOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc291cmNlOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogc291cmNlLnJlYWQoNCAqIDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgY2FuZGlkYXRlX2FycmF5KHNuYXBzaG90czogU2VxdWVuY2VbU25hcHNob3RdLCBmaWVsZDogc3RyKSAtPiBucC5uZGFycmF5OgogICAgcm93cyA9IFtnZXRhdHRyKHNuYXBzaG90LCBmaWVsZCkgZm9yIHNuYXBzaG90IGluIHNuYXBzaG90c10KICAgIG4gPSBsZW4ocm93c1swXSkKICAgIGlmIGFueShsZW4oaXRlbSkgIT0gbiBmb3IgaXRlbSBpbiByb3dzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5kaWRhdGUgc25hcHNob3RzIGhhdmUgZGlmZmVyZW50IGRpbWVuc2lvbnMiKQogICAgd2lkdGggPSBtYXgobGVuKHJvdykgZm9yIHNlZWRfcm93cyBpbiByb3dzIGZvciByb3cgaW4gc2VlZF9yb3dzKQogICAgaWYgd2lkdGggPCAxIG9yIHdpZHRoID49IG46CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgY2FuZGlkYXRlIHdpZHRoIHt3aWR0aH0iKQogICAgb3V0ID0gbnAuZnVsbCgobGVuKHJvd3MpLCBuLCB3aWR0aCksIC0xLCBkdHlwZT1ucC5pbnQzMikKICAgIGZvciBtLCBzZWVkX3Jvd3MgaW4gZW51bWVyYXRlKHJvd3MpOgogICAgICAgIGZvciBpLCByb3cgaW4gZW51bWVyYXRlKHNlZWRfcm93cyk6CiAgICAgICAgICAgIGlmIHJvdzoKICAgICAgICAgICAgICAgIG91dFttLCBpLCA6bGVuKHJvdyldID0gW3RvIC0gMSBmb3IgdG8sIF8gaW4gcm93XQogICAgcmV0dXJuIG91dAoKCmRlZiBmaXJzdF90b3VyX2FycmF5cyhzbmFwc2hvdHM6IFNlcXVlbmNlW1NuYXBzaG90XSwgeHk6IG5wLm5kYXJyYXksIFQpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgc3VjY2Vzc29ycywgY29zdHMgPSBbXSwgW10KICAgIGZvciBzbmFwc2hvdCBpbiBzbmFwc2hvdHM6CiAgICAgICAgdG91ciA9IG5wLmFzYXJyYXkoc25hcHNob3QuZmlyc3RfdG91ciwgZHR5cGU9bnAuaW50MzIpIC0gMQogICAgICAgIGlmIG5vdCBULmlzX3ZhbGlkX3RvdXIodG91ciwgbGVuKHh5KSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIGZpcnN0IHRvdXIgZnJvbSBzZWVkIHtzbmFwc2hvdC5zZWVkfSIpCiAgICAgICAgc3VjY2Vzc29ycy5hcHBlbmQoVC50b3VyX3N1Y2NfcHJlZCh0b3VyKVswXSkKICAgICAgICBjb3N0cy5hcHBlbmQoVC50b3VyX2xlbmd0aCh4eSwgdG91ciwgIkVVQ18yRCIpKQogICAgcmV0dXJuIG5wLnN0YWNrKHN1Y2Nlc3NvcnMsIGF4aXM9MSkuYXN0eXBlKG5wLmludDMyKSwgbnAuYXNhcnJheShjb3N0cywgZHR5cGU9bnAuZmxvYXQ2NCkKCgpkZWYgX3NhdmVfbnB6X2F0b21pYyhwYXRoOiBQYXRoLCAqKmFycmF5cykgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSArICIudG1wLm5weiIpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKHRtcCwgKiphcnJheXMpCiAgICBvcy5yZXBsYWNlKHRtcCwgcGF0aCkKCgpkZWYgX2FjdGlvbnNfZnJvbV9kZXN0aW5hdGlvbnMoZGVzdGluYXRpb25zOiBucC5uZGFycmF5LCBzbG90OiBpbnQsIGFscGhhOiBpbnQpIC0+IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dOgogICAgcmV0dXJuIFtbKGludCh0bykgKyAxLCBzbG90LCBhbHBoYSkgaWYgdG8gPj0gMCBlbHNlICgwLCAwLCA1MDApCiAgICAgICAgICAgICBmb3IgdG8gaW4gcm93XSBmb3Igcm93IGluIGRlc3RpbmF0aW9uc10KCgpkZWYgbWFrZV9hY3Rpb25fY2FsbGJhY2soa2luZDogc3RyLCB4eTogbnAubmRhcnJheSwgbW9kZWwsIGRldmljZSwgVCwgTSwKICAgICAgICAgICAgICAgICAgICAgICAgIGF1ZGl0X2RpcjogUGF0aCwgZ3JhcGhfazogaW50ID0gMTYsCiAgICAgICAgICAgICAgICAgICAgICAgICBzcGFyc2VfZnJhY3Rpb246IGZsb2F0ID0gMC4yNSk6CiAgICBpZiBraW5kIG5vdCBpbiBMSVZFX0FSTVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gbGl2ZSBhcm0ge2tpbmR9IikKICAgIGF1ZGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGNhbGxiYWNrKHNuYXBzaG90czogbGlzdFtTbmFwc2hvdF0pIC0+IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dOgogICAgICAgIGlmIGxlbihzbmFwc2hvdHMpICE9IDY6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRvdXJGb2xkIHJlcXVpcmVzIHNpeCBmaXJzdCB0b3VycyIpCiAgICAgICAgc3VjYywgY29zdHMgPSBmaXJzdF90b3VyX2FycmF5cyhzbmFwc2hvdHMsIHh5LCBUKQogICAgICAgIGxpdmUgPSBjYW5kaWRhdGVfYXJyYXkoc25hcHNob3RzLCAibGl2ZV9jYW5kaWRhdGVzIikKICAgICAgICBuYXRpdmUgPSBjYW5kaWRhdGVfYXJyYXkoc25hcHNob3RzLCAibmF0aXZlX2NhbmRpZGF0ZXMiKQogICAgICAgIHByZWRpY3Rpb24gPSBNLnByZWRpY3RfdG91cmZvbGQobW9kZWwsIHh5LCBncmFwaF9rLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbnNfc3VjYz1zdWNjLCBlbnNfY29zdD1jb3N0cykKICAgICAgICBncmFwaCA9IG5wLmFzYXJyYXkocHJlZGljdGlvblsiTkIiXSwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgcHJvYiA9IG5wLmFzYXJyYXkocHJlZGljdGlvblsicHJvYiJdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIGlmIGtpbmQgPT0gInNoYW0iOgogICAgICAgICAgICBkZXN0aW5hdGlvbnMgPSBucC5mdWxsKCg2LCBsZW4oeHkpKSwgLTEsIGR0eXBlPW5wLmludDMyKQogICAgICAgICAgICBhY3Rpb25zID0gW25vb3BfYWN0aW9ucyhsZW4oeHkpKSBmb3IgXyBpbiByYW5nZSg2KV0KICAgICAgICBlbGlmIGtpbmQgPT0gImJldGFfcHJvbW90ZSI6CiAgICAgICAgICAgIHJlb3JkZXJlZCA9IHJlcmFua19zY29yZWRfcG9wKGxpdmUsIGdyYXBoLCBwcm9iKQogICAgICAgICAgICBkZXN0aW5hdGlvbnMgPSBucC53aGVyZShyZW9yZGVyZWRbOiwgOiwgMF0gIT0gbGl2ZVs6LCA6LCAwXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVvcmRlcmVkWzosIDosIDBdLCAtMSkuYXN0eXBlKG5wLmludDMyKQogICAgICAgICAgICBhY3Rpb25zID0gX2FjdGlvbnNfZnJvbV9kZXN0aW5hdGlvbnMoZGVzdGluYXRpb25zLCBzbG90PTEsIGFscGhhPTApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3BhcnNlID0gcHJvcG9zZV9zcGFyc2VfYWRkKAogICAgICAgICAgICAgICAgbGl2ZSwgZ3JhcGgsIHByb2IsCiAgICAgICAgICAgICAgICBTcGFyc2VBZGRDb25maWcobWF4X2ZyYWN0aW9uPXNwYXJzZV9mcmFjdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtaW5fc2NvcmVkX3BvcD0yLCBtaW5fbWFyZ2luPTAuMCksCiAgICAgICAgICAgICkKICAgICAgICAgICAgZGVzdGluYXRpb25zID0gc3BhcnNlLmRlc3RpbmF0aW9uLmNvcHkoKQogICAgICAgICAgICBpZiBraW5kID09ICJmcmVxdWVuY3lfc2FtZV9nYXRlIjoKICAgICAgICAgICAgICAgICMgS2VlcCBCZXRhJ3Mgbm9kZS1sZXZlbCBnYXRlIGFuZCBpbnRlcnZlbnRpb24gY291bnQsIHdoaWxlCiAgICAgICAgICAgICAgICAjIHJlcGxhY2luZyBpdHMgZWRnZSBjaG9pY2Ugd2l0aCBlbnNlbWJsZSBmcmVxdWVuY3kuCiAgICAgICAgICAgICAgICBmcmVxdWVuY3kgPSBULmVuc2VtYmxlX2ZyZXF1ZW5jeShncmFwaCwgc3VjYykKICAgICAgICAgICAgICAgIGZyZXF1ZW5jeV9yYW5rID0gVC5mcmVxdWVuY3lfY2FuZGlkYXRlcygKICAgICAgICAgICAgICAgICAgICBncmFwaCwgcHJlZGljdGlvblsiRCJdLCBmcmVxdWVuY3ksIGs9Z3JhcGguc2hhcGVbMV0pCiAgICAgICAgICAgICAgICBuID0gbGVuKHh5KQogICAgICAgICAgICAgICAgZm9yIHNlZWQgaW4gcmFuZ2UoNik6CiAgICAgICAgICAgICAgICAgICAgZXhpc3RpbmcgPSB7CiAgICAgICAgICAgICAgICAgICAgICAgIChtaW4oaSwgaW50KHYpKSwgbWF4KGksIGludCh2KSkpCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpLCByb3cgaW4gZW51bWVyYXRlKGxpdmVbc2VlZF0pIGZvciB2IGluIHJvdyBpZiB2ID49IDAKICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICAgICAgZm9yIGkgaW4gbnAuZmxhdG5vbnplcm8oZGVzdGluYXRpb25zW3NlZWRdID49IDApOgogICAgICAgICAgICAgICAgICAgICAgICBjaG9pY2UgPSAtMQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgdiBpbiBmcmVxdWVuY3lfcmFua1tpXToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHYgPSBpbnQodikKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIChtaW4oaSwgdiksIG1heChpLCB2KSkgbm90IGluIGV4aXN0aW5nOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNob2ljZSA9IHYKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgICAgICBkZXN0aW5hdGlvbnNbc2VlZCwgaV0gPSBjaG9pY2UKICAgICAgICAgICAgIyBTbG90IDMgdGVzdHMgd2hldGhlciBhIHNwYXJzZSBwcm9wb3NhbCBjYW4gYmUgc2VhcmNoZWQgZWFybHkKICAgICAgICAgICAgIyB3aGlsZSByZXRhaW5pbmcgZXZlcnkgbmF0aXZlIGNhbmRpZGF0ZSBlZGdlIGluIHRoZSBsaXZlIHNldC4KICAgICAgICAgICAgYWN0aW9ucyA9IF9hY3Rpb25zX2Zyb21fZGVzdGluYXRpb25zKGRlc3RpbmF0aW9ucywgc2xvdD0zLCBhbHBoYT01MDApCiAgICAgICAgX3NhdmVfbnB6X2F0b21pYyhhdWRpdF9kaXIgLyAicG9saWN5X2FycmF5cy5ucHoiLCBmaXJzdF9zdWNjZXNzb3JzPXN1Y2MsCiAgICAgICAgICAgICAgICAgICAgICAgICBmaXJzdF9jb3N0cz1jb3N0cywgbmF0aXZlX2NhbmRpZGF0ZXM9bmF0aXZlLAogICAgICAgICAgICAgICAgICAgICAgICAgbGl2ZV9jYW5kaWRhdGVzPWxpdmUsIG1vZGVsX2dyYXBoPWdyYXBoLAogICAgICAgICAgICAgICAgICAgICAgICAgbW9kZWxfcHJvYmFiaWxpdHk9cHJvYiwgY2hvc2VuX2Rlc3RpbmF0aW9ucz1kZXN0aW5hdGlvbnMpCiAgICAgICAgKGF1ZGl0X2RpciAvICJwb2xpY3lfbWV0YS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAgICAgImFybSI6IGtpbmQsCiAgICAgICAgICAgICJhY3Rpb25zX3Blcl9zZWVkIjogW2ludChucC5zdW0ocm93ID49IDApKSBmb3Igcm93IGluIGRlc3RpbmF0aW9uc10sCiAgICAgICAgICAgICJtb2RlbF9pbmZlcmVuY2VfcyI6IGZsb2F0KHByZWRpY3Rpb25bInRfdG90YWwiXSksCiAgICAgICAgICAgICJzcGFyc2VfZnJhY3Rpb24iOiBzcGFyc2VfZnJhY3Rpb24gaWYga2luZCBpbiAoImJldGFfc3BhcnNlIiwgImZyZXF1ZW5jeV9zYW1lX2dhdGUiKSBlbHNlIE5vbmUsCiAgICAgICAgfSwgaW5kZW50PTIpICsgIlxuIikKICAgICAgICByZXR1cm4gYWN0aW9ucwoKICAgIHJldHVybiBjYWxsYmFjawoKCmRlZiBfcmVhZF9iZXN0KHBhdGg6IFBhdGgsIHh5OiBucC5uZGFycmF5LCBUKSAtPiBpbnQ6CiAgICAjIFRoZSBob29rIHdyaXRlcyBhIGNvbXBhY3QgYXRvbWljIGludGVnZXIgbGlzdDsgbmF0aXZlIExLSCB3cml0ZXMgYQogICAgIyBUU1BMSUIgVE9VUl9TRUNUSU9OIGZpbGUuICBSZWNvbXB1dGUgY29zdHMgZnJvbSBlaXRoZXIgZXhhY3QgdG91ci4KICAgIGZpcnN0X3Rva2VuID0gcGF0aC5yZWFkX3RleHQoKS5zcGxpdChtYXhzcGxpdD0xKVswXQogICAgaWYgZmlyc3RfdG9rZW4uaXNkaWdpdCgpOgogICAgICAgIHRvdXIgPSBucC5hc2FycmF5KHJlYWRfdG91cihwYXRoKSwgZHR5cGU9bnAuaW50MzIpIC0gMQogICAgZWxzZToKICAgICAgICB0b3VyID0gVC5yZWFkX3RvdXIocGF0aCkuYXN0eXBlKG5wLmludDMyKQogICAgaWYgbm90IFQuaXNfdmFsaWRfdG91cih0b3VyLCBsZW4oeHkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBiZXN0IHRvdXIge3BhdGh9IikKICAgIHJldHVybiBpbnQoVC50b3VyX2xlbmd0aCh4eSwgdG91ciwgIkVVQ18yRCIpKQoKCmRlZiBydW5fbGl2ZV9hcm0oa2luZDogc3RyLCBwYXRjaGVkX2JpbmFyeTogUGF0aCwgcHJvYmxlbTogUGF0aCwgeHk6IG5wLm5kYXJyYXksCiAgICAgICAgICAgICAgICAgd29ya19kaXI6IFBhdGgsIHNlZWRzOiBTZXF1ZW5jZVtpbnRdLCBidWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgbW9kZWwsIGRldmljZSwgVCwgTSwgKiwgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICAgICAgICAgICAgICAgc3BhcnNlX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMjUpIC0+IGRpY3Q6CiAgICBjYWxsYmFjayA9IG1ha2VfYWN0aW9uX2NhbGxiYWNrKGtpbmQsIHh5LCBtb2RlbCwgZGV2aWNlLCBULCBNLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB3b3JrX2Rpci5wYXJlbnQgLyAod29ya19kaXIubmFtZSArICJfYXVkaXQiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3BhcnNlX2ZyYWN0aW9uPXNwYXJzZV9mcmFjdGlvbikKICAgIHJlc3VsdCA9IHJ1bl9zaXgocGF0Y2hlZF9iaW5hcnksIHByb2JsZW0sIHdvcmtfZGlyLCBzZWVkcywgYnVkZ2V0X3MsCiAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrLCBtYXhfdHJpYWxzPW1heF90cmlhbHMpCiAgICBjb3N0cyA9IFtdCiAgICBmb3IgcnVuIGluIHJlc3VsdFsicnVucyJdOgogICAgICAgIGlmIHJ1blsiYmVzdF90b3VyIl0gaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTm8gYmVzdCB0b3VyIGluIHtraW5kfSwgc2VlZCB7cnVuWydzZWVkJ119IikKICAgICAgICBjb3N0cy5hcHBlbmQoX3JlYWRfYmVzdChQYXRoKHJ1blsiYmVzdF90b3VyIl0pLCB4eSwgVCkpCiAgICBpZiBub3QgcmVzdWx0WyJiYXJyaWVyX2NvbXBsZXRlZCJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZpcnN0LXRvdXIgYmFycmllciBpbmNvbXBsZXRlIGluIHtraW5kfSIpCiAgICBpZiBhbnkocnVuWyJhcHBsaWVkX3N0YXRzIl0gaXMgTm9uZSBmb3IgcnVuIGluIHJlc3VsdFsicnVucyJdKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJPbmUgb3IgbW9yZSBob29rIGFjdGlvbnMgd2VyZSBub3QgYXBwbGllZCBpbiB7a2luZH0iKQogICAgcmVzdWx0LnVwZGF0ZShhcm09a2luZCwgc2VlZF9jb3N0cz1jb3N0cywgYmVzdF9jb3N0PW1pbihjb3N0cyksCiAgICAgICAgICAgICAgICAgIHBvbGljeV9tZXRhPXN0cih3b3JrX2Rpci5wYXJlbnQgLyAod29ya19kaXIubmFtZSArICJfYXVkaXQiKSksCiAgICAgICAgICAgICAgICAgIGJpbmFyeV9zaGEyNTY9c2hhMjU2X2ZpbGUocGF0Y2hlZF9iaW5hcnkpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBydW5fbmF0aXZlX2FybShraW5kOiBzdHIsIG5hdGl2ZV9iaW5hcnk6IFBhdGgsIHByb2JsZW06IFBhdGgsIHh5OiBucC5uZGFycmF5LAogICAgICAgICAgICAgICAgICAgd29ya19kaXI6IFBhdGgsIHNlZWRzOiBTZXF1ZW5jZVtpbnRdLCBidWRnZXRfczogZmxvYXQsIFQsCiAgICAgICAgICAgICAgICAgICAqLCBtYXhfdHJpYWxzOiBpbnQgPSAxMDBfMDAwXzAwMCkgLT4gZGljdDoKICAgIGlmIGtpbmQgbm90IGluIE5BVElWRV9BUk1TIG9yIGxlbihzZWVkcykgIT0gNiBvciB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIG5hdGl2ZSBhcm0sIHNlZWQgcGFuZWwsIG9yIHdvcmsgZGlyZWN0b3J5IikKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIGsgPSA1IGlmIGtpbmQgPT0gIm5hdGl2ZV9wb3A1IiBlbHNlIDYKICAgIHByb2Nlc3Nlcywgc3RyZWFtcyA9IFtdLCBbXQogICAgc3RhcnQgPSB0aW1lLm1vbm90b25pYygpCiAgICBkZWFkbGluZSA9IHN0YXJ0ICsgYnVkZ2V0X3MKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgYmFzZS5ta2RpcigpCiAgICAgICAgICAgIHBhcmFtcyA9IHsKICAgICAgICAgICAgICAgICJQUk9CTEVNX0ZJTEUiOiBzdHIocHJvYmxlbS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICAgIkNBTkRJREFURV9TRVRfVFlQRSI6ICJQT1BNVVNJQyIsICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgICAgICAgICAiTUFYX0NBTkRJREFURVMiOiBrLCAiU0VFRCI6IHNlZWQsICJSVU5TIjogMSwKICAgICAgICAgICAgICAgICJNQVhfVFJJQUxTIjogbWF4X3RyaWFscywgIlRJTUVfTElNSVQiOiBtYXgoMTAuMCwgMTAuMCAqIGJ1ZGdldF9zKSwKICAgICAgICAgICAgICAgICJUUkFDRV9MRVZFTCI6IDAsICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgICAgICAgICAgICAgIk9VVFBVVF9UT1VSX0ZJTEUiOiBzdHIoYmFzZSAvICJiZXN0LnRvdXIiKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBwYXIgPSBiYXNlIC8gInJ1bi5wYXIiCiAgICAgICAgICAgIFQud3JpdGVfcGFyKHBhciwgcGFyYW1zKQogICAgICAgICAgICBzdGRvdXQgPSAoYmFzZSAvICJzdGRvdXQudHh0Iikub3BlbigidyIpCiAgICAgICAgICAgIHN0ZGVyciA9IChiYXNlIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgc3RyZWFtcy5leHRlbmQoKHN0ZG91dCwgc3RkZXJyKSkKICAgICAgICAgICAgZW52ID0gZGljdChvcy5lbnZpcm9uKQogICAgICAgICAgICBlbnYucG9wKCJCRVRBVE9VUl9IT09LX0RJUiIsIE5vbmUpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3BlbigKICAgICAgICAgICAgICAgIFtzdHIobmF0aXZlX2JpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyLnJlc29sdmUoKSldLCBjd2Q9YmFzZSwKICAgICAgICAgICAgICAgIHN0ZG91dD1zdGRvdXQsIHN0ZGVycj1zdGRlcnIsIGVudj1lbnYpKQogICAgICAgIHdoaWxlIGFueShwcm9jZXNzLnBvbGwoKSBpcyBOb25lIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlcykgYW5kIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgdGltZS5zbGVlcCgwLjAyKQogICAgZmluYWxseToKICAgICAgICBmb3IgcHJvY2VzcyBpbiBwcm9jZXNzZXM6CiAgICAgICAgICAgIGlmIHByb2Nlc3MucG9sbCgpIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBwcm9jZXNzLnRlcm1pbmF0ZSgpCiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQodGltZW91dD0yKQogICAgICAgICAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZDoKICAgICAgICAgICAgICAgIHByb2Nlc3Mua2lsbCgpCiAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQoKQogICAgICAgIGZvciBzdHJlYW0gaW4gc3RyZWFtczoKICAgICAgICAgICAgc3RyZWFtLmNsb3NlKCkKICAgIGNvc3RzID0gW10KICAgIHJ1bnMgPSBbXQogICAgZm9yIHNlZWQsIHByb2Nlc3MgaW4gemlwKHNlZWRzLCBwcm9jZXNzZXMpOgogICAgICAgIGJlc3QgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iIC8gImJlc3QudG91ciIKICAgICAgICBpZiBub3QgYmVzdC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTm8gYmVzdCB0b3VyIGluIHtraW5kfSwgc2VlZCB7c2VlZH0iKQogICAgICAgIGNvc3QgPSBfcmVhZF9iZXN0KGJlc3QsIHh5LCBUKQogICAgICAgIGNvc3RzLmFwcGVuZChjb3N0KQogICAgICAgIHJ1bnMuYXBwZW5kKHsic2VlZCI6IHNlZWQsICJiZXN0X3RvdXIiOiBzdHIoYmVzdCksCiAgICAgICAgICAgICAgICAgICAgICJyZXR1cm5jb2RlIjogcHJvY2Vzcy5yZXR1cm5jb2RlLCAiY29zdCI6IGNvc3R9KQogICAgcmVzdWx0ID0geyJhcm0iOiBraW5kLCAic2VlZHMiOiBsaXN0KHNlZWRzKSwgIndhbGxfYnVkZ2V0X3MiOiBidWRnZXRfcywKICAgICAgICAgICAgICAid2FsbF9lbGFwc2VkX3MiOiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnQsICJzZWVkX2Nvc3RzIjogY29zdHMsCiAgICAgICAgICAgICAgImJlc3RfY29zdCI6IG1pbihjb3N0cyksICJydW5zIjogcnVucywKICAgICAgICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IHNoYTI1Nl9maWxlKG5hdGl2ZV9iaW5hcnkpfQogICAgKHdvcmtfZGlyIC8gInJ1bl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAiXG4iKQogICAgcmV0dXJuIHJlc3VsdAo=', 'v10_repair/BetaTourRepairHook.h': 'LyogQmV0YVRvdXIgdjEwIGRlbGF5ZWQtYmFycmllciByZXNlYXJjaCBob29rIGZvciBMS0ggMy4wLjE0LiAqLwojaWZuZGVmIEJFVEFUT1VSX0hPT0tfSAojZGVmaW5lIEJFVEFUT1VSX0hPT0tfSAoKI2luY2x1ZGUgPGVycm5vLmg+CiNpbmNsdWRlIDxsaW1pdHMuaD4KI2luY2x1ZGUgPHN0ZGlvLmg+CiNpbmNsdWRlIDxzdGRsaWIuaD4KI2luY2x1ZGUgPHN0cmluZy5oPgojaW5jbHVkZSA8c3lzL3RpbWUuaD4KI2luY2x1ZGUgPHVuaXN0ZC5oPgoKc3RhdGljIGRvdWJsZSBCVF9TZWFyY2hTdGFydFdhbGwgPSAwLjA7CnN0YXRpYyBpbnQgQlRfQmFycmllclRyaWdnZXJlZCA9IDA7CgpzdGF0aWMgY29uc3QgY2hhciAqQlRfSG9va0RpcmVjdG9yeSh2b2lkKQp7CiAgICBjb25zdCBjaGFyICpQYXRoID0gZ2V0ZW52KCJCRVRBVE9VUl9IT09LX0RJUiIpOwogICAgcmV0dXJuIFJ1biA9PSAxICYmIFByb2JsZW1UeXBlID09IFRTUCAmJiBEaW1lbnNpb24gPT0gRGltZW5zaW9uU2F2ZWQgJiYKICAgICAgICBQYXRoICYmICpQYXRoID8gUGF0aCA6IDA7Cn0KCnN0YXRpYyB2b2lkIEJUX1BhdGgoY2hhciAqT3V0LCBzaXplX3QgU2l6ZSwgY29uc3QgY2hhciAqRGlyZWN0b3J5LAogICAgICAgICAgICAgICAgICAgIGNvbnN0IGNoYXIgKk5hbWUpCnsKICAgIGlmIChzbnByaW50ZihPdXQsIFNpemUsICIlcy8lcyIsIERpcmVjdG9yeSwgTmFtZSkgPj0gKGludCkgU2l6ZSkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayBwYXRoIGV4Y2VlZHMgUEFUSF9NQVhcbiIpOwogICAgICAgIGV4aXQoOTcpOwogICAgfQp9CgpzdGF0aWMgZG91YmxlIEJUX1dhbGxUaW1lKHZvaWQpCnsKICAgIHN0cnVjdCB0aW1ldmFsIFR2OwogICAgZ2V0dGltZW9mZGF5KCZUdiwgMCk7CiAgICByZXR1cm4gVHYudHZfc2VjICsgVHYudHZfdXNlYyAvIDEwMDAwMDAuMDsKfQoKc3RhdGljIEZJTEUgKkJUX09wZW5UZW1wb3JhcnkoY29uc3QgY2hhciAqRGlyZWN0b3J5LCBjb25zdCBjaGFyICpOYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjaGFyICpGaW5hbCwgY2hhciAqVGVtcG9yYXJ5KQp7CiAgICBGSUxFICpGaWxlOwogICAgQlRfUGF0aChGaW5hbCwgUEFUSF9NQVgsIERpcmVjdG9yeSwgTmFtZSk7CiAgICBpZiAoc25wcmludGYoVGVtcG9yYXJ5LCBQQVRIX01BWCwgIiVzLnRtcC4lbGQiLCBGaW5hbCwgKGxvbmcpIGdldHBpZCgpKQogICAgICAgID49IFBBVEhfTUFYKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIHRlbXBvcmFyeSBwYXRoIGV4Y2VlZHMgUEFUSF9NQVhcbiIpOwogICAgICAgIGV4aXQoOTcpOwogICAgfQogICAgRmlsZSA9IGZvcGVuKFRlbXBvcmFyeSwgInciKTsKICAgIGlmICghRmlsZSkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayBjYW5ub3Qgd3JpdGUgJXM6ICVzXG4iLCBUZW1wb3JhcnksCiAgICAgICAgICAgICAgICBzdHJlcnJvcihlcnJubykpOwogICAgICAgIGV4aXQoOTcpOwogICAgfQogICAgcmV0dXJuIEZpbGU7Cn0KCnN0YXRpYyB2b2lkIEJUX0NvbW1pdChGSUxFICpGaWxlLCBjb25zdCBjaGFyICpUZW1wb3JhcnksIGNvbnN0IGNoYXIgKkZpbmFsKQp7CiAgICBpZiAoZmNsb3NlKEZpbGUpICE9IDAgfHwgcmVuYW1lKFRlbXBvcmFyeSwgRmluYWwpICE9IDApIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IGNvbW1pdCAlczogJXNcbiIsIEZpbmFsLAogICAgICAgICAgICAgICAgc3RyZXJyb3IoZXJybm8pKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKLyogRWFjaCByb3c6IG5vZGUgSUQsIGNvdW50LCB0aGVuIChuZWlnaGJvciBJRCwgYWxwaGEpIHBhaXJzLiAqLwpzdGF0aWMgdm9pZCBCVF9Xcml0ZUNhbmRpZGF0ZXMoY29uc3QgY2hhciAqRGlyZWN0b3J5LCBjb25zdCBjaGFyICpOYW1lKQp7CiAgICBjaGFyIEZpbmFsW1BBVEhfTUFYXSwgVGVtcG9yYXJ5W1BBVEhfTUFYXTsKICAgIEZJTEUgKkZpbGUgPSBCVF9PcGVuVGVtcG9yYXJ5KERpcmVjdG9yeSwgTmFtZSwgRmluYWwsIFRlbXBvcmFyeSk7CiAgICBpbnQgaTsKICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBEaW1lbnNpb25TYXZlZCk7CiAgICBmb3IgKGkgPSAxOyBpIDw9IERpbWVuc2lvblNhdmVkOyBpKyspIHsKICAgICAgICBDYW5kaWRhdGUgKkNhbmRpZGF0ZVB0cjsKICAgICAgICBpbnQgQ291bnQgPSAwOwogICAgICAgIE5vZGUgKkZyb20gPSAmTm9kZVNldFtpXTsKICAgICAgICBmb3IgKENhbmRpZGF0ZVB0ciA9IEZyb20tPkNhbmRpZGF0ZVNldDsKICAgICAgICAgICAgIENhbmRpZGF0ZVB0ciAmJiBDYW5kaWRhdGVQdHItPlRvOyBDYW5kaWRhdGVQdHIrKykKICAgICAgICAgICAgQ291bnQrKzsKICAgICAgICBmcHJpbnRmKEZpbGUsICIlZCAlZCIsIGksIENvdW50KTsKICAgICAgICBmb3IgKENhbmRpZGF0ZVB0ciA9IEZyb20tPkNhbmRpZGF0ZVNldDsKICAgICAgICAgICAgIENhbmRpZGF0ZVB0ciAmJiBDYW5kaWRhdGVQdHItPlRvOyBDYW5kaWRhdGVQdHIrKykKICAgICAgICAgICAgZnByaW50ZihGaWxlLCAiICVkICVkIiwgQ2FuZGlkYXRlUHRyLT5Uby0+SWQsCiAgICAgICAgICAgICAgICAgICAgQ2FuZGlkYXRlUHRyLT5BbHBoYSk7CiAgICAgICAgZnB1dGMoJ1xuJywgRmlsZSk7CiAgICB9CiAgICBCVF9Db21taXQoRmlsZSwgVGVtcG9yYXJ5LCBGaW5hbCk7Cn0KCnN0YXRpYyB2b2lkIEJUX1dyaXRlQmFycmllclRvdXIoY29uc3QgY2hhciAqRGlyZWN0b3J5KQp7CiAgICBjaGFyIEZpbmFsW1BBVEhfTUFYXSwgVGVtcG9yYXJ5W1BBVEhfTUFYXTsKICAgIEZJTEUgKkZpbGUgPSBCVF9PcGVuVGVtcG9yYXJ5KERpcmVjdG9yeSwgImJhcnJpZXIudG91ciIsIEZpbmFsLCBUZW1wb3JhcnkpOwogICAgaW50IGk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKLyogQW4gYXRvbWljIGluY3VtYmVudCBzdXJ2aXZlcyBhbiBleHRlcm5hbCB3YWxsLWNsb2NrIGRlYWRsaW5lL3Rlcm1pbmF0aW9uLiAqLwpzdGF0aWMgdm9pZCBCVF9SZWNvcmRCZXN0KHZvaWQpCnsKICAgIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSA9IEJUX0hvb2tEaXJlY3RvcnkoKTsKICAgIGNoYXIgRmluYWxbUEFUSF9NQVhdLCBUZW1wb3JhcnlbUEFUSF9NQVhdLCBFdmVudHNbUEFUSF9NQVhdOwogICAgRklMRSAqRmlsZTsKICAgIGludCBpOwogICAgaWYgKCFEaXJlY3RvcnkpCiAgICAgICAgcmV0dXJuOwogICAgRmlsZSA9IEJUX09wZW5UZW1wb3JhcnkoRGlyZWN0b3J5LCAiYmVzdC50b3VyIiwgRmluYWwsIFRlbXBvcmFyeSk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKICAgIEJUX1BhdGgoRXZlbnRzLCBzaXplb2YoRXZlbnRzKSwgRGlyZWN0b3J5LCAiYmVzdC5ldmVudHMiKTsKICAgIEZpbGUgPSBmb3BlbihFdmVudHMsICJhIik7CiAgICBpZiAoIUZpbGUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IGFwcGVuZCAlczogJXNcbiIsIEV2ZW50cywKICAgICAgICAgICAgICAgIHN0cmVycm9yKGVycm5vKSk7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmcHJpbnRmKEZpbGUsICIlLjZmICVsbGRcbiIsIEJUX1dhbGxUaW1lKCksIChsb25nIGxvbmcpIEJldHRlckNvc3QpOwogICAgaWYgKGZjbG9zZShGaWxlKSAhPSAwKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIGNhbm5vdCBmbHVzaCAlc1xuIiwgRXZlbnRzKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKc3RhdGljIGludCBCVF9BcHBseU9uZShpbnQgRnJvbUlkLCBpbnQgVG9JZCwgaW50IFNsb3QsIGludCBBbHBoYSwKICAgICAgICAgICAgICAgICAgICAgICBpbnQgKldhc0V4aXN0aW5nLCBpbnQgKldhc01vdmVkKQp7CiAgICBOb2RlICpGcm9tLCAqVG87CiAgICBDYW5kaWRhdGUgKlN0YXJ0LCAqQ2hvc2VuLCBUZW1wOwogICAgaW50IENvdW50ID0gMCwgSW5kZXgsIERlc3RpbmF0aW9uOwoKICAgIGlmIChGcm9tSWQgPCAxIHx8IEZyb21JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPCAxIHx8CiAgICAgICAgVG9JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPT0gRnJvbUlkIHx8IFNsb3QgPCAwIHx8CiAgICAgICAgQWxwaGEgPCAwIHx8IEFscGhhID09IElOVF9NQVgpCiAgICAgICAgcmV0dXJuIC0xOwogICAgRnJvbSA9ICZOb2RlU2V0W0Zyb21JZF07CiAgICBUbyA9ICZOb2RlU2V0W1RvSWRdOwogICAgU3RhcnQgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICBmb3IgKENob3NlbiA9IFN0YXJ0OyBDaG9zZW4gJiYgQ2hvc2VuLT5UbzsgQ2hvc2VuKyspCiAgICAgICAgaWYgKENob3Nlbi0+VG8gPT0gVG8pCiAgICAgICAgICAgICpXYXNFeGlzdGluZyA9IDE7CiAgICAvKiBBIHJhbmstb25seSBhY3Rpb24gbXVzdCByZXRhaW4gdGhlIG5hdGl2ZSBlZGdlJ3MgYWxwaGEgdmFsdWUuICovCiAgICBpZiAoISpXYXNFeGlzdGluZyAmJiAhQWRkQ2FuZGlkYXRlKEZyb20sIFRvLCBDKEZyb20sIFRvKSwgQWxwaGEpKQogICAgICAgIHJldHVybiAtMTsKICAgIFN0YXJ0ID0gRnJvbS0+Q2FuZGlkYXRlU2V0OwogICAgZm9yIChDaG9zZW4gPSBTdGFydDsgQ2hvc2VuICYmIENob3Nlbi0+VG87IENob3NlbisrKQogICAgICAgIENvdW50Kys7CiAgICBmb3IgKEluZGV4ID0gMDsgSW5kZXggPCBDb3VudDsgSW5kZXgrKykKICAgICAgICBpZiAoU3RhcnRbSW5kZXhdLlRvID09IFRvKQogICAgICAgICAgICBicmVhazsKICAgIGlmIChJbmRleCA9PSBDb3VudCkKICAgICAgICByZXR1cm4gLTE7CiAgICBpZiAoU2xvdCA+IDApIHsKICAgICAgICBEZXN0aW5hdGlvbiA9IFNsb3QgLSAxOwogICAgICAgIGlmIChEZXN0aW5hdGlvbiA+PSBDb3VudCkKICAgICAgICAgICAgRGVzdGluYXRpb24gPSBDb3VudCAtIDE7CiAgICAgICAgaWYgKEluZGV4ICE9IERlc3RpbmF0aW9uKSB7CiAgICAgICAgICAgIGludCBqOwogICAgICAgICAgICBUZW1wID0gU3RhcnRbSW5kZXhdOwogICAgICAgICAgICBpZiAoSW5kZXggPiBEZXN0aW5hdGlvbikKICAgICAgICAgICAgICAgIGZvciAoaiA9IEluZGV4OyBqID4gRGVzdGluYXRpb247IGotLSkKICAgICAgICAgICAgICAgICAgICBTdGFydFtqXSA9IFN0YXJ0W2ogLSAxXTsKICAgICAgICAgICAgZWxzZQogICAgICAgICAgICAgICAgZm9yIChqID0gSW5kZXg7IGogPCBEZXN0aW5hdGlvbjsgaisrKQogICAgICAgICAgICAgICAgICAgIFN0YXJ0W2pdID0gU3RhcnRbaiArIDFdOwogICAgICAgICAgICBTdGFydFtEZXN0aW5hdGlvbl0gPSBUZW1wOwogICAgICAgICAgICAqV2FzTW92ZWQgPSAxOwogICAgICAgIH0KICAgIH0KICAgIHJldHVybiAwOwp9CgpzdGF0aWMgdm9pZCBCVF9XYWl0QW5kQXBwbHkoY29uc3QgY2hhciAqRGlyZWN0b3J5KQp7CiAgICBjaGFyIEFjdGlvbltQQVRIX01BWF0sIEZpbmFsW1BBVEhfTUFYXSwgVGVtcG9yYXJ5W1BBVEhfTUFYXTsKICAgIEZJTEUgKkZpbGU7CiAgICBpbnQgTiwgaSwgRnJvbSwgVG8sIFNsb3QsIEFscGhhOwogICAgaW50IEFkZGVkID0gMCwgRXhpc3RpbmcgPSAwLCBSZWplY3RlZCA9IDAsIE1vdmVkID0gMDsKICAgIGRvdWJsZSBTdGFydCA9IEJUX1dhbGxUaW1lKCksIFRpbWVvdXQgPSAzMDAuMDsKICAgIGNvbnN0IGNoYXIgKlRpbWVvdXRUZXh0ID0gZ2V0ZW52KCJCRVRBVE9VUl9IT09LX1RJTUVPVVRfUyIpOwogICAgaWYgKFRpbWVvdXRUZXh0ICYmIGF0b2YoVGltZW91dFRleHQpID4gMCkKICAgICAgICBUaW1lb3V0ID0gYXRvZihUaW1lb3V0VGV4dCk7CiAgICBCVF9QYXRoKEFjdGlvbiwgc2l6ZW9mKEFjdGlvbiksIERpcmVjdG9yeSwgImFjdGlvbi50eHQiKTsKICAgIHdoaWxlICghKEZpbGUgPSBmb3BlbihBY3Rpb24sICJyIikpKSB7CiAgICAgICAgaWYgKEJUX1dhbGxUaW1lKCkgLSBTdGFydCA+IFRpbWVvdXQpIHsKICAgICAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIHRpbWVkIG91dCB3YWl0aW5nIGZvciAlc1xuIiwgQWN0aW9uKTsKICAgICAgICAgICAgZXhpdCg5Nyk7CiAgICAgICAgfQogICAgICAgIHVzbGVlcCgxMDAwMCk7CiAgICB9CiAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCIsICZOKSAhPSAxIHx8IE4gIT0gRGltZW5zaW9uU2F2ZWQpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgYWN0aW9uIGRpbWVuc2lvbiBtaXNtYXRjaFxuIik7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmb3IgKGkgPSAxOyBpIDw9IE47IGkrKykgewogICAgICAgIGludCBXYXNFeGlzdGluZyA9IDAsIFdhc01vdmVkID0gMDsKICAgICAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCAlZCAlZCAlZCIsICZGcm9tLCAmVG8sICZTbG90LCAmQWxwaGEpICE9IDQgfHwKICAgICAgICAgICAgRnJvbSAhPSBpKSB7CiAgICAgICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayBtYWxmb3JtZWQgYWN0aW9uIHJvdyAlZFxuIiwgaSk7CiAgICAgICAgICAgIGV4aXQoOTcpOwogICAgICAgIH0KICAgICAgICBpZiAoVG8gPT0gMCkKICAgICAgICAgICAgY29udGludWU7CiAgICAgICAgaWYgKEJUX0FwcGx5T25lKEZyb20sIFRvLCBTbG90LCBBbHBoYSwgJldhc0V4aXN0aW5nLAogICAgICAgICAgICAgICAgICAgICAgICAmV2FzTW92ZWQpICE9IDApCiAgICAgICAgICAgIFJlamVjdGVkKys7CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIEFkZGVkICs9ICFXYXNFeGlzdGluZzsKICAgICAgICAgICAgRXhpc3RpbmcgKz0gV2FzRXhpc3Rpbmc7CiAgICAgICAgICAgIE1vdmVkICs9IFdhc01vdmVkOwogICAgICAgIH0KICAgIH0KICAgIGZjbG9zZShGaWxlKTsKICAgIGlmIChSZWplY3RlZCkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayByZWplY3RlZCAlZCBjYW5kaWRhdGUgYWN0aW9uc1xuIiwKICAgICAgICAgICAgICAgIFJlamVjdGVkKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIEJUX1dyaXRlQ2FuZGlkYXRlcyhEaXJlY3RvcnksICJwb3N0LmNhbmRpZGF0ZXMiKTsKICAgIEZpbGUgPSBCVF9PcGVuVGVtcG9yYXJ5KERpcmVjdG9yeSwgImFwcGxpZWQuc3RhdHMiLCBGaW5hbCwgVGVtcG9yYXJ5KTsKICAgIGZwcmludGYoRmlsZSwgImFkZGVkICVkXG5leGlzdGluZyAlZFxubW92ZWQgJWRcbnJlamVjdGVkICVkXG53YWl0X3dhbGxfcyAlLjZmXG4iLAogICAgICAgICAgICBBZGRlZCwgRXhpc3RpbmcsIE1vdmVkLCBSZWplY3RlZCwgQlRfV2FsbFRpbWUoKSAtIFN0YXJ0KTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKc3RhdGljIHZvaWQgQlRfRXhwb3J0TmF0aXZlQ2FuZGlkYXRlcyh2b2lkKQp7CiAgICBjb25zdCBjaGFyICpEaXJlY3RvcnkgPSBCVF9Ib29rRGlyZWN0b3J5KCk7CiAgICBpZiAoRGlyZWN0b3J5KSB7CiAgICAgICAgQlRfU2VhcmNoU3RhcnRXYWxsID0gQlRfV2FsbFRpbWUoKTsKICAgICAgICBCVF9Xcml0ZUNhbmRpZGF0ZXMoRGlyZWN0b3J5LCAibmF0aXZlLmNhbmRpZGF0ZXMiKTsKICAgIH0KfQoKc3RhdGljIHZvaWQgQlRfQWZ0ZXJGaXJzdFRyaWFsKHZvaWQpCnsKICAgIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSA9IEJUX0hvb2tEaXJlY3RvcnkoKTsKICAgIGNvbnN0IGNoYXIgKkFmdGVyVGV4dCA9IGdldGVudigiQkVUQVRPVVJfSE9PS19BRlRFUl9TIik7CiAgICBkb3VibGUgQWZ0ZXIgPSBBZnRlclRleHQgJiYgYXRvZihBZnRlclRleHQpID4gMCA/IGF0b2YoQWZ0ZXJUZXh0KSA6IDAuMDsKICAgIGlmIChEaXJlY3RvcnkgJiYgIUJUX0JhcnJpZXJUcmlnZ2VyZWQgJiYgVHJpYWwgPj0gMSAmJgogICAgICAgIEJUX1dhbGxUaW1lKCkgLSBCVF9TZWFyY2hTdGFydFdhbGwgPj0gQWZ0ZXIpIHsKICAgICAgICBCVF9CYXJyaWVyVHJpZ2dlcmVkID0gMTsKICAgICAgICBCVF9Xcml0ZUNhbmRpZGF0ZXMoRGlyZWN0b3J5LCAibGl2ZS5jYW5kaWRhdGVzIik7CiAgICAgICAgQlRfV3JpdGVCYXJyaWVyVG91cihEaXJlY3RvcnkpOyAvKiByZWFkaW5lc3MgbWFya2VyOyBwdWJsaXNoZWQgbGFzdCAqLwogICAgICAgIEJUX1dhaXRBbmRBcHBseShEaXJlY3RvcnkpOwogICAgfQp9CgojZW5kaWYK', 'v10_repair/build_repair_hook.py': 'IiIiQnVpbGQgdGhlIGRlbGF5ZWQtYmFycmllciBCZXRhVG91ciB2MTAgaG9vayBmcm9tIExLSC0zLjAuMTQuCgpUaGUgTEtIIHNvdXJjZSBpcyBmZXRjaGVkIHNlcGFyYXRlbHkgYnkgdGhlIGV4cGVyaW1lbnQgbm90ZWJvb2suIFRoaXMgc2NyaXB0CnNoaXBzIG9ubHkgdGhlIEJldGFUb3VyIHBhdGNoIGFuZCBjaGVja3MgdGhlIHVwc3RyZWFtIGZpbGUgYmVmb3JlIGVkaXRpbmcgaXQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKVVBTVFJFQU1fRklORFRPVVJfU0hBMjU2ID0gKAogICAgIjU2N2U3MmFjYzliNzM5MjQ4OGRkMTU4YmU0ZGYwMjY3NTYwNDI0MjA0YTBlODhkMmQ5YjYyZjkyM2E3YzBlZWUiCikKCgpkZWYgX3JlcGxhY2Vfb25jZShzb3VyY2U6IHN0ciwgb2xkOiBzdHIsIG5ldzogc3RyKSAtPiBzdHI6CiAgICBjb3VudCA9IHNvdXJjZS5jb3VudChvbGQpCiAgICBpZiBjb3VudCAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBvbmUgRmluZFRvdXIgYW5jaG9yLCBmb3VuZCB7Y291bnR9OiB7b2xkWzo4MF0hcn0iKQogICAgcmV0dXJuIHNvdXJjZS5yZXBsYWNlKG9sZCwgbmV3LCAxKQoKCmRlZiBwYXRjaF9maW5kdG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbiNpbmNsdWRlICJCZXRhVG91ckhvb2suaCJcbicsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgIGRvdWJsZSBFbnRyeVRpbWUgPSBHZXRUaW1lKCk7XG5cbiAgICB0ID0gRmlyc3ROb2RlOyIsCiAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4iCiAgICAgICAgIiAgICBCVF9FeHBvcnROYXRpdmVDYW5kaWRhdGVzKCk7XG4gICAgdCA9IEZpcnN0Tm9kZTsiLAogICAgKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiIsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiAgICAgICAgICAgIEJUX1JlY29yZEJlc3QoKTtcbiIsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgICAgICB9XG4gICAgfVxuICAgIGlmIChCYWNrYm9uZVRyaWFscyA+IDAgJiYgQmFja2JvbmVUcmlhbHMgPCBNYXhUcmlhbHMpIHsiLAogICAgICAgICIgICAgICAgIH1cbiAgICAgICAgQlRfQWZ0ZXJGaXJzdFRyaWFsKCk7XG4iCiAgICAgICAgIiAgICB9XG4gICAgaWYgKEJhY2tib25lVHJpYWxzID4gMCAmJiBCYWNrYm9uZVRyaWFscyA8IE1heFRyaWFscykgeyIsCiAgICApCiAgICByZXR1cm4gc291cmNlCgoKZGVmIGJ1aWxkKHNvdXJjZV9kaXI6IFBhdGgsIG91dHB1dF9kaXI6IFBhdGgsIGpvYnM6IGludCA9IDQpIC0+IFBhdGg6CiAgICBzb3VyY2VfZGlyID0gc291cmNlX2Rpci5yZXNvbHZlKCkKICAgIG91dHB1dF9kaXIgPSBvdXRwdXRfZGlyLnJlc29sdmUoKQogICAgdXBzdHJlYW0gPSBzb3VyY2VfZGlyIC8gIlNSQyIgLyAiRmluZFRvdXIuYyIKICAgIGlmIG5vdCB1cHN0cmVhbS5leGlzdHMoKSBvciBub3QgKHNvdXJjZV9kaXIgLyAiTWFrZWZpbGUiKS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkxLSC0zLjAuMTQgc291cmNlIHRyZWUgbWlzc2luZzoge3NvdXJjZV9kaXJ9IikKICAgIG9yaWdpbmFsX2J5dGVzID0gdXBzdHJlYW0ucmVhZF9ieXRlcygpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihvcmlnaW5hbF9ieXRlcykuaGV4ZGlnZXN0KCkKICAgIGlmIGRpZ2VzdCAhPSBVUFNUUkVBTV9GSU5EVE9VUl9TSEEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgTEtIIEZpbmRUb3VyLmMgU0hBMjU2OiB7ZGlnZXN0fSIpCiAgICBpZiBvdXRwdXRfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSB7b3V0cHV0X2Rpcn0iKQogICAgc2h1dGlsLmNvcHl0cmVlKAogICAgICAgIHNvdXJjZV9kaXIsCiAgICAgICAgb3V0cHV0X2RpciwKICAgICAgICBpZ25vcmU9c2h1dGlsLmlnbm9yZV9wYXR0ZXJucygiT0JKIiwgIkxLSCIsICIqLm8iKSwKICAgICkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiT0JKIikubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGNoZWQgPSBwYXRjaF9maW5kdG91cihvcmlnaW5hbF9ieXRlcy5kZWNvZGUoInV0Zi04IikpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIkZpbmRUb3VyLmMiKS53cml0ZV90ZXh0KHBhdGNoZWQpCiAgICBob29rID0gUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJCZXRhVG91clJlcGFpckhvb2suaCIpCiAgICBzaHV0aWwuY29weTIoaG9vaywgb3V0cHV0X2RpciAvICJTUkMiIC8gIkJldGFUb3VySG9vay5oIikKICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgIFsibWFrZSIsICItaiIsIHN0cihqb2JzKV0sCiAgICAgICAgY3dkPW91dHB1dF9kaXIsCiAgICAgICAgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwKICAgICAgICBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgdGV4dD1UcnVlLAogICAgICAgIGNoZWNrPUZhbHNlLAogICAgKQogICAgaWYgcmVzdWx0LnJldHVybmNvZGU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTEtIIGhvb2sgYnVpbGQgZmFpbGVkOlxue3Jlc3VsdC5zdGRvdXRbLTYwMDA6XX0iKQogICAgYmluYXJ5ID0gb3V0cHV0X2RpciAvICJMS0giCiAgICBpZiBub3QgYmluYXJ5LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUGF0Y2hlZCBiaW5hcnkgbWlzc2luZzoge2JpbmFyeX0iKQogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgInVwc3RyZWFtX2ZpbmR0b3VyX3NoYTI1NiI6IGRpZ2VzdCwKICAgICAgICAicGF0Y2hfaGVhZGVyX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGhvb2sucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGJpbmFyeS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgfQogICAgKG91dHB1dF9kaXIgLyAiYmV0YXRvdXJfYnVpbGQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIpICsgIlxuIgogICAgKQogICAgcmV0dXJuIGJpbmFyeQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWpvYnMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHByaW50KGJ1aWxkKGFyZ3Muc291cmNlX2RpciwgYXJncy5vdXRwdXRfZGlyLCBhcmdzLmpvYnMpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK', 'v10_repair/repair_engine.py': 'IiIiQnVkZ2V0ZWQgdG91ciByZXBhaXIgYnkgZXhhY3RseSByZWNvbm5lY3Rpbmcgc2VsZWN0ZWQgZml4ZWQgc2VnbWVudHMuCgpUaGUgY3V0IGVkZ2VzIG1heSBiZSBzZWxlY3RlZCBmcm9tIHNwYXRpYWxseSByZWxhdGVkIGJ1dCB0b3VyLWRpc3RhbnQgcGFydHMgb2YKdGhlIGN1cnJlbnQgc29sdXRpb24uICBBbGwgdW5jdXQgZWRnZXMgcmVtYWluIGludGFjdC4gIEhlbGQtLUthcnAgZHluYW1pYwpwcm9ncmFtbWluZyB0aGVuIGZpbmRzIHRoZSBjaGVhcGVzdCBjeWNsaWMgb3JkZXIgYW5kIG9yaWVudGF0aW9uIG9mIHRoZSBmaXhlZApzZWdtZW50cyB1bmRlciBUU1BMSUIgRVVDXzJEIGNvc3RzLiAgVGhlIG9yaWdpbmFsIHRvdXIgaXMgYWx3YXlzIGZlYXNpYmxlLCBhbmQKb25seSBhIHN0cmljdGx5IHNob3J0ZXIgdG91ciBpcyByZXR1cm5lZC4KClRoaXMgaXMgZGVsaWJlcmF0ZWx5IGEgc21hbGwsIGF1ZGl0YWJsZSAqbWVjaGFuaXNtIHBpbG90Ki4gIEl0IGRvZXMgbm90IGNsYWltCnRoYXQgYSBuZXVyYWwgc2VsZWN0b3IgaGVscHMsIG9yIHRoYXQgc2VnbWVudCByZWNvbm5lY3Rpb24gYmVhdHMgTEtILiAgVGhlCnNhbWUgb3B0aW1pemVyIGFuZCBjdXQgYnVkZ2V0IGNhbiBiZSB1c2VkIHdpdGggcmFuZG9tLCBsZW5ndGgsIHRvdXItZGlzYWdyZWVtZW50CmFuZCBtb2RlbC1iYXNlZCBzZWxlY3RvcnMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCByZXBsYWNlCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgTWFwcGluZywgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKCk1vZGVsU2NvcmVzID0gbnAubmRhcnJheSB8IENhbGxhYmxlW1tucC5uZGFycmF5XSwgbnAubmRhcnJheV0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBSZXBhaXJSZXN1bHQ6CiAgICB0b3VyOiBucC5uZGFycmF5CiAgICBjb3N0X2JlZm9yZTogaW50CiAgICBjb3N0X2FmdGVyOiBpbnQKICAgIGF0dGVtcHRzOiBpbnQKICAgIGFjY2VwdGVkX21vdmVzOiBpbnQKICAgIGVsYXBzZWRfczogZmxvYXQKICAgIGN1dHM6IHR1cGxlW2ludCwgLi4uXSA9ICgpCgogICAgQHByb3BlcnR5CiAgICBkZWYgaW1wcm92ZW1lbnQoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLmNvc3RfYmVmb3JlIC0gc2VsZi5jb3N0X2FmdGVyCgoKZGVmIF92YWxpZGF0ZWRfeHlfdG91cih4eTogbnAubmRhcnJheSwgdG91cjogbnAubmRhcnJheSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICB4eSA9IG5wLmFzYXJyYXkoeHksIGR0eXBlPW5wLmZsb2F0NjQpCiAgICB0b3VyID0gbnAuYXNhcnJheSh0b3VyLCBkdHlwZT1ucC5pbnQzMikKICAgIGlmIHh5Lm5kaW0gIT0gMiBvciB4eS5zaGFwZVsxXSAhPSAyIG9yIGxlbih4eSkgPCA0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInh5IG11c3QgaGF2ZSBzaGFwZSAobiwgMiksIG4gPj0gNCIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoeHkpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInh5IG11c3QgYmUgZmluaXRlIikKICAgIGlmIHRvdXIuc2hhcGUgIT0gKGxlbih4eSksKSBvciBub3QgbnAuYXJyYXlfZXF1YWwobnAuc29ydCh0b3VyKSwgbnAuYXJhbmdlKGxlbih4eSkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0b3VyIG11c3QgYmUgYSB6ZXJvLWJhc2VkIHBlcm11dGF0aW9uIG9mIGFsbCBub2RlcyIpCiAgICByZXR1cm4geHksIHRvdXIKCgpkZWYgZXVjMmRfbGVuZ3RocyhhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiVFNQTElCIEVVQ18yRDogbmVhcmVzdCBpbnRlZ2VyLCB3aXRoIC41IHJvdW5kZWQgdXB3YXJkcy4iIiIKICAgIGEsIGIgPSBucC5hc2FycmF5KGEsIGR0eXBlPW5wLmZsb2F0NjQpLCBucC5hc2FycmF5KGIsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICByZXR1cm4gbnAuZmxvb3IobnAuc3FydChucC5zdW0oKGEgLSBiKSAqKiAyLCBheGlzPS0xKSkgKyAwLjUpLmFzdHlwZShucC5pbnQ2NCkKCgpkZWYgdG91cl9jb3N0KHh5OiBucC5uZGFycmF5LCB0b3VyOiBucC5uZGFycmF5KSAtPiBpbnQ6CiAgICAiIiJJbnRlZ2VyIEVVQ18yRCBsZW5ndGggb2YgYSBjeWNsaWMgdG91ci4iIiIKICAgIHh5LCB0b3VyID0gX3ZhbGlkYXRlZF94eV90b3VyKHh5LCB0b3VyKQogICAgcmV0dXJuIGludChldWMyZF9sZW5ndGhzKHh5W3RvdXJdLCB4eVtucC5yb2xsKHRvdXIsIC0xKV0pLnN1bShkdHlwZT1ucC5pbnQ2NCkpCgoKZGVmIF9lZGdlX2tleXModG91cjogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgIG4gPSBsZW4odG91cikKICAgIG54dCA9IG5wLnJvbGwodG91ciwgLTEpLmFzdHlwZShucC5pbnQ2NCkKICAgIGEgPSB0b3VyLmFzdHlwZShucC5pbnQ2NCkKICAgIHJldHVybiBucC5taW5pbXVtKGEsIG54dCkgKiBucC5pbnQ2NChuKSArIG5wLm1heGltdW0oYSwgbnh0KQoKCmRlZiBfZW5zZW1ibGVfY291bnRzKGVuc2VtYmxlX3RvdXJzOiBTZXF1ZW5jZVtucC5uZGFycmF5XSwgbjogaW50KSAtPiBkaWN0W2ludCwgaW50XToKICAgIGlmIGxlbihlbnNlbWJsZV90b3VycykgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkaXNhZ3JlZW1lbnQgcmVxdWlyZXMgYXQgbGVhc3Qgb25lIGVuc2VtYmxlIHRvdXIiKQogICAgY291bnRzOiBkaWN0W2ludCwgaW50XSA9IHt9CiAgICBmb3IgaXRlbSBpbiBlbnNlbWJsZV90b3VyczoKICAgICAgICB0ID0gbnAuYXNhcnJheShpdGVtLCBkdHlwZT1ucC5pbnQzMikKICAgICAgICBpZiB0LnNoYXBlICE9IChuLCkgb3Igbm90IG5wLmFycmF5X2VxdWFsKG5wLnNvcnQodCksIG5wLmFyYW5nZShuKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2ZXJ5IGVuc2VtYmxlIHRvdXIgbXVzdCBiZSBhIHplcm8tYmFzZWQgcGVybXV0YXRpb24iKQogICAgICAgIGZvciBrZXkgaW4gX2VkZ2Vfa2V5cyh0KToKICAgICAgICAgICAgayA9IGludChrZXkpCiAgICAgICAgICAgIGNvdW50c1trXSA9IGNvdW50cy5nZXQoaywgMCkgKyAxCiAgICByZXR1cm4gY291bnRzCgoKZGVmIF9yaXNrX3Njb3JlcygKICAgIHN0cmF0ZWd5OiBzdHIsCiAgICB4eTogbnAubmRhcnJheSwKICAgIHRvdXI6IG5wLm5kYXJyYXksCiAgICBlbnNlbWJsZV90b3VyczogU2VxdWVuY2VbbnAubmRhcnJheV0gfCBNYXBwaW5nW2ludCwgaW50XSB8IE5vbmUsCiAgICBtb2RlbF9zY29yZXM6IE1vZGVsU2NvcmVzIHwgTm9uZSwKKSAtPiBucC5uZGFycmF5OgogICAgbiA9IGxlbih0b3VyKQogICAgaWYgc3RyYXRlZ3kgPT0gInJhbmRvbSI6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKG4sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpZiBzdHJhdGVneSA9PSAibG9uZ2VzdCI6CiAgICAgICAgcmV0dXJuIGV1YzJkX2xlbmd0aHMoeHlbdG91cl0sIHh5W25wLnJvbGwodG91ciwgLTEpXSkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBpZiBzdHJhdGVneSA9PSAiZGlzYWdyZWVtZW50IjoKICAgICAgICBpZiBlbnNlbWJsZV90b3VycyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkaXNhZ3JlZW1lbnQgcmVxdWlyZXMgZW5zZW1ibGVfdG91cnMiKQogICAgICAgIGNvdW50cyA9IChlbnNlbWJsZV90b3VycyBpZiBpc2luc3RhbmNlKGVuc2VtYmxlX3RvdXJzLCBNYXBwaW5nKQogICAgICAgICAgICAgICAgICBlbHNlIF9lbnNlbWJsZV9jb3VudHMoZW5zZW1ibGVfdG91cnMsIG4pKQogICAgICAgIGRlbm9taW5hdG9yID0gbWF4KGNvdW50cy52YWx1ZXMoKSkgaWYgY291bnRzIGVsc2UgMQogICAgICAgIHJldHVybiBucC5mcm9taXRlcigoMS4wIC0gY291bnRzLmdldChpbnQoayksIDApIC8gZGVub21pbmF0b3IKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrIGluIF9lZGdlX2tleXModG91cikpLCBkdHlwZT1ucC5mbG9hdDY0LCBjb3VudD1uKQogICAgaWYgc3RyYXRlZ3kgaW4gKCJzY29yZSIsICJtb2RlbCIpOgogICAgICAgIGlmIG1vZGVsX3Njb3JlcyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbCBzZWxlY3RvciByZXF1aXJlcyBtb2RlbF9zY29yZXMiKQogICAgICAgIGlmIGNhbGxhYmxlKG1vZGVsX3Njb3Jlcyk6CiAgICAgICAgICAgIHNjb3JlcyA9IG5wLmFzYXJyYXkobW9kZWxfc2NvcmVzKHRvdXIuY29weSgpKSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBieV9ub2RlID0gbnAuYXNhcnJheShtb2RlbF9zY29yZXMsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICAgICAgICAgIGlmIGJ5X25vZGUuc2hhcGUgIT0gKG4sKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0YXRpYyBtb2RlbF9zY29yZXMgbXVzdCBiZSBub2RlLWluZGV4ZWQgc2hhcGUgKG4sKSIpCiAgICAgICAgICAgIHNjb3JlcyA9IGJ5X25vZGVbdG91cl0KICAgICAgICBpZiBzY29yZXMuc2hhcGUgIT0gKG4sKSBvciBub3QgbnAuaXNmaW5pdGUoc2NvcmVzKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWxfc2NvcmVzIG11c3QgeWllbGQgZmluaXRlIHJpc2sgYnkgdG91ciBwb3NpdGlvbiIpCiAgICAgICAgcmV0dXJuIHNjb3JlcwogICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gY3V0IHNlbGVjdGlvbiBzdHJhdGVneToge3N0cmF0ZWd5fSIpCgoKZGVmIF9jaXJjdWxhcl9nYXAoYTogaW50LCBiOiBpbnQsIG46IGludCkgLT4gaW50OgogICAgcmV0dXJuIG1pbigoYSAtIGIpICUgbiwgKGIgLSBhKSAlIG4pCgoKZGVmIHNlbGVjdF9jdXRzKAogICAgc3RyYXRlZ3k6IHN0ciwKICAgIHh5OiBucC5uZGFycmF5LAogICAgdG91cjogbnAubmRhcnJheSwKICAgIGVuc2VtYmxlX3RvdXJzOiBTZXF1ZW5jZVtucC5uZGFycmF5XSB8IE1hcHBpbmdbaW50LCBpbnRdIHwgTm9uZSA9IE5vbmUsCiAgICBtb2RlbF9zY29yZXM6IE1vZGVsU2NvcmVzIHwgTm9uZSA9IE5vbmUsCiAgICBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IgfCBOb25lID0gTm9uZSwKICAgIGs6IGludCA9IDgsCiAgICBtaW5fc2VnbWVudF9zaXplOiBpbnQgPSA4LAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDaG9vc2Ugc3BhdGlhbGx5IG5lYXJieSBhbmQgdG91ci1kaXN0YW50IGN1dCBlZGdlcy4KCiAgICBgYG1vZGVsX3Njb3Jlc2BgIGFyZSAqcmlzayogc2NvcmVzOiBoaWdoZXIgbWVhbnMgbW9yZSBkZXNpcmFibGUgdG8gY3V0LgogICAgQSBjYWxsYWJsZSBpcyBldmFsdWF0ZWQgb24gdGhlIGN1cnJlbnQgdG91ciBhbmQgcmV0dXJucyBzY29yZXMgYnkgdG91cgogICAgcG9zaXRpb24uICBBIHN0YXRpYyBhcnJheSBpcyBpbmRleGVkIGJ5IHNvdXJjZSBub2RlIElELCBzbyBpdCByZXByZXNlbnRzCiAgICBub2RlLWxldmVsIHJpc2sgcmF0aGVyIHRoYW4gYSBmcm96ZW4gcG9zaXRpb24gb3IgZWRnZSBwcm9iYWJpbGl0eS4KICAgIGBgZW5zZW1ibGVfdG91cnNgYCBhcmUgemVyby1iYXNlZCB0b3VycyBvciBwcmVjb21wdXRlZCB1bmRpcmVjdGVkIGtleSBjb3VudHMuCiAgICBUaGUgc2FtZSBob3RzcG90IGdlb21ldHJ5IGlzIHVzZWQgZm9yIGV2ZXJ5IHN0cmF0ZWd5OyB0aGUgc3RyYXRlZ3kgY2hhbmdlcwogICAgb25seSB0aGUgcmlzayBzY29yZSB0aGF0IGJpYXNlcyB0aGUgYW5jaG9yIGFuZCBwYXJ0bmVyIGNob2ljZXMuCiAgICAiIiIKICAgIHh5LCB0b3VyID0gX3ZhbGlkYXRlZF94eV90b3VyKHh5LCB0b3VyKQogICAgbiA9IGxlbih0b3VyKQogICAgaWYgbm90IDMgPD0gayA8PSAxMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJrIG11c3QgYmUgYmV0d2VlbiAzIGFuZCAxMCIpCiAgICBpZiBtaW5fc2VnbWVudF9zaXplIDwgMSBvciBuIDwgayAqIG1pbl9zZWdtZW50X3NpemU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5mZWFzaWJsZSBtaW5pbXVtIHNlZ21lbnQgc2l6ZSIpCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoKSBpZiBybmcgaXMgTm9uZSBlbHNlIHJuZwogICAgcmlzayA9IF9yaXNrX3Njb3JlcyhzdHJhdGVneSwgeHksIHRvdXIsIGVuc2VtYmxlX3RvdXJzLCBtb2RlbF9zY29yZXMpCgogICAgIyBSb2J1c3QgWzAsMV0gbm9ybWFsaXphdGlvbiBtYWtlcyBHdW1iZWwgcGVydHVyYmF0aW9ucyBjb21wYXJhYmxlIGFjcm9zcwogICAgIyBsZW5ndGgsIGRpc2FncmVlbWVudCBhbmQgbGVhcm5lZCBzY29yZXMgd2l0aG91dCBhc3N1bWluZyBjYWxpYnJhdGlvbi4KICAgIGxvLCBoaSA9IGZsb2F0KG5wLnF1YW50aWxlKHJpc2ssIDAuMDUpKSwgZmxvYXQobnAucXVhbnRpbGUocmlzaywgMC45NSkpCiAgICBpZiBoaSA8PSBsbzoKICAgICAgICBub3JtYWxpemVkID0gbnAuemVyb3MobiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGVsc2U6CiAgICAgICAgbm9ybWFsaXplZCA9IG5wLmNsaXAoKHJpc2sgLSBsbykgLyAoaGkgLSBsbyksIDAuMCwgMS4wKQogICAgaWYgc3RyYXRlZ3kgPT0gInJhbmRvbSI6CiAgICAgICAgYW5jaG9yID0gaW50KHJuZy5pbnRlZ2VycyhuKSkKICAgIGVsc2U6CiAgICAgICAgYW5jaG9yID0gaW50KG5wLmFyZ21heChub3JtYWxpemVkICsgMC4zMCAqIHJuZy5ndW1iZWwoc2l6ZT1uKSkpCgogICAgbWlkcG9pbnQgPSAoeHlbdG91cl0gKyB4eVtucC5yb2xsKHRvdXIsIC0xKV0pICogMC41CiAgICBzcXVhcmVkID0gbnAuc3VtKChtaWRwb2ludCAtIG1pZHBvaW50W2FuY2hvcl0pICoqIDIsIGF4aXM9MSkKICAgIG5lYXJieSA9IG5wLmFyZ3NvcnQoc3F1YXJlZCwga2luZD0ic3RhYmxlIikKICAgIGNob3NlbiA9IFthbmNob3JdCiAgICBwb29sX3NpemUgPSBtaW4obiwgbWF4KDEyOCwgMzIgKiBrKSkKICAgIHdoaWxlIGxlbihjaG9zZW4pIDwgazoKICAgICAgICBwb29sID0gbmVhcmJ5Wzpwb29sX3NpemVdCiAgICAgICAgcmFuayA9IG5wLmFyYW5nZShsZW4ocG9vbCksIGR0eXBlPW5wLmZsb2F0NjQpCiAgICAgICAgIyBSYW5kb20gcGlja3MgcmV0YWluIHRoZSBzYW1lIHNwYXRpYWwgcG9vbCBhcyBsZWFybmVkIHNlbGVjdG9ycy4KICAgICAgICBwcmlvcml0eSA9IG5vcm1hbGl6ZWRbcG9vbF0gKyAwLjM1ICogcm5nLmd1bWJlbChzaXplPWxlbihwb29sKSkgLSAwLjMwICogcmFuayAvIGxlbihwb29sKQogICAgICAgIGZvciBwb3NpdGlvbiBpbiBwb29sW25wLmFyZ3NvcnQoLXByaW9yaXR5LCBraW5kPSJzdGFibGUiKV06CiAgICAgICAgICAgIHBvc2l0aW9uID0gaW50KHBvc2l0aW9uKQogICAgICAgICAgICBpZiBwb3NpdGlvbiBpbiBjaG9zZW46CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBhbGwoX2NpcmN1bGFyX2dhcChwb3NpdGlvbiwgb3RoZXIsIG4pID49IG1pbl9zZWdtZW50X3NpemUgZm9yIG90aGVyIGluIGNob3Nlbik6CiAgICAgICAgICAgICAgICBjaG9zZW4uYXBwZW5kKHBvc2l0aW9uKQogICAgICAgICAgICAgICAgaWYgbGVuKGNob3NlbikgPT0gazoKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIGxlbihjaG9zZW4pID09IGs6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaWYgcG9vbF9zaXplID09IG46CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiZmFpbGVkIHRvIHNlbGVjdCBmZWFzaWJsZSBjdXRzIikKICAgICAgICBwb29sX3NpemUgPSBtaW4obiwgcG9vbF9zaXplICogMikKICAgIHJldHVybiBucC5zb3J0KG5wLmFzYXJyYXkoY2hvc2VuLCBkdHlwZT1ucC5pbnQzMikpCgoKZGVmIF9zZWdtZW50c19mcm9tX2N1dHModG91cjogbnAubmRhcnJheSwgY3V0czogbnAubmRhcnJheSkgLT4gbGlzdFtucC5uZGFycmF5XToKICAgIG4gPSBsZW4odG91cikKICAgIG91dCA9IFtdCiAgICBmb3IgaWR4LCBjdXQgaW4gZW51bWVyYXRlKGN1dHMpOgogICAgICAgIGJlZ2luID0gKGludChjdXQpICsgMSkgJSBuCiAgICAgICAgZW5kID0gaW50KGN1dHNbKGlkeCArIDEpICUgbGVuKGN1dHMpXSkKICAgICAgICBpZiBiZWdpbiA8PSBlbmQ6CiAgICAgICAgICAgIHBhcnQgPSB0b3VyW2JlZ2luOmVuZCArIDFdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcGFydCA9IG5wLmNvbmNhdGVuYXRlKCh0b3VyW2JlZ2luOl0sIHRvdXJbOmVuZCArIDFdKSkKICAgICAgICBvdXQuYXBwZW5kKHBhcnQpCiAgICBpZiBzdW0obGVuKHApIGZvciBwIGluIG91dCkgIT0gbjoKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiY3V0IGRlY29tcG9zaXRpb24gZGlkIG5vdCBjb3ZlciB0aGUgdG91ciIpCiAgICByZXR1cm4gb3V0CgoKZGVmIHJlcGFpcl90b3VyKHh5OiBucC5uZGFycmF5LCB0b3VyOiBucC5uZGFycmF5LCBjdXRzOiBTZXF1ZW5jZVtpbnRdKSAtPiBSZXBhaXJSZXN1bHQ6CiAgICAiIiJFeGFjdGx5IG9wdGltaXplIGN5Y2xpYyByZWNvbm5lY3Rpb24gb2YgdGhlIGZpeGVkIHNlZ21lbnRzLgoKICAgIEN1dCBpbmRpY2VzIHJlZmVyIHRvIGVkZ2VzIGBgdG91cltpXSAtPiB0b3VyWyhpKzEpICUgbl1gYC4gIFNlZ21lbnQgemVybyBpcwogICAgaGVsZCBpbiBpdHMgZm9yd2FyZCBvcmllbnRhdGlvbjsgZm9yIGEgc3ltbWV0cmljIFRTUCB0aGlzIGxvc2VzIG5vIGN5Y2xlcywKICAgIGJlY2F1c2UgYW55IGN5Y2xlIHdpdGggc2VnbWVudCB6ZXJvIHJldmVyc2VkIGNhbiBiZSByZXZlcnNlZCBnbG9iYWxseS4KICAgICIiIgogICAgYmVnYW4gPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICB4eSwgdG91ciA9IF92YWxpZGF0ZWRfeHlfdG91cih4eSwgdG91cikKICAgIG4gPSBsZW4odG91cikKICAgIGN1dHMgPSBucC5hc2FycmF5KGN1dHMsIGR0eXBlPW5wLmludDMyKQogICAgaWYgY3V0cy5uZGltICE9IDEgb3Igbm90IDMgPD0gbGVuKGN1dHMpIDw9IDEwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImN1dHMgbXVzdCBjb250YWluIDMgdG8gMTAgcG9zaXRpb25zIikKICAgIGlmIG5wLmFueSgoY3V0cyA8IDApIHwgKGN1dHMgPj0gbikpIG9yIGxlbihucC51bmlxdWUoY3V0cykpICE9IGxlbihjdXRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjdXQgcG9zaXRpb25zIG11c3QgYmUgdW5pcXVlIGFuZCB3aXRoaW4gdGhlIHRvdXIiKQogICAgY3V0cyA9IG5wLnNvcnQoY3V0cykKICAgIHNlZ21lbnRzID0gX3NlZ21lbnRzX2Zyb21fY3V0cyh0b3VyLCBjdXRzKQogICAgayA9IGxlbihzZWdtZW50cykKICAgIGJlZm9yZSA9IHRvdXJfY29zdCh4eSwgdG91cikKCiAgICBzdGFydHMgPSBucC5hc2FycmF5KFtbcGFydFswXSwgcGFydFstMV1dIGZvciBwYXJ0IGluIHNlZ21lbnRzXSwgZHR5cGU9bnAuaW50MzIpCiAgICBlbmRzID0gbnAuYXNhcnJheShbW3BhcnRbLTFdLCBwYXJ0WzBdXSBmb3IgcGFydCBpbiBzZWdtZW50c10sIGR0eXBlPW5wLmludDMyKQogICAgY29ubmVjdG9yID0gbnAuZW1wdHkoKGssIDIsIGssIDIpLCBkdHlwZT1ucC5pbnQ2NCkKICAgIGZvciBhIGluIHJhbmdlKGspOgogICAgICAgIGZvciBvYSBpbiByYW5nZSgyKToKICAgICAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoayk6CiAgICAgICAgICAgICAgICBjb25uZWN0b3JbYSwgb2EsIGJdID0gZXVjMmRfbGVuZ3RocygKICAgICAgICAgICAgICAgICAgICB4eVtlbmRzW2EsIG9hXV0sIHh5W3N0YXJ0c1tiXV0KICAgICAgICAgICAgICAgICkKCiAgICBmdWxsID0gKDEgPDwgaykgLSAxCiAgICBpbmZpbml0eSA9IG5wLmlpbmZvKG5wLmludDY0KS5tYXggLy8gOAogICAgZHAgPSBucC5mdWxsKCgxIDw8IGssIGssIDIpLCBpbmZpbml0eSwgZHR5cGU9bnAuaW50NjQpCiAgICBwYXJlbnRfbGFzdCA9IG5wLmZ1bGwoKDEgPDwgaywgaywgMiksIC0xLCBkdHlwZT1ucC5pbnQ4KQogICAgcGFyZW50X29yaSA9IG5wLmZ1bGwoKDEgPDwgaywgaywgMiksIC0xLCBkdHlwZT1ucC5pbnQ4KQogICAgZHBbMSwgMCwgMF0gPSAwCiAgICBmb3IgbWFzayBpbiByYW5nZSgxLCBmdWxsICsgMSwgMik6ICAjIGJpdCAwIGlzIGFsd2F5cyBwcmVzZW50CiAgICAgICAgZm9yIGxhc3QgaW4gcmFuZ2Uoayk6CiAgICAgICAgICAgIGlmIG5vdCAobWFzayAmICgxIDw8IGxhc3QpKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBvcmkgaW4gcmFuZ2UoMik6CiAgICAgICAgICAgICAgICBjdXJyZW50ID0gaW50KGRwW21hc2ssIGxhc3QsIG9yaV0pCiAgICAgICAgICAgICAgICBpZiBjdXJyZW50ID09IGluZmluaXR5OgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBmb3Igbnh0IGluIHJhbmdlKDEsIGspOgogICAgICAgICAgICAgICAgICAgIGlmIG1hc2sgJiAoMSA8PCBueHQpOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIG5ld19tYXNrID0gbWFzayB8ICgxIDw8IG54dCkKICAgICAgICAgICAgICAgICAgICBmb3IgbmV4dF9vcmkgaW4gcmFuZ2UoMik6CiAgICAgICAgICAgICAgICAgICAgICAgIHZhbHVlID0gY3VycmVudCArIGludChjb25uZWN0b3JbbGFzdCwgb3JpLCBueHQsIG5leHRfb3JpXSkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgdmFsdWUgPCBkcFtuZXdfbWFzaywgbnh0LCBuZXh0X29yaV06CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBkcFtuZXdfbWFzaywgbnh0LCBuZXh0X29yaV0gPSB2YWx1ZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFyZW50X2xhc3RbbmV3X21hc2ssIG54dCwgbmV4dF9vcmldID0gbGFzdAogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFyZW50X29yaVtuZXdfbWFzaywgbnh0LCBuZXh0X29yaV0gPSBvcmkKCiAgICBiZXN0X2Nvbm5lY3RvcnMgPSBpbmZpbml0eQogICAgYmVzdF9sYXN0ID0gYmVzdF9vcmkgPSAtMQogICAgZm9yIGxhc3QgaW4gcmFuZ2UoMSwgayk6CiAgICAgICAgZm9yIG9yaSBpbiByYW5nZSgyKToKICAgICAgICAgICAgdmFsdWUgPSBpbnQoZHBbZnVsbCwgbGFzdCwgb3JpXSArIGNvbm5lY3RvcltsYXN0LCBvcmksIDAsIDBdKQogICAgICAgICAgICBpZiB2YWx1ZSA8IGJlc3RfY29ubmVjdG9yczoKICAgICAgICAgICAgICAgIGJlc3RfY29ubmVjdG9ycywgYmVzdF9sYXN0LCBiZXN0X29yaSA9IHZhbHVlLCBsYXN0LCBvcmkKCiAgICBvcmRlcjogbGlzdFt0dXBsZVtpbnQsIGludF1dID0gW10KICAgIG1hc2ssIGxhc3QsIG9yaSA9IGZ1bGwsIGJlc3RfbGFzdCwgYmVzdF9vcmkKICAgIHdoaWxlIG1hc2sgIT0gMToKICAgICAgICBvcmRlci5hcHBlbmQoKGxhc3QsIG9yaSkpCiAgICAgICAgcHJpb3JfbGFzdCwgcHJpb3Jfb3JpID0gaW50KHBhcmVudF9sYXN0W21hc2ssIGxhc3QsIG9yaV0pLCBpbnQocGFyZW50X29yaVttYXNrLCBsYXN0LCBvcmldKQogICAgICAgIG1hc2sgXj0gMSA8PCBsYXN0CiAgICAgICAgbGFzdCwgb3JpID0gcHJpb3JfbGFzdCwgcHJpb3Jfb3JpCiAgICBvcmRlci5hcHBlbmQoKDAsIDApKQogICAgb3JkZXIucmV2ZXJzZSgpCiAgICByZWJ1aWx0ID0gbnAuY29uY2F0ZW5hdGUoW3NlZ21lbnRzW3NdIGlmIG8gPT0gMCBlbHNlIHNlZ21lbnRzW3NdWzo6LTFdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBzLCBvIGluIG9yZGVyXSkuYXN0eXBlKG5wLmludDMyLCBjb3B5PUZhbHNlKQogICAgYWZ0ZXIgPSB0b3VyX2Nvc3QoeHksIHJlYnVpbHQpCiAgICBpZiBhZnRlciA+IGJlZm9yZToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiZXhhY3QgcmVjb25uZWN0aW9uIHdvcnNlbmVkIHRoZSBvcmlnaW5hbCB0b3VyIikKICAgICMgQW4gZXF1aXZhbGVudCBvcmllbnRhdGlvbi9wZXJtdXRhdGlvbiBuZWVkbGVzc2x5IGNoYW5nZXMgdGhlIHRvdXIgb3JkZXIuCiAgICBpZiBhZnRlciA9PSBiZWZvcmU6CiAgICAgICAgcmVidWlsdCA9IHRvdXIuY29weSgpCiAgICByZXR1cm4gUmVwYWlyUmVzdWx0KHJlYnVpbHQsIGJlZm9yZSwgYWZ0ZXIsIDEsIGludChhZnRlciA8IGJlZm9yZSksCiAgICAgICAgICAgICAgICAgICAgICAgIHRpbWUucGVyZl9jb3VudGVyKCkgLSBiZWdhbiwgdHVwbGUobWFwKGludCwgY3V0cykpKQoKCmRlZiByZXBlYXRlZF9yZXBhaXIoCiAgICB4eTogbnAubmRhcnJheSwKICAgIHRvdXI6IG5wLm5kYXJyYXksCiAgICAqLAogICAgc3RyYXRlZ3k6IHN0ciwKICAgIGVuc2VtYmxlX3RvdXJzOiBTZXF1ZW5jZVtucC5uZGFycmF5XSB8IE5vbmUgPSBOb25lLAogICAgbW9kZWxfc2NvcmVzOiBNb2RlbFNjb3JlcyB8IE5vbmUgPSBOb25lLAogICAgc2VlZDogaW50ID0gMCwKICAgIGs6IGludCA9IDgsCiAgICBhdHRlbXB0czogaW50ID0gMTYsCiAgICBtYXhfc2Vjb25kczogZmxvYXQgPSA1LjAsCiAgICBtaW5fc2VnbWVudF9zaXplOiBpbnQgPSA4LAopIC0+IFJlcGFpclJlc3VsdDoKICAgICIiIlRyeSBzZXZlcmFsIG1hdGNoZWQgc3BhdGlhbCByZXBhaXIgb3Bwb3J0dW5pdGllcyB3aXRoaW4gYSB3YWxsIGNhcC4KCiAgICBUaGUgY2FwIGlzIGNoZWNrZWQgYmVmb3JlIGVhY2ggYm91bmRlZCBrPD0xMCBkeW5hbWljLXByb2dyYW1taW5nIHNvbHZlLgogICAgSW5kaXZpZHVhbCBzb2x2ZXMgbWF5IGV4Y2VlZCB0aGUgY2FwIHNsaWdodGx5OyBjYWxsZXJzIHNob3VsZCByZXNlcnZlIGEKICAgIHNhZmV0eSBtYXJnaW4gaW4gdGhlaXIgdG90YWwgZXhwZXJpbWVudCBidWRnZXQuCiAgICAiIiIKICAgIGJlZ2FuID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgeHksIGN1cnJlbnQgPSBfdmFsaWRhdGVkX3h5X3RvdXIoeHksIHRvdXIpCiAgICBpZiBhdHRlbXB0cyA8IDAgb3IgbWF4X3NlY29uZHMgPCAwIG9yIG5vdCBtYXRoLmlzZmluaXRlKG1heF9zZWNvbmRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhdHRlbXB0cyBhbmQgZmluaXRlIG1heF9zZWNvbmRzIG11c3QgYmUgbm9ubmVnYXRpdmUiKQogICAgaW5pdGlhbF9jb3N0ID0gdG91cl9jb3N0KHh5LCBjdXJyZW50KQogICAgYmVzdF9jb3N0ID0gaW5pdGlhbF9jb3N0CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgIGNvdW50czogTWFwcGluZ1tpbnQsIGludF0gfCBOb25lID0gTm9uZQogICAgaWYgc3RyYXRlZ3kgPT0gImRpc2FncmVlbWVudCI6CiAgICAgICAgaWYgZW5zZW1ibGVfdG91cnMgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGlzYWdyZWVtZW50IHJlcXVpcmVzIGVuc2VtYmxlX3RvdXJzIikKICAgICAgICBjb3VudHMgPSBfZW5zZW1ibGVfY291bnRzKGVuc2VtYmxlX3RvdXJzLCBsZW4oY3VycmVudCkpCiAgICB0cmllZCA9IGFjY2VwdGVkID0gMAogICAgZmluYWxfY3V0czogdHVwbGVbaW50LCAuLi5dID0gKCkKICAgIHdoaWxlIHRyaWVkIDwgYXR0ZW1wdHMgYW5kIHRpbWUucGVyZl9jb3VudGVyKCkgLSBiZWdhbiA8IG1heF9zZWNvbmRzOgogICAgICAgIGN1dHMgPSBzZWxlY3RfY3V0cyhzdHJhdGVneSwgeHksIGN1cnJlbnQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGVuc2VtYmxlX3RvdXJzPWNvdW50cyBpZiBjb3VudHMgaXMgbm90IE5vbmUgZWxzZSBlbnNlbWJsZV90b3VycywKICAgICAgICAgICAgICAgICAgICAgICAgICAgbW9kZWxfc2NvcmVzPW1vZGVsX3Njb3Jlcywgcm5nPXJuZywgaz1rLAogICAgICAgICAgICAgICAgICAgICAgICAgICBtaW5fc2VnbWVudF9zaXplPW1pbl9zZWdtZW50X3NpemUpCiAgICAgICAgdHJpYWwgPSByZXBhaXJfdG91cih4eSwgY3VycmVudCwgY3V0cykKICAgICAgICB0cmllZCArPSAxCiAgICAgICAgaWYgdHJpYWwuY29zdF9hZnRlciA8IGJlc3RfY29zdDoKICAgICAgICAgICAgY3VycmVudCwgYmVzdF9jb3N0ID0gdHJpYWwudG91ciwgdHJpYWwuY29zdF9hZnRlcgogICAgICAgICAgICBhY2NlcHRlZCArPSAxCiAgICAgICAgICAgIGZpbmFsX2N1dHMgPSB0cmlhbC5jdXRzCiAgICByZXR1cm4gUmVwYWlyUmVzdWx0KGN1cnJlbnQuY29weSgpLCBpbml0aWFsX2Nvc3QsIGJlc3RfY29zdCwgdHJpZWQsIGFjY2VwdGVkLAogICAgICAgICAgICAgICAgICAgICAgICB0aW1lLnBlcmZfY291bnRlcigpIC0gYmVnYW4sIGZpbmFsX2N1dHMpCg==', 'v10_repair/compressed_repair.py': 'IiIiRXhhY3QgdG91ci1wcmVzZXJ2aW5nIGNvbXByZXNzaW9uIGZvbGxvd2VkIGJ5IGEgc21hbGwgTEtIIHJlcGFpci4KClNlbGVjdGVkIHRvdXIgd2luZG93cyBhcmUgZnJlZS4gRXZlcnkgdW50b3VjaGVkIHBhdGggaXMgcmVwcmVzZW50ZWQgYnkgaXRzIHR3bwplbmRwb2ludHMgam9pbmVkIGJ5IGEgbWFuZGF0b3J5IHplcm8tY29zdCBlZGdlIGluIHRoZSByZWR1Y2VkIFRTUC4gRXhwYW5kaW5nCnRoYXQgZWRnZSByZXN0b3JlcyB0aGUgZnVsbCBwYXRoIGluIGVpdGhlciBvcmllbnRhdGlvbi4gVGh1cyBldmVyeSByZWR1Y2VkCkhhbWlsdG9uIHRvdXIgdGhhdCBvYmV5cyB0aGUgbWFuZGF0b3J5IGVkZ2VzIGV4cGFuZHMgdG8gYSB2YWxpZCBmdWxsIHRvdXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHNodXRpbAppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCByZXBsYWNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIHJlcGFpcl9lbmdpbmUgaW1wb3J0IHNlbGVjdF9jdXRzLCB0b3VyX2Nvc3QKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDb21wcmVzc2VkVG91cjoKICAgIHh5OiBucC5uZGFycmF5CiAgICBvcmlnaW5hbF9ub2RlczogbnAubmRhcnJheQogICAgaW5pdGlhbF90b3VyOiBucC5uZGFycmF5CiAgICBmaXhlZF9lZGdlczogbnAubmRhcnJheQogICAgZml4ZWRfcGF0aHM6IGRpY3RbdHVwbGVbaW50LCBpbnRdLCBucC5uZGFycmF5XQogICAgZnVsbF9uOiBpbnQKICAgIGZyZWVfY291bnQ6IGludAoKICAgIGRlZiBleHBhbmQoc2VsZiwgcmVkdWNlZF90b3VyOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgICAgIHJvdXRlID0gbnAuYXNhcnJheShyZWR1Y2VkX3RvdXIsIGR0eXBlPW5wLmludDMyKQogICAgICAgIG0gPSBsZW4oc2VsZi5vcmlnaW5hbF9ub2RlcykKICAgICAgICBpZiByb3V0ZS5zaGFwZSAhPSAobSwpIG9yIG5vdCBucC5hcnJheV9lcXVhbChucC5zb3J0KHJvdXRlKSwgbnAuYXJhbmdlKG0pKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVkdWNlZCB0b3VyIG11c3QgYmUgYSBIYW1pbHRvbiBwZXJtdXRhdGlvbiIpCiAgICAgICAgZXhwYW5kZWQ6IGxpc3RbaW50XSA9IFtdCiAgICAgICAgdXNlZF9maXhlZDogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgICAgIGZvciBpbmRleCwgcmlkIGluIGVudW1lcmF0ZShyb3V0ZSk6CiAgICAgICAgICAgIHJpZCA9IGludChyaWQpCiAgICAgICAgICAgIG54dCA9IGludChyb3V0ZVsoaW5kZXggKyAxKSAlIG1dKQogICAgICAgICAgICBleHBhbmRlZC5hcHBlbmQoaW50KHNlbGYub3JpZ2luYWxfbm9kZXNbcmlkXSkpCiAgICAgICAgICAgIGtleSA9IChtaW4ocmlkLCBueHQpLCBtYXgocmlkLCBueHQpKQogICAgICAgICAgICBwYXRoID0gc2VsZi5maXhlZF9wYXRocy5nZXQoa2V5KQogICAgICAgICAgICBpZiBwYXRoIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB1c2VkX2ZpeGVkLmFkZChrZXkpCiAgICAgICAgICAgIGlmIGV4cGFuZGVkWy0xXSA9PSBpbnQocGF0aFswXSkgYW5kIGludChzZWxmLm9yaWdpbmFsX25vZGVzW254dF0pID09IGludChwYXRoWy0xXSk6CiAgICAgICAgICAgICAgICBleHBhbmRlZC5leHRlbmQobWFwKGludCwgcGF0aFsxOi0xXSkpCiAgICAgICAgICAgIGVsaWYgZXhwYW5kZWRbLTFdID09IGludChwYXRoWy0xXSkgYW5kIGludChzZWxmLm9yaWdpbmFsX25vZGVzW254dF0pID09IGludChwYXRoWzBdKToKICAgICAgICAgICAgICAgIGV4cGFuZGVkLmV4dGVuZChtYXAoaW50LCBwYXRoWy0yOjA6LTFdKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJtYW5kYXRvcnkgcGF0aCBlbmRwb2ludHMgaW5jb25zaXN0ZW50IikKICAgICAgICBpZiB1c2VkX2ZpeGVkICE9IHNldChzZWxmLmZpeGVkX3BhdGhzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVkdWNlZCB0b3VyIG9taXR0ZWQgYSBtYW5kYXRvcnkgcGF0aCBlZGdlIikKICAgICAgICBvdXQgPSBucC5hc2FycmF5KGV4cGFuZGVkLCBkdHlwZT1ucC5pbnQzMikKICAgICAgICBpZiBvdXQuc2hhcGUgIT0gKHNlbGYuZnVsbF9uLCkgb3Igbm90IG5wLmFycmF5X2VxdWFsKG5wLnNvcnQob3V0KSwgbnAuYXJhbmdlKHNlbGYuZnVsbF9uKSk6CiAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJleHBhbmRlZCB0b3VyIGlzIG5vdCBIYW1pbHRvbmlhbiIpCiAgICAgICAgcmV0dXJuIG91dAoKCmRlZiBmcmVlX3dpbmRvd3MobjogaW50LCBjdXRzOiBucC5uZGFycmF5LCB3aWR0aDogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiTWFyayB3aWR0aCB0b3VyIHBvc2l0aW9ucyBhcm91bmQgZXZlcnkgY3V0LCBpbmNsdWRpbmcgYm90aCBlbmRwb2ludHMuIiIiCiAgICBpZiB3aWR0aCA8IDIgb3Igd2lkdGggPj0gbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ3aW5kb3cgd2lkdGggbXVzdCBiZSBpbiBbMixuKSIpCiAgICBmcmVlID0gbnAuemVyb3MobiwgZHR5cGU9Ym9vbCkKICAgIGxlZnQgPSB3aWR0aCAvLyAyIC0gMQogICAgZm9yIGN1dCBpbiBucC5hc2FycmF5KGN1dHMsIGR0eXBlPW5wLmludDMyKToKICAgICAgICBpZiBub3QgMCA8PSBpbnQoY3V0KSA8IG46CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImN1dCBvdXQgb2YgcmFuZ2UiKQogICAgICAgIGZyZWVbKGludChjdXQpICsgbnAuYXJhbmdlKC1sZWZ0LCB3aWR0aCAtIGxlZnQpKSAlIG5dID0gVHJ1ZQogICAgcmV0dXJuIGZyZWUKCgpkZWYgY29tcHJlc3NfdG91cih4eTogbnAubmRhcnJheSwgdG91cjogbnAubmRhcnJheSwgZnJlZTogbnAubmRhcnJheSkgLT4gQ29tcHJlc3NlZFRvdXI6CiAgICB4eSA9IG5wLmFzYXJyYXkoeHkpCiAgICB0b3VyID0gbnAuYXNhcnJheSh0b3VyLCBkdHlwZT1ucC5pbnQzMikKICAgIGZyZWUgPSBucC5hc2FycmF5KGZyZWUsIGR0eXBlPWJvb2wpCiAgICBuID0gbGVuKHRvdXIpCiAgICBpZiB4eS5zaGFwZSAhPSAobiwgMikgb3IgZnJlZS5zaGFwZSAhPSAobiwpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInh5L3RvdXIvZnJlZSBkaW1lbnNpb24gbWlzbWF0Y2giKQogICAgaWYgbm90IG5wLmlzZmluaXRlKHh5KS5hbGwoKSBvciBub3QgbnAuYXJyYXlfZXF1YWwoeHksIG5wLnJpbnQoeHkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFVUNfMkQgY29tcHJlc3Npb24gcmVxdWlyZXMgZmluaXRlIGludGVncmFsIGNvb3JkaW5hdGVzIikKICAgIGlmIG5vdCBucC5hcnJheV9lcXVhbChucC5zb3J0KHRvdXIpLCBucC5hcmFuZ2UobikpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgZnVsbCB0b3VyIikKICAgIGlmIGZyZWUuc3VtKCkgPCAzIG9yIGZyZWUuYWxsKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibmVlZCBhdCBsZWFzdCB0aHJlZSBmcmVlIG5vZGVzIGFuZCBvbmUgZml4ZWQgbm9kZSIpCgogICAgZml4ZWQgPSB+ZnJlZQogICAgc3RhcnRzID0gbnAuZmxhdG5vbnplcm8oZml4ZWQgJiBucC5yb2xsKGZyZWUsIDEpKQogICAgcGF0aHM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAga2VlcCA9IGZyZWUuY29weSgpCiAgICBmb3Igc3RhcnQgaW4gc3RhcnRzOgogICAgICAgIHBvc2l0aW9ucyA9IFtdCiAgICAgICAgcG9zID0gaW50KHN0YXJ0KQogICAgICAgIHdoaWxlIGZpeGVkW3Bvc106CiAgICAgICAgICAgIHBvc2l0aW9ucy5hcHBlbmQocG9zKQogICAgICAgICAgICBwb3MgPSAocG9zICsgMSkgJSBuCiAgICAgICAgcGF0aCA9IHRvdXJbbnAuYXNhcnJheShwb3NpdGlvbnMsIGR0eXBlPW5wLmludDMyKV0KICAgICAgICBwYXRocy5hcHBlbmQocGF0aCkKICAgICAgICBrZWVwW3Bvc2l0aW9uc1swXV0gPSBUcnVlCiAgICAgICAga2VlcFtwb3NpdGlvbnNbLTFdXSA9IFRydWUKCiAgICBvcmlnaW5hbF9ub2RlcyA9IHRvdXJba2VlcF0uYXN0eXBlKG5wLmludDMyKQogICAgb2xkX3RvX25ldyA9IHtpbnQob2xkKTogaSBmb3IgaSwgb2xkIGluIGVudW1lcmF0ZShvcmlnaW5hbF9ub2Rlcyl9CiAgICBwYWlyX3BhdGhzOiBkaWN0W3R1cGxlW2ludCwgaW50XSwgbnAubmRhcnJheV0gPSB7fQogICAgZml4ZWRfcGFpcnMgPSBbXQogICAgZm9yIHBhdGggaW4gcGF0aHM6CiAgICAgICAgaWYgbGVuKHBhdGgpIDwgMjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBhLCBiID0gb2xkX3RvX25ld1tpbnQocGF0aFswXSldLCBvbGRfdG9fbmV3W2ludChwYXRoWy0xXSldCiAgICAgICAga2V5ID0gKG1pbihhLCBiKSwgbWF4KGEsIGIpKQogICAgICAgIGlmIGtleSBpbiBwYWlyX3BhdGhzOgogICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiZHVwbGljYXRlIG1hbmRhdG9yeSBwYWlyIikKICAgICAgICBwYWlyX3BhdGhzW2tleV0gPSBwYXRoLmNvcHkoKQogICAgICAgIGZpeGVkX3BhaXJzLmFwcGVuZCgoYSwgYikpCiAgICBwYWlycyA9IG5wLmFzYXJyYXkoZml4ZWRfcGFpcnMsIGR0eXBlPW5wLmludDMyKS5yZXNoYXBlKC0xLCAyKQogICAgcmVkdWNlZCA9IENvbXByZXNzZWRUb3VyKAogICAgICAgIHh5PW5wLmFzYXJyYXkoeHlbb3JpZ2luYWxfbm9kZXNdLCBkdHlwZT1ucC5pbnQ2NCksCiAgICAgICAgb3JpZ2luYWxfbm9kZXM9b3JpZ2luYWxfbm9kZXMsCiAgICAgICAgaW5pdGlhbF90b3VyPW5wLmFyYW5nZShsZW4ob3JpZ2luYWxfbm9kZXMpLCBkdHlwZT1ucC5pbnQzMiksCiAgICAgICAgZml4ZWRfZWRnZXM9cGFpcnMsCiAgICAgICAgZml4ZWRfcGF0aHM9cGFpcl9wYXRocywKICAgICAgICBmdWxsX249biwKICAgICAgICBmcmVlX2NvdW50PWludChmcmVlLnN1bSgpKSwKICAgICkKICAgIHJlY29uc3RydWN0ZWQgPSByZWR1Y2VkLmV4cGFuZChyZWR1Y2VkLmluaXRpYWxfdG91cikKICAgIGlmIHRvdXJfY29zdCh4eSwgcmVjb25zdHJ1Y3RlZCkgIT0gdG91cl9jb3N0KHh5LCB0b3VyKToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiY29tcHJlc3Npb24gZGlkIG5vdCBwcmVzZXJ2ZSBvcmlnaW5hbCBjb3N0IikKICAgIHJldHVybiByZWR1Y2VkCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUmVwYWlyQXR0ZW1wdDoKICAgIHRvdXI6IG5wLm5kYXJyYXkKICAgIGJlZm9yZTogaW50CiAgICBhZnRlcjogaW50CiAgICByZWR1Y2VkX246IGludAogICAgZnJlZV9uOiBpbnQKICAgIGVsYXBzZWRfczogZmxvYXQKICAgIHNvbHZlcl9yZXR1cm5jb2RlOiBpbnQgfCBOb25lCiAgICBzb2x2ZXJfdGltZWRfb3V0OiBib29sCiAgICBpbXByb3ZlZDogYm9vbAogICAgY3V0czogdHVwbGVbaW50LCAuLi5dID0gKCkKCgpkZWYgc29sdmVfcmVkdWNlZCgKICAgIHBhY2tlZDogQ29tcHJlc3NlZFRvdXIsCiAgICBmdWxsX3h5OiBucC5uZGFycmF5LAogICAgZnVsbF90b3VyOiBucC5uZGFycmF5LAogICAgbGtoX2JpbmFyeTogUGF0aCwKICAgIHdvcmtfcm9vdDogUGF0aCwKICAgICosCiAgICBzZWVkOiBpbnQsCiAgICBtYXhfdHJpYWxzOiBpbnQgPSAyMCwKICAgIHRpbWVvdXRfczogZmxvYXQgPSAyLjAsCikgLT4gUmVwYWlyQXR0ZW1wdDoKICAgICIiIlNvbHZlIG9uZSBlcXVpdmFsZW50IHJlZHVjZWQgVFNQOyByZXRhaW4gdGhlIG9yaWdpbmFsIGlmIG5vIGdhaW4uIiIiCiAgICBpbXBvcnQgdHNwbGFiIGFzIFQKCiAgICBiZWdhbiA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGJlZm9yZSA9IHRvdXJfY29zdChmdWxsX3h5LCBmdWxsX3RvdXIpCiAgICBpZiB0aW1lb3V0X3MgPD0gMCBvciBtYXhfdHJpYWxzIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwb3NpdGl2ZSBzb2x2ZXIgYnVkZ2V0IGFuZCB0cmlhbHMgcmVxdWlyZWQiKQogICAgd29ya19yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGRpcmVjdG9yeSA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9InJlcGFpcl8iLCBkaXI9d29ya19yb290KSkKICAgIHRyeToKICAgICAgICBwbGFpbiA9IGRpcmVjdG9yeSAvICJwbGFpbi50c3AiCiAgICAgICAgcHJvYmxlbSA9IGRpcmVjdG9yeSAvICJmaXhlZC50c3AiCiAgICAgICAgaW5pdCA9IGRpcmVjdG9yeSAvICJpbml0aWFsLnRvdXIiCiAgICAgICAgVC53cml0ZV90c3AocGxhaW4sIHBhY2tlZC54eSwgbmFtZT0iQmV0YVRvdXJfcmVwYWlyIiwgZXc9IkVVQ18yRCIpCiAgICAgICAgaWYgbGVuKHBhY2tlZC5maXhlZF9lZGdlcyk6CiAgICAgICAgICAgIFQuYWRkX2ZpeGVkX2VkZ2VzKHBsYWluLCBwcm9ibGVtLCBwYWNrZWQuZml4ZWRfZWRnZXMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJvYmxlbSA9IHBsYWluCiAgICAgICAgVC53cml0ZV90b3VyKGluaXQsIHBhY2tlZC5pbml0aWFsX3RvdXIpCiAgICAgICAgcGFyYW1zID0gewogICAgICAgICAgICAiQ0FORElEQVRFX1NFVF9UWVBFIjogIkFMUEhBIiwKICAgICAgICAgICAgIk1BWF9DQU5ESURBVEVTIjogNSwKICAgICAgICAgICAgIlNVQkdSQURJRU5UIjogIk5PIiwKICAgICAgICAgICAgIklOSVRJQUxfVE9VUl9GSUxFIjogc3RyKGluaXQpLAogICAgICAgICAgICAiU0VFRCI6IGludChzZWVkKSwKICAgICAgICAgICAgIlJVTlMiOiAxLAogICAgICAgICAgICAiTUFYX1RSSUFMUyI6IGludChtYXhfdHJpYWxzKSwKICAgICAgICAgICAgIlRJTUVfTElNSVQiOiBtYXgoMS4wLCB0aW1lb3V0X3MpLAogICAgICAgICAgICAiVFJBQ0VfTEVWRUwiOiAwLAogICAgICAgICAgICAiU1RPUF9BVF9PUFRJTVVNIjogIk5PIiwKICAgICAgICB9CiAgICAgICAgcmVzdWx0ID0gVC5ydW5fbGtoKHN0cihsa2hfYmluYXJ5KSwgc3RyKHByb2JsZW0pLCBwYXJhbXMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0cihkaXJlY3RvcnkgLyAicnVuIiksIHRpbWVvdXQ9dGltZW91dF9zKQogICAgICAgIGNhbmRpZGF0ZSA9IHJlc3VsdFsidG91ciJdCiAgICAgICAgYWZ0ZXIgPSBiZWZvcmUKICAgICAgICBzZWxlY3RlZCA9IG5wLmFzYXJyYXkoZnVsbF90b3VyLCBkdHlwZT1ucC5pbnQzMikuY29weSgpCiAgICAgICAgaWYgY2FuZGlkYXRlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjYW5kaWRhdGUgPSBwYWNrZWQuZXhwYW5kKGNhbmRpZGF0ZSkKICAgICAgICAgICAgY29zdCA9IHRvdXJfY29zdChmdWxsX3h5LCBjYW5kaWRhdGUpCiAgICAgICAgICAgIGlmIGNvc3QgPCBiZWZvcmU6CiAgICAgICAgICAgICAgICBzZWxlY3RlZCwgYWZ0ZXIgPSBjYW5kaWRhdGUsIGNvc3QKICAgICAgICByZXR1cm4gUmVwYWlyQXR0ZW1wdChzZWxlY3RlZCwgYmVmb3JlLCBhZnRlciwgbGVuKHBhY2tlZC5vcmlnaW5hbF9ub2RlcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFja2VkLmZyZWVfY291bnQsIHRpbWUucGVyZl9jb3VudGVyKCkgLSBiZWdhbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICByZXN1bHRbInJldHVybmNvZGUiXSwgYm9vbChyZXN1bHRbInRpbWVkX291dCJdKSwgYWZ0ZXIgPCBiZWZvcmUpCiAgICBmaW5hbGx5OgogICAgICAgIHNodXRpbC5ybXRyZWUoZGlyZWN0b3J5LCBpZ25vcmVfZXJyb3JzPVRydWUpCgoKZGVmIHJlcGFpcl9vbmNlKAogICAgeHk6IG5wLm5kYXJyYXksCiAgICB0b3VyOiBucC5uZGFycmF5LAogICAgbGtoX2JpbmFyeTogUGF0aCwKICAgIHdvcmtfcm9vdDogUGF0aCwKICAgICosCiAgICBzdHJhdGVneTogc3RyLAogICAgZW5zZW1ibGVfdG91cnM6IGxpc3RbbnAubmRhcnJheV0gfCBOb25lLAogICAgbW9kZWxfc2NvcmVzLAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgY3V0X2NvdW50OiBpbnQgPSAzLAogICAgd2luZG93X3NpemU6IGludCA9IDQ4LAogICAgdGltZW91dF9zOiBmbG9hdCA9IDIuMCwKICAgIG1heF90cmlhbHM6IGludCA9IDIwLAopIC0+IFJlcGFpckF0dGVtcHQ6CiAgICBjdXRzID0gc2VsZWN0X2N1dHMoc3RyYXRlZ3ksIHh5LCB0b3VyLCBlbnNlbWJsZV90b3Vycz1lbnNlbWJsZV90b3VycywKICAgICAgICAgICAgICAgICAgICAgICBtb2RlbF9zY29yZXM9bW9kZWxfc2NvcmVzLCBybmc9cm5nLCBrPWN1dF9jb3VudCwKICAgICAgICAgICAgICAgICAgICAgICBtaW5fc2VnbWVudF9zaXplPXdpbmRvd19zaXplKQogICAgZnJlZSA9IGZyZWVfd2luZG93cyhsZW4odG91ciksIGN1dHMsIHdpbmRvd19zaXplKQogICAgcGFja2VkID0gY29tcHJlc3NfdG91cih4eSwgdG91ciwgZnJlZSkKICAgIHJlc3VsdCA9IHNvbHZlX3JlZHVjZWQocGFja2VkLCB4eSwgdG91ciwgbGtoX2JpbmFyeSwgd29ya19yb290LAogICAgICAgICAgICAgICAgICAgICAgICAgICBzZWVkPWludChybmcuaW50ZWdlcnMoMSwgMioqMzEgLSAxKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF90cmlhbHM9bWF4X3RyaWFscywgdGltZW91dF9zPXRpbWVvdXRfcykKICAgIHJldHVybiByZXBsYWNlKHJlc3VsdCwgY3V0cz10dXBsZShtYXAoaW50LCBjdXRzKSkpCg==', 'v10_repair/repair_pilot.py': 'IiIiU2l4LXNlZWQsIHdhbGwtY2xvY2stY2hhcmdlZCBCZXRhVG91ciBjb21wcmVzc2VkLXJlcGFpciBwaWxvdCBydW5uZXIuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gY29tcHJlc3NlZF9yZXBhaXIgaW1wb3J0IHJlcGFpcl9vbmNlCmZyb20gY29vcmRpbmF0b3IgaW1wb3J0IFNuYXBzaG90LCBub29wX2FjdGlvbnMsIHJlYWRfdG91ciwgcnVuX3NpeApmcm9tIHJlcGFpcl9lbmdpbmUgaW1wb3J0IHRvdXJfY29zdAoKClJFUEFJUl9BUk1TID0gKCJyZXBhaXJfcmFuZG9tIiwgInJlcGFpcl9sb25nZXN0IiwgInJlcGFpcl9kaXNhZ3JlZW1lbnQiLCAicmVwYWlyX2JldGF0b3VyIikKTElWRV9BUk1TID0gKCJzaGFtIiwpICsgUkVQQUlSX0FSTVMKCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHZhbHVlOiBkaWN0KSAtPiBOb25lOgogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiwgYWxsb3dfbmFuPUZhbHNlKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX2F0b21pY190b3VyKHBhdGg6IFBhdGgsIHRvdXI6IG5wLm5kYXJyYXkpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnRtcCIpCiAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChzdHIobGVuKHRvdXIpKSArICJcbiIgKyAiXG4iLmpvaW4obWFwKHN0ciwgdG91ciArIDEpKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX2hhc2hfdG91cih0b3VyOiBucC5uZGFycmF5KSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYobnAuYXNhcnJheSh0b3VyLCBkdHlwZT1ucC5pbnQzMikudG9ieXRlcygpKS5oZXhkaWdlc3QoKQoKCmRlZiBfbW9kZWxfZWRnZV9yaXNrKGdyYXBoOiBucC5uZGFycmF5LCBwcm9iYWJpbGl0eTogbnAubmRhcnJheSk6CiAgICBncmFwaCA9IG5wLmFzYXJyYXkoZ3JhcGgsIGR0eXBlPW5wLmludDMyKQogICAgcHJvYmFiaWxpdHkgPSBucC5hc2FycmF5KHByb2JhYmlsaXR5LCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgZ3JhcGguc2hhcGUgIT0gcHJvYmFiaWxpdHkuc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgZ3JhcGgvcHJvYmFiaWxpdHkgc2hhcGUgbWlzbWF0Y2giKQoKICAgIGRlZiByaXNrX2J5X3Bvc2l0aW9uKHRvdXI6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgdG91ciA9IG5wLmFzYXJyYXkodG91ciwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgc3VjY2Vzc29yID0gbnAucm9sbCh0b3VyLCAtMSkKICAgICAgICBmb3J3YXJkID0gbnAubWF4KG5wLndoZXJlKGdyYXBoW3RvdXJdID09IHN1Y2Nlc3Nvcls6LCBOb25lXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb2JhYmlsaXR5W3RvdXJdLCAwLjApLCBheGlzPTEpCiAgICAgICAgYmFja3dhcmQgPSBucC5tYXgobnAud2hlcmUoZ3JhcGhbc3VjY2Vzc29yXSA9PSB0b3VyWzosIE5vbmVdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb2JhYmlsaXR5W3N1Y2Nlc3Nvcl0sIDAuMCksIGF4aXM9MSkKICAgICAgICByZXR1cm4gKDEuMCAtIG5wLm1heGltdW0oZm9yd2FyZCwgYmFja3dhcmQpKS5hc3R5cGUobnAuZmxvYXQ2NCkKCiAgICByZXR1cm4gcmlza19ieV9wb3NpdGlvbgoKCmRlZiBfdmFsaWRfY29zdChULCB4eTogbnAubmRhcnJheSwgdG91cjogbnAubmRhcnJheSkgLT4gaW50OgogICAgaWYgbm90IFQuaXNfdmFsaWRfdG91cih0b3VyLCBsZW4oeHkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIEhhbWlsdG9uIHRvdXIiKQogICAgYSA9IHRvdXJfY29zdCh4eSwgdG91cikKICAgIGIgPSBpbnQoVC50b3VyX2xlbmd0aCh4eSwgdG91ciwgIkVVQ18yRCIpKQogICAgaWYgYSAhPSBiOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiaW5kZXBlbmRlbnQgRVVDXzJEIGNoZWNrcyBkaXNhZ3JlZToge2F9ICE9IHtifSIpCiAgICByZXR1cm4gYQoKCmRlZiBydW5fcmVwYWlyX2FybSgKICAgIGtpbmQ6IHN0ciwKICAgIHBhdGNoZWRfYmluYXJ5OiBQYXRoLAogICAgc2lkZWNhcl9iaW5hcnk6IFBhdGgsCiAgICBwcm9ibGVtOiBQYXRoLAogICAgeHk6IG5wLm5kYXJyYXksCiAgICB3b3JrX2RpcjogUGF0aCwKICAgIHNlZWRzOiBsaXN0W2ludF0sCiAgICBidWRnZXRfczogZmxvYXQsCiAgICBtb2RlbCwKICAgIGRldmljZSwKICAgIFQsCiAgICBNLAogICAgKiwKICAgIGF0dGVtcHRzX3Blcl9zZWVkOiBpbnQgPSA4LAogICAgY3V0X2NvdW50OiBpbnQgPSAzLAogICAgd2luZG93X3NpemU6IGludCA9IDI1NiwKICAgIHNpZGVjYXJfdGltZW91dF9zOiBmbG9hdCA9IDIuMCwKICAgIHNpZGVjYXJfdHJpYWxzOiBpbnQgPSAxMDAsCiAgICByZXBhaXJfZnJhY3Rpb25fY2FwOiBmbG9hdCA9IDAuMzAsCiAgICBiYXJyaWVyX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMzUsCikgLT4gZGljdDoKICAgIGlmIGtpbmQgbm90IGluIExJVkVfQVJNUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBsaXZlIHJlcGFpciBhcm06IHtraW5kfSIpCiAgICBpZiBsZW4oc2VlZHMpICE9IDYgb3IgbGVuKHNldChzZWVkcykpICE9IDY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXhhY3RseSBzaXggdW5pcXVlIHNlZWRzIHJlcXVpcmVkIikKICAgIGlmIGF0dGVtcHRzX3Blcl9zZWVkIDwgMSBvciBjdXRfY291bnQgPCAzIG9yIHdpbmRvd19zaXplIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIHJlcGFpciBzY2hlZHVsZSIpCiAgICBpZiBub3QgMCA8IHJlcGFpcl9mcmFjdGlvbl9jYXAgPCAwLjggb3Igbm90IDAgPD0gYmFycmllcl9mcmFjdGlvbiA8IDAuOCBvciBidWRnZXRfcyA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgd2FsbCBidWRnZXQgb3IgcmVwYWlyIGNhcCIpCiAgICB4eSA9IG5wLmFzYXJyYXkoeHkpCiAgICBzdHJhdGVneSA9ICJtb2RlbCIgaWYga2luZCA9PSAicmVwYWlyX2JldGF0b3VyIiBlbHNlIGtpbmQucmVtb3ZlcHJlZml4KCJyZXBhaXJfIikKICAgIG91dGVyX3N0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgb3V0ZXJfZGVhZGxpbmUgPSBvdXRlcl9zdGFydCArIGJ1ZGdldF9zCiAgICBhdWRpdF9wYXRoID0gd29ya19kaXIgLyAicmVwYWlyX21ldGEuanNvbiIKICAgIGNhbGxiYWNrX21ldGE6IGRpY3QgPSB7fQoKICAgIGRlZiBjYWxsYmFjayhzbmFwc2hvdHM6IGxpc3RbU25hcHNob3RdKToKICAgICAgICBpZiBsZW4oc25hcHNob3RzKSAhPSA2OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbmNvbXBsZXRlIGZpcnN0LXRvdXIgYmFycmllciIpCiAgICAgICAgZmlyc3QgPSBbbnAuYXNhcnJheShzLmZpcnN0X3RvdXIsIGR0eXBlPW5wLmludDMyKSAtIDEgZm9yIHMgaW4gc25hcHNob3RzXQogICAgICAgIGZpcnN0X2Nvc3RzID0gW192YWxpZF9jb3N0KFQsIHh5LCB0KSBmb3IgdCBpbiBmaXJzdF0KICAgICAgICBiYXJyaWVyX3NlZW4gPSB0aW1lLm1vbm90b25pYygpIC0gb3V0ZXJfc3RhcnQKICAgICAgICBzdWNjZXNzb3JzID0gbnAuc3RhY2soW1QudG91cl9zdWNjX3ByZWQodClbMF0gZm9yIHQgaW4gZmlyc3RdLCBheGlzPTEpLmFzdHlwZShucC5pbnQzMikKICAgICAgICBwcmVkaWN0aW9uID0gTS5wcmVkaWN0X3RvdXJmb2xkKG1vZGVsLCB4eSwgMTYsIGRldmljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVuc19zdWNjPXN1Y2Nlc3NvcnMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbnNfY29zdD1ucC5hc2FycmF5KGZpcnN0X2Nvc3RzLCBkdHlwZT1ucC5mbG9hdDY0KSkKICAgICAgICByZXBhaXJfc3RvcCA9IG1pbih0aW1lLm1vbm90b25pYygpICsgYnVkZ2V0X3MgKiByZXBhaXJfZnJhY3Rpb25fY2FwLAogICAgICAgICAgICAgICAgICAgICAgICAgIG91dGVyX2RlYWRsaW5lIC0gNS4wKQogICAgICAgIG1vZGVsX3Jpc2sgPSBfbW9kZWxfZWRnZV9yaXNrKHByZWRpY3Rpb25bIk5CIl0sIHByZWRpY3Rpb25bInByb2IiXSkKICAgICAgICBjYWxsYmFja19tZXRhLnVwZGF0ZSh7CiAgICAgICAgICAgICJiYXJyaWVyX2Nvc3RzIjogZmlyc3RfY29zdHMsCiAgICAgICAgICAgICJiYXJyaWVyX3RvdXJfaGFzaGVzIjogW19oYXNoX3RvdXIodCkgZm9yIHQgaW4gZmlyc3RdLAogICAgICAgICAgICAiYmFycmllcl93YWxsX2Zyb21fc3RhcnRfcyI6IGJhcnJpZXJfc2VlbiwKICAgICAgICAgICAgIm1vZGVsX2luZmVyZW5jZV9zIjogZmxvYXQocHJlZGljdGlvblsidF90b3RhbCJdKSwKICAgICAgICAgICAgImF0dGVtcHRzIjoge3N0cihzZWVkKTogW10gZm9yIHNlZWQgaW4gc2VlZHN9LAogICAgICAgIH0pCiAgICAgICAgY3VycmVudCA9IFt0LmNvcHkoKSBmb3IgdCBpbiBmaXJzdF0KICAgICAgICBybmdzID0gW25wLnJhbmRvbS5kZWZhdWx0X3JuZyg4OTAxICsgc2VlZCkgZm9yIHNlZWQgaW4gc2VlZHNdCiAgICAgICAgcmVwYWlyX3N0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgIGlmIGtpbmQgIT0gInNoYW0iOgogICAgICAgICAgICAjIFJvdW5kLXJvYmluIGtlZXBzIHNlZWQgZWZmb3J0IGJhbGFuY2VkIGlmIHRoZSBjYXAgaXMgYXBwcm9hY2hlZC4KICAgICAgICAgICAgZm9yIHJvdW5kX2luZGV4IGluIHJhbmdlKGF0dGVtcHRzX3Blcl9zZWVkKToKICAgICAgICAgICAgICAgIGZvciBpbmRleCwgc2VlZCBpbiBlbnVtZXJhdGUoc2VlZHMpOgogICAgICAgICAgICAgICAgICAgIHJlbWFpbmluZyA9IHJlcGFpcl9zdG9wIC0gdGltZS5tb25vdG9uaWMoKQogICAgICAgICAgICAgICAgICAgIGlmIHJlbWFpbmluZyA8IDAuNToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJyZXBhaXIgd2FsbCBjYXAgcmVhY2hlZCBiZWZvcmUgYWxsIG1hdGNoZWQgYXR0ZW1wdHMiKQogICAgICAgICAgICAgICAgICAgIGF0dGVtcHQgPSByZXBhaXJfb25jZSgKICAgICAgICAgICAgICAgICAgICAgICAgeHksIGN1cnJlbnRbaW5kZXhdLCBzaWRlY2FyX2JpbmFyeSwgd29ya19kaXIgLyAic2lkZWNhcl90bXAiLAogICAgICAgICAgICAgICAgICAgICAgICBzdHJhdGVneT1zdHJhdGVneSwKICAgICAgICAgICAgICAgICAgICAgICAgZW5zZW1ibGVfdG91cnM9Zmlyc3QsCiAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsX3Njb3Jlcz1tb2RlbF9yaXNrIGlmIGtpbmQgPT0gInJlcGFpcl9iZXRhdG91ciIgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgICBybmc9cm5nc1tpbmRleF0sIGN1dF9jb3VudD1jdXRfY291bnQsIHdpbmRvd19zaXplPXdpbmRvd19zaXplLAogICAgICAgICAgICAgICAgICAgICAgICB0aW1lb3V0X3M9bWluKHNpZGVjYXJfdGltZW91dF9zLCBtYXgoMC4xLCByZW1haW5pbmcgLSAwLjI1KSksCiAgICAgICAgICAgICAgICAgICAgICAgIG1heF90cmlhbHM9c2lkZWNhcl90cmlhbHMsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgIGlmIGF0dGVtcHQuYWZ0ZXIgPiBhdHRlbXB0LmJlZm9yZToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoInJlcGFpciB3b3JzZW5lZCBhIHRvdXIiKQogICAgICAgICAgICAgICAgICAgIGN1cnJlbnRbaW5kZXhdID0gYXR0ZW1wdC50b3VyCiAgICAgICAgICAgICAgICAgICAgY2FsbGJhY2tfbWV0YVsiYXR0ZW1wdHMiXVtzdHIoc2VlZCldLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAgICAgICAgICJyb3VuZCI6IHJvdW5kX2luZGV4LAogICAgICAgICAgICAgICAgICAgICAgICAiY3V0cyI6IGxpc3QoYXR0ZW1wdC5jdXRzKSwKICAgICAgICAgICAgICAgICAgICAgICAgImJlZm9yZSI6IGF0dGVtcHQuYmVmb3JlLAogICAgICAgICAgICAgICAgICAgICAgICAiYWZ0ZXIiOiBhdHRlbXB0LmFmdGVyLAogICAgICAgICAgICAgICAgICAgICAgICAicmVkdWNlZF9uIjogYXR0ZW1wdC5yZWR1Y2VkX24sCiAgICAgICAgICAgICAgICAgICAgICAgICJmcmVlX24iOiBhdHRlbXB0LmZyZWVfbiwKICAgICAgICAgICAgICAgICAgICAgICAgImVsYXBzZWRfcyI6IGF0dGVtcHQuZWxhcHNlZF9zLAogICAgICAgICAgICAgICAgICAgICAgICAic29sdmVyX3JldHVybmNvZGUiOiBhdHRlbXB0LnNvbHZlcl9yZXR1cm5jb2RlLAogICAgICAgICAgICAgICAgICAgICAgICAic29sdmVyX3RpbWVkX291dCI6IGF0dGVtcHQuc29sdmVyX3RpbWVkX291dCwKICAgICAgICAgICAgICAgICAgICAgICAgImltcHJvdmVkIjogYXR0ZW1wdC5pbXByb3ZlZCwKICAgICAgICAgICAgICAgICAgICAgICAgImVuZF93YWxsX2Zyb21fc3RhcnRfcyI6IHRpbWUubW9ub3RvbmljKCkgLSBvdXRlcl9zdGFydCwKICAgICAgICAgICAgICAgICAgICB9KQogICAgICAgIGNhbGxiYWNrX21ldGFbInJlcGFpcl93YWxsX3MiXSA9IHRpbWUubW9ub3RvbmljKCkgLSByZXBhaXJfc3RhcnQKICAgICAgICBjYWxsYmFja19tZXRhWyJyZXBhaXJfY29zdHMiXSA9IFtfdmFsaWRfY29zdChULCB4eSwgdCkgZm9yIHQgaW4gY3VycmVudF0KICAgICAgICBjYWxsYmFja19tZXRhWyJjb21wbGV0ZWRfYXR0ZW1wdHNfcGVyX3NlZWQiXSA9IFtsZW4oY2FsbGJhY2tfbWV0YVsiYXR0ZW1wdHMiXVtzdHIocyldKSBmb3IgcyBpbiBzZWVkc10KICAgICAgICBmb3Igc2VlZCwgdG91ciBpbiB6aXAoc2VlZHMsIGN1cnJlbnQpOgogICAgICAgICAgICBfYXRvbWljX3RvdXIod29ya19kaXIgLyBmInNlZWRfe3NlZWR9IiAvICJyZXBhaXIudG91ciIsIHRvdXIpCiAgICAgICAgX2F0b21pY19qc29uKGF1ZGl0X3BhdGgsIGNhbGxiYWNrX21ldGEpCiAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBvdXRlcl9kZWFkbGluZSAtIDIuMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJubyB0aW1lIHJlbWFpbnMgdG8gcmVsZWFzZSBMS0ggYWN0aW9uIGJhcnJpZXIiKQogICAgICAgIHJldHVybiBbbm9vcF9hY3Rpb25zKGxlbih4eSkpIGZvciBfIGluIHNlZWRzXQoKICAgIHJlc3VsdCA9IHJ1bl9zaXgocGF0Y2hlZF9iaW5hcnksIHByb2JsZW0sIHdvcmtfZGlyLCBzZWVkcywgYnVkZ2V0X3MsCiAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrLCBleHRyYV9wYXJhbXM9eyJNQVhfQ0FORElEQVRFUyI6IDZ9LAogICAgICAgICAgICAgICAgICAgICBiYXJyaWVyX25hbWU9ImJhcnJpZXIudG91ciIsCiAgICAgICAgICAgICAgICAgICAgIGhvb2tfYWZ0ZXJfcz1idWRnZXRfcyAqIGJhcnJpZXJfZnJhY3Rpb24pCiAgICBpZiBub3QgcmVzdWx0WyJiYXJyaWVyX2NvbXBsZXRlZCJdIG9yIG5vdCBhdWRpdF9wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigicmVwYWlyIGJhcnJpZXIgb3IgYXVkaXQgaW5jb21wbGV0ZSIpCiAgICBpZiBhbnkocnVuWyJhcHBsaWVkX3N0YXRzIl0gaXMgTm9uZSBvciBydW5bImJlc3RfdG91ciJdIGlzIE5vbmUgZm9yIHJ1biBpbiByZXN1bHRbInJ1bnMiXSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJvbmUgb3IgbW9yZSBMS0ggcHJvY2Vzc2VzIGRpZCBub3QgcmVzdW1lIikKCiAgICBsa2hfY29zdHM6IGxpc3RbaW50XSA9IFtdCiAgICByZXBhaXJfY29zdHM6IGxpc3RbaW50XSA9IFtdCiAgICBoeWJyaWRfY29zdHM6IGxpc3RbaW50XSA9IFtdCiAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICBzZWVkX2RpciA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICBsa2hfdG91ciA9IG5wLmFzYXJyYXkocmVhZF90b3VyKHNlZWRfZGlyIC8gImJlc3QudG91ciIpLCBkdHlwZT1ucC5pbnQzMikgLSAxCiAgICAgICAgcmVwYWlyX3RvdXIgPSBucC5hc2FycmF5KHJlYWRfdG91cihzZWVkX2RpciAvICJyZXBhaXIudG91ciIpLCBkdHlwZT1ucC5pbnQzMikgLSAxCiAgICAgICAgbGtoX2Nvc3QgPSBfdmFsaWRfY29zdChULCB4eSwgbGtoX3RvdXIpCiAgICAgICAgcmVwYWlyX2Nvc3QgPSBfdmFsaWRfY29zdChULCB4eSwgcmVwYWlyX3RvdXIpCiAgICAgICAgbGtoX2Nvc3RzLmFwcGVuZChsa2hfY29zdCkKICAgICAgICByZXBhaXJfY29zdHMuYXBwZW5kKHJlcGFpcl9jb3N0KQogICAgICAgIGh5YnJpZF9jb3N0cy5hcHBlbmQobWluKGxraF9jb3N0LCByZXBhaXJfY29zdCkpCiAgICByZXN1bHQudXBkYXRlKAogICAgICAgIGFybT1raW5kLAogICAgICAgIGJhcnJpZXJfY29zdHM9Y2FsbGJhY2tfbWV0YVsiYmFycmllcl9jb3N0cyJdLAogICAgICAgIGJhcnJpZXJfdG91cl9oYXNoZXM9Y2FsbGJhY2tfbWV0YVsiYmFycmllcl90b3VyX2hhc2hlcyJdLAogICAgICAgIGxraF9jb3N0cz1sa2hfY29zdHMsCiAgICAgICAgcmVwYWlyX2Nvc3RzPXJlcGFpcl9jb3N0cywKICAgICAgICBzZWVkX2Nvc3RzPWh5YnJpZF9jb3N0cywKICAgICAgICBiZXN0X2Nvc3Q9bWluKGh5YnJpZF9jb3N0cyksCiAgICAgICAgbW9kZWxfaW5mZXJlbmNlX3M9Y2FsbGJhY2tfbWV0YVsibW9kZWxfaW5mZXJlbmNlX3MiXSwKICAgICAgICByZXBhaXJfd2FsbF9zPWNhbGxiYWNrX21ldGFbInJlcGFpcl93YWxsX3MiXSwKICAgICAgICBiYXJyaWVyX3dhbGxfZnJvbV9zdGFydF9zPWNhbGxiYWNrX21ldGFbImJhcnJpZXJfd2FsbF9mcm9tX3N0YXJ0X3MiXSwKICAgICAgICBjb21wbGV0ZWRfYXR0ZW1wdHNfcGVyX3NlZWQ9Y2FsbGJhY2tfbWV0YVsiY29tcGxldGVkX2F0dGVtcHRzX3Blcl9zZWVkIl0sCiAgICAgICAgd2FsbF9lbGFwc2VkX3M9cmVzdWx0WyJ3YWxsX2VsYXBzZWRfcyJdLAogICAgKQogICAgX2F0b21pY19qc29uKHdvcmtfZGlyIC8gImh5YnJpZF9yZXN1bHQuanNvbiIsIHsKICAgICAgICBrZXk6IHJlc3VsdFtrZXldIGZvciBrZXkgaW4gKAogICAgICAgICAgICAiYXJtIiwgImJhcnJpZXJfY29zdHMiLCAiYmFycmllcl90b3VyX2hhc2hlcyIsICJsa2hfY29zdHMiLCAicmVwYWlyX2Nvc3RzIiwKICAgICAgICAgICAgInNlZWRfY29zdHMiLCAiYmVzdF9jb3N0IiwgIm1vZGVsX2luZmVyZW5jZV9zIiwgInJlcGFpcl93YWxsX3MiLAogICAgICAgICAgICAiYmFycmllcl93YWxsX2Zyb21fc3RhcnRfcyIsCiAgICAgICAgICAgICJjb21wbGV0ZWRfYXR0ZW1wdHNfcGVyX3NlZWQiLCAid2FsbF9lbGFwc2VkX3MiKQogICAgfSkKICAgIHJldHVybiByZXN1bHQK', 'v10_repair/test_compressed_repair.py': 'IiIiRm9jdXNlZCBpbnZhcmlhbnRzIGZvciB0b3VyIGNvbXByZXNzaW9uIGFuZCB0aGUgZXhhY3QgRVVDXzJEIG9iamVjdGl2ZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKSEVSRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihIRVJFKSkKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihIRVJFLnBhcmVudHNbMV0gLyAic3JjIikpCgpmcm9tIGNvbXByZXNzZWRfcmVwYWlyIGltcG9ydCBjb21wcmVzc190b3VyLCBmcmVlX3dpbmRvd3MsIHJlcGFpcl9vbmNlICAjIG5vcWE6IEU0MDIKZnJvbSByZXBhaXJfZW5naW5lIGltcG9ydCB0b3VyX2Nvc3QgICMgbm9xYTogRTQwMgppbXBvcnQgdHNwbGFiIGFzIFQgICMgbm9xYTogRTQwMgoKCmRlZiBlZGdlcyh0b3VyOiBucC5uZGFycmF5KSAtPiBzZXRbdHVwbGVbaW50LCBpbnRdXToKICAgIHJldHVybiB7dHVwbGUoc29ydGVkKChpbnQodG91cltpXSksIGludCh0b3VyWyhpICsgMSkgJSBsZW4odG91cildKSkpKQogICAgICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4odG91cikpfQoKCmNsYXNzIENvbXByZXNzaW9uVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHNldFVwKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDcyMSkKICAgICAgICBzZWxmLnh5ID0gcm5nLmludGVnZXJzKDAsIDEwMDAwLCBzaXplPSgxMDAsIDIpLCBkdHlwZT1ucC5pbnQ2NCkKICAgICAgICBzZWxmLnRvdXIgPSBybmcucGVybXV0YXRpb24oMTAwKS5hc3R5cGUobnAuaW50MzIpCgogICAgZGVmIHRlc3RfaW5pdGlhbF90b3VyX2lzX2V4YWN0bHlfcHJlc2VydmVkX3VwX3RvX3JvdGF0aW9uKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgZnJlZSA9IGZyZWVfd2luZG93cygxMDAsIG5wLmFycmF5KFsyLCAzMSwgNjVdKSwgMTIpCiAgICAgICAgcGFja2VkID0gY29tcHJlc3NfdG91cihzZWxmLnh5LCBzZWxmLnRvdXIsIGZyZWUpCiAgICAgICAgcmVzdG9yZWQgPSBwYWNrZWQuZXhwYW5kKHBhY2tlZC5pbml0aWFsX3RvdXIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChlZGdlcyhyZXN0b3JlZCksIGVkZ2VzKHNlbGYudG91cikpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbCh0b3VyX2Nvc3Qoc2VsZi54eSwgcmVzdG9yZWQpLCB0b3VyX2Nvc3Qoc2VsZi54eSwgc2VsZi50b3VyKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHRvdXJfY29zdChzZWxmLnh5LCByZXN0b3JlZCksCiAgICAgICAgICAgICAgICAgICAgICAgICBpbnQoVC50b3VyX2xlbmd0aChzZWxmLnh5LCByZXN0b3JlZCwgIkVVQ18yRCIpKSkKCiAgICBkZWYgdGVzdF9tYW5kYXRvcnlfZWRnZV9leHBhbnNpb25fYWxsb3dzX3Jlb3JkZXJpbmcoc2VsZikgLT4gTm9uZToKICAgICAgICBmcmVlID0gZnJlZV93aW5kb3dzKDEwMCwgbnAuYXJyYXkoWzIsIDMxLCA2NV0pLCAxMikKICAgICAgICBwYWNrZWQgPSBjb21wcmVzc190b3VyKHNlbGYueHksIHNlbGYudG91ciwgZnJlZSkKICAgICAgICByb3V0ZSA9IHBhY2tlZC5pbml0aWFsX3RvdXIuY29weSgpCiAgICAgICAgIyBSZXZlcnNpbmcgdGhlIGVudGlyZSByZWR1Y2VkIGN5Y2xlIHJldGFpbnMgZXZlcnkgbWFuZGF0b3J5IHBhaXIuCiAgICAgICAgcm91dGUgPSByb3V0ZVs6Oi0xXS5jb3B5KCkKICAgICAgICByZXN0b3JlZCA9IHBhY2tlZC5leHBhbmQocm91dGUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4obnAudW5pcXVlKHJlc3RvcmVkKSksIDEwMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGVkZ2VzKHJlc3RvcmVkKSwgZWRnZXMoc2VsZi50b3VyKSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHRvdXJfY29zdChzZWxmLnh5LCByZXN0b3JlZCksIHRvdXJfY29zdChzZWxmLnh5LCBzZWxmLnRvdXIpKQoKICAgIGRlZiB0ZXN0X21pc3NpbmdfbWFuZGF0b3J5X3BhaXJfaXNfcmVqZWN0ZWQoc2VsZikgLT4gTm9uZToKICAgICAgICBmcmVlID0gZnJlZV93aW5kb3dzKDEwMCwgbnAuYXJyYXkoWzIsIDMxLCA2NV0pLCAxMikKICAgICAgICBwYWNrZWQgPSBjb21wcmVzc190b3VyKHNlbGYueHksIHNlbGYudG91ciwgZnJlZSkKICAgICAgICBhLCBiID0gbWFwKGludCwgcGFja2VkLmZpeGVkX2VkZ2VzWzBdKQogICAgICAgIHJvdXRlID0gcGFja2VkLmluaXRpYWxfdG91ci5jb3B5KCkKICAgICAgICBpYSwgaWIgPSBpbnQobnAud2hlcmUocm91dGUgPT0gYSlbMF1bMF0pLCBpbnQobnAud2hlcmUocm91dGUgPT0gYilbMF1bMF0pCiAgICAgICAgc3dhcCA9IChpYiArIDIpICUgbGVuKHJvdXRlKQogICAgICAgIHJvdXRlW2liXSwgcm91dGVbc3dhcF0gPSByb3V0ZVtzd2FwXSwgcm91dGVbaWJdCiAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgcGFja2VkLmV4cGFuZChyb3V0ZSkKCiAgICBAdW5pdHRlc3Quc2tpcFVubGVzcyhvcy5lbnZpcm9uLmdldCgiQkVUQVRPVVJfVEVTVF9MS0giKSwgInNldCBCRVRBVE9VUl9URVNUX0xLSCBmb3IgTEtIIHNtb2tlIikKICAgIGRlZiB0ZXN0X3NtYWxsX2xraF9yZXBhaXJfbmV2ZXJfd29yc2VucyhzZWxmKSAtPiBOb25lOgogICAgICAgIGJpbmFyeSA9IFBhdGgob3MuZW52aXJvblsiQkVUQVRPVVJfVEVTVF9MS0giXSkKICAgICAgICByZXN1bHQgPSByZXBhaXJfb25jZShzZWxmLnh5LCBzZWxmLnRvdXIsIGJpbmFyeSwgUGF0aCgiL3ByaXZhdGUvdG1wL2JldGF0b3VyLXYxMC11bml0dGVzdCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0cmF0ZWd5PSJyYW5kb20iLCBlbnNlbWJsZV90b3Vycz1Ob25lLCBtb2RlbF9zY29yZXM9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBybmc9bnAucmFuZG9tLmRlZmF1bHRfcm5nKDIpLCBjdXRfY291bnQ9Mywgd2luZG93X3NpemU9MTIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGltZW91dF9zPTIuMCwgbWF4X3RyaWFscz0zMCkKICAgICAgICBzZWxmLmFzc2VydExlc3NFcXVhbChyZXN1bHQuYWZ0ZXIsIHJlc3VsdC5iZWZvcmUpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZXN1bHQuYWZ0ZXIsIHRvdXJfY29zdChzZWxmLnh5LCByZXN1bHQudG91cikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'v10_repair/README.md': 'IyBCZXRhVG91ciB2MTA6IMOnb2sga2VuYXJsxLEgdHVyIG9uYXLEsW0gcGlsb3R1CgpCdSDDp2FsxLHFn21hICoqYWxnb3JpdG1hIGdlbGnFn3Rpcm1lIHBpbG90dWR1cioqLiB2OC92OSdkYSBpbmNlbGVubWnFnyBzZWtpeiBFVUNfMkQgw7ZybmXEn2luaSBrdWxsYW7EsXI7IGJhxJ/EsW1zxLF6IGRvxJ9ydWxhbWEgdmVyaXNpIGRlxJ9pbGRpci4gU29udcOnbGFyxLEgb2x1bWx1IG9sc2EgYmlsZSB0ZWsgYmHFn8SxbmEgbWFrYWxlZGUgw7xzdMO8bmzDvGsgaWRkaWFzxLEgb2x1xZ90dXJtYXouCgojIyBTb3J1IHZlIHnDtm50ZW0KCnY5LCBpbGsgdHVydW4gYXJkxLFuZGFuIHRlayBzZWZlcmRlIGFkYXkga2VuYXIgZWtsZWRpOyBQT1BNVVNJQy02IGthcsWfxLFzxLFuZGEgZ8O8dmVuaWxpciBiaXIga2F6YW7DpyBnw7ZzdGVybWVkaS4gdjEwLCBtZXZjdXQgdHVydSBkb8SfcnVkYW4gZGXEn2nFn3RpcmViaWxlbiBiaXIgbWVrYW5pem1hIHPEsW5hcjogw7zDpyBzb3J1bmx1IGtlc2ltIMOnZXZyZXNpbmRla2kgdHVyIHBlbmNlcmVsZXJpbmkgc2VyYmVzdCBixLFyYWvEsXIsIGthbGFuIHV6dW4gcGFyw6dhbGFyxLEgem9ydW5sdSBrZW5hcmxhcmEgc8Sxa8SxxZ90xLFyxLFyIHZlIHlha2xhxZ/EsWsgMyDDlyAyNTYgc2VyYmVzdCBkw7zEn8O8bWzDvCBrw7zDp8O8ayBiaXIgVFNQJ3lpIExLSCBpbGUgw6fDtnplci4gWm9ydW5sdSBrZW5hcmxhciBhw6fEsWxhcmFrIHRhbSB0dXIgZ2VyaSBrdXJ1bHVyLiBZYWxuxLF6Y2EgZ2XDp2VybGkgdmUga2VzaW4gb2xhcmFrIGRhaGEga8Sxc2EgYmlyIHR1ciBrYWJ1bCBlZGlsaXIuIE9yaWppbmFsIHR1ciBoZXIgemFtYW4gbcO8bWvDvG5kw7xyLiBCdSwgw7bEn3JlbmlsbWnFnyBvbmFyxLFtIHNlw6dpY2lzaW5lIHlhdMSxcsSxbSB5YXBtYWRhbiDDtm5jZSDDp29rIGtlbmFybMSxIG9uYXLEsW3EsW4ga8Sxc2EgQ29sYWIgYsO8dMOnZXNpbmRlIGnFn2UgeWFyYXnEsXAgeWFyYW1hZMSxxJ/EsW7EsSDDtmzDp2VyLgoKQWx0xLEgZcWfbGVubWnFnyBQT1BNVVNJQy02L0xLSCBzw7xyZWNpIGLDvHTDp2VuaW4geWFrbGHFn8SxayAlMzUnaW5kZSwgdGFtYW1sYW5hbiBpbGsgdXlndW4gZGVuZW1lZGVuIHNvbnJhIHYxMCBrYW5jYXPEsW5kYSBiZWtsZXRpbGlyLiBCw7Z5bGVjZSBvbmFyxLFtLCBpbGsgdHVyIHllcmluZSBhcmFtYSBzxLFyYXPEsW5kYSBvbGd1bmxhxZ9txLHFnyBiaXIgdHVyYSB1eWd1bGFuxLFyLiBPbmFyxLFtbGFyIGTEscWfIHPDvHJlw6d0ZSB5YXDEsWzEsXI7IGF5bsSxIExLSCBzw7xyZcOnbGVyaSBtw7xkYWhhbGVzaXogZGV2YW0gZWRlci4gU29udcOnLCBzw7xyZW5pbiBzb251bmRhIExLSCduaW4gYnVsZHXEn3UgdHVyIGlsZSBvbmFyxLFsbcSxxZ8gdHVydW4ga8Sxc2Egb2xhbsSxZMSxci4gT25hcsSxbSB2ZSBtb2RlbCDDp8Sxa2FyxLFtxLEgYXluxLEgMTIwLzMwMCBzYW5peWVsaWsgZHV2YXIgc2FhdGkgc8SxbsSxcsSxbmEgZGFoaWxkaXIuIE9uYXLEsWxtxLHFnyB0dXIgTEtIJ25pbiBpw6cgZHVydW11bmEgZW5qZWt0ZSBlZGlsbWV6OyByYXBvcmRhIHnDtm50ZW0gYcOnxLFrw6dhICoqeWFuIG9uYXLEsW1sxLEgaGlicml0Kiogb2xhcmFrIGFkbGFuZMSxcsSxbMSxci4KCiMjIEthcsWfxLFsYcWfdMSxcm1hbGFyCgp8IEtvbCB8IFJvbCB8CnwgLS0tIHwgLS0tIHwKfCBLZXNpbnRpc2l6IFBPUE1VU0lDLTYgfCBQcmF0aWsgZ8O8w6dsw7wga2FyxZ/EsWxhxZ90xLFybWEuIHwKfCBLYW5jYWzEsSBzaGFtIHwgQmVrbGVtZSB2ZSBkZXZhbSBldGtpc2luaSDDtmzDp2VyOyBvbmFyxLFtIHlhcG1hei4gfAp8IFJhc3RnZWxlIGtlc2ltbGVyIHwgQXluxLEgb25hcsSxbSBtb3RvcnV5bGEgc2XDp2ljaSBrb250cm9sw7wuIHwKfCBVenVuL29sYcSfYW5kxLHFn8SxIGtlbmFyIHNlw6dpbWkgfCBHw7zDp2zDvCwgbW9kZWxzaXogeWVyZWwgc2V6Z2lzZWwga29udHJvbC4gfAp8IEFsdMSxIHR1cnVuIHV5dcWfbWF6bMSxxJ/EsSB8IFBvcMO8bGFzeW9uIGJpbGdpc2luaSBtb2RlbHNpeiBrdWxsYW5hbiBrb250cm9sLiB8CnwgQmV0YVRvdXIgc2XDp2ltaSB8IFZhcnNhIG1ldmN1dCBUb3VyRm9sZCBwdWFuxLF5bGEga2VzaW0gc2XDp2ltaTsgYnUgYcSfIGhlbsO8eiBnZXLDp2VrIG9uYXLEsW0gZmF5ZGFzxLFuYSBnw7ZyZSBlxJ9pdGlsbWVtacWfdGlyLiB8CgpPbmFyxLFtbMSxIGtvbGxhciBheW7EsSBrZXNpbSBzYXnEsXPEsSwgZGVuZW1lIMO8c3Qgc8SxbsSxcsSxIHZlIHRvcGxhbSBzw7xyZSBwYXnEsW7EsSBrdWxsYW5tYWzEsWTEsXIuIE1vZGVsIMOnxLFrYXLEsW3EsSBkYSBCZXRhVG91ciBrb2x1bnVuIHPDvHJlIGLDvHTDp2VzaW5kZW4gZMO8xZ/DvGzDvHIuIFBPUE1VU0lDLTYgZ2Vyw6dlayBrdWxsYW7EsW0ga2FyxZ/EsWxhxZ90xLFybWFzxLFkxLFyOyB5YWxuxLF6IGJhxZ/EsW5hIHnDtm50ZW0gZXRraXNpbmkgaXpvbGUgZXRtZXosIMOnw7xua8O8IHlhbiBvbmFyxLFtIGVrIGhlc2FwbGFtYSBrdWxsYW7EsXIuIEJldGFUb3VyJ3VuIHNlw6dpY2kga2F0a8Sxc8SxIGFuY2FrICoqYXluxLEgeWFuIG9uYXLEsW0gbW90b3J1bnUga3VsbGFuYW4qKiBrb250cm9sIGtvbGxhcsSxbmEga2FyxZ/EsSB5b3J1bWxhbsSxci4KCiMjIEthecSxdCB2ZSBrYXJhcgoKSGVyIHRvaHVtIGnDp2luIGJhcml5ZXJkZWtpIGVuIGl5aSB0dXIsIExLSCBzb24gdHVydSwgb25hcsSxbSBzb24gdHVydSwgc2XDp2lsZW4ga2VzaW1sZXIsIG9uYXLEsW0gZGVuZW1lbGVyaSwgbWFsaXlldGxlciwgemFtYW4gZGFtZ2FsYXLEsSB2ZSBrYXluYWsvYmluYXJ5L21vZGVsIGhhc2gnbGVyaSBzYWtsYW7EsXIuIFR1cmxhcsSxbiBIYW1pbHRvbiBnZcOnZXJsaWxpxJ9pIHZlIFRTUExJQiBFVUNfMkQgdGFtc2F5xLEgbWFsaXlldGkga29vcmRpbmF0bGFyZGFuIHllbmlkZW4gaGVzYXBsYW7EsXIuIMSwbmNlbGVtZSBiaXJpbWkgc2VraXogKirDtnJuZWt0aXIqKiwgYWx0xLEgdG9odW0gYmHEn8SxbXPEsXogw7ZybmVrIHNhecSxbG1hei4gRcWfaXQgdG9wbGFtIHPDvHJlIGnDp2luZGUgw7ZybmVrIGJhxZ/EsW5hIGVuIGvEsXNhIGdlw6dlcmxpIHR1ciwgemFtYW7igJNrYWxpdGUgZcSfcmlzaSB2ZSBvbmFyxLFtxLFuIGtlbmRpIGthemFuY8SxIHJhcG9ybGFuxLFyLgoKUGlsb3QgacOnaW4ga2FyYXIgc8SxcmFzxLE6CgoxLiBPbmFyxLFtIG1vdG9ydSBnZcOnZXJsaSwga2VzaW4gaXlpbGXFn3RpcmVuIHR1cmxhciDDvHJldGVtaXlvcnNhIGJ1IGvEsXNhIGLDvHTDp2VsaSB0YXNhcsSxbcSxIGR1cmR1ci4KMi4gTW9kZWxzaXogc2XDp2ljaWxlciBmYXlkYSBzYcSfbMSxeW9yIGFtYSBCZXRhVG91ciBzYcSfbGFtxLF5b3JzYSwgbWV2Y3V0IGtlbmFyIHPEsW7EsWZsYW5kxLFyxLFjxLFzxLFuxLEgYmHFn2FyxLEgZGl5ZSBzdW5tYTsgZ2Vyw6dlayBvbmFyxLFtIGthemFuY8SxL3Nhbml5ZSBldGlrZXRsZXJpeWxlIGR1cnVtIGtvxZ91bGx1IGJpciBzZcOnaWNpIGXEn2l0LgozLiBCZXRhVG91ciBoZW0gUE9QTVVTSUMtNid5xLEgaGVtIGRlIGfDvMOnbMO8IG1vZGVsc2l6IHlhbiBvbmFyxLFtIGtvbnRyb2zDvG7DvCB0ZWtyYXJsxLEgYmnDp2ltZGUgZ2XDp2Vyc2Uga29kdSB2ZSBwYXJhbWV0cmVsZXJpIGRvbmR1cnVwIHllbmksIGJhxJ/EsW1zxLF6IMO2cm5la2xlcmRlIGRhaGEgZ2VuacWfIGRlbmV5IHlhcC4KCk1ha2FsZXllIHlhbG7EsXpjYSBiYcSfxLFtc8SxeiBkZW5leWxlciBvbHVtbHUgc29udcOnIHZlcmRpa3RlbiBzb25yYSB5ZW5pIHBlcmZvcm1hbnMgaWRkaWFzxLEgZWtsZW5pci4gT2x1bXN1eiBwaWxvdCwgbWV2Y3V0IHY5IG1ha2FsZSB0YXNsYcSfxLFuxLEgZGXEn2nFn3Rpcm1lei4K'}
CODE_ROOT = WORK_ROOT / "code"
for rel, encoded in BUNDLED_FILES.items():
    dest = CODE_ROOT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    content = base64.b64decode(encoded)
    if dest.exists():
        assert dest.read_bytes() == content, f"Kod dosyası değişti: {rel}"
    else:
        dest.write_bytes(content)
    drive_dest = OUTPUT_ROOT / "code" / rel
    drive_dest.parent.mkdir(parents=True, exist_ok=True)
    if drive_dest.exists():
        assert drive_dest.read_bytes() == content, f"Drive kod dosyası değişti: {rel}"
    else:
        drive_dest.write_bytes(content)
CODE_SHA256 = {rel: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
               for rel, encoded in BUNDLED_FILES.items()}
sys.path[:0] = [str(CODE_ROOT / "livehook"), str(CODE_ROOT / "policy"),
                str(CODE_ROOT / "v9_live"), str(CODE_ROOT / "v10_repair")]
import build_repair_hook
from coordinator import read_tour
from live_pilot import run_native_arm, sha256_file
from repair_pilot import run_repair_arm
from compressed_repair import compress_tour, free_windows
print("Deney kodu doğrulandı:", CODE_SHA256)


Deney kodu doğrulandı: {'livehook/BetaTourHook.h': '078b46b97517749c73144a3a272b9680436578e9beb897687d5d824fab842cf7', 'livehook/build_livehook.py': '1c1d86cae6b9fede0458d38b969a02cd49d0ff3fa20470288e877c2b7e078a82', 'livehook/coordinator.py': '9b254c175eda013690465a6658eea76dcd86da2f95b3ed69125e93d7fc789ed8', 'policy/adaptive_policy.py': '1646995c3075d60088e0a73c42044ad6065987028e48193780ac47d7e77075b8', 'v9_live/live_pilot.py': '2cd13337ef7c9c84f18b75c01bc98738fef8cb8c433416b43eaad4a56fca1000', 'v10_repair/BetaTourRepairHook.h': '274b8ffeb8a1426a5e415939f426480b3d4a0609288818ee8d00a7cbaa938f09', 'v10_repair/build_repair_hook.py': '08ed29ba9a94f07b711959c62de00083bc528542fecb7b77bc217d509f466ed2', 'v10_repair/repair_engine.py': 'f11e5f736c4475989af8a592706cc5eb5f4169264e45b7539c3ef4e6fbe95bda', 'v10_repair/compressed_repair.py': '8232d399240b864115131c775e41f85587f05b4e80dd877eeab103fc0d4c6611', 'v10_repair/repair_pilot.py': '14dc12253b09ce45f4971caab0dfffeb0dab9fa2d99d534742f8faa76a4

In [ ]:
LKH_ROOT = WORK_ROOT / "lkh"
LKH_ROOT.mkdir(exist_ok=True)
SOURCE_DIR = LKH_ROOT / "LKH-3.0.14"
SOURCE_INFO_PATH = LKH_ROOT / "source_info.json"
if not SOURCE_DIR.exists():
    sources = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned_mirror", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar", LKH_TAR_SHA256, 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout_s in sources:
        archive = LKH_ROOT / ("LKH-3.0.14.tgz" if url.endswith(".tgz") else "LKH-3.0.14.tar")
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout_s) as response, archive.open("wb") as out:
                shutil.copyfileobj(response, out)
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if expected_sha:
                assert digest == expected_sha, f"LKH arşiv SHA256 uyuşmuyor: {digest}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(LKH_ROOT, filter="data")
                except TypeError:
                    tar.extractall(LKH_ROOT)
            assert SOURCE_DIR.exists()
            actual_source_sha = hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest()
            assert actual_source_sha == LKH_FINDTOUR_SHA256, f"LKH 3.0.14 kaynağı beklenenden farklı: {actual_source_sha}"
            SOURCE_INFO_PATH.write_text(json.dumps({"label": label, "url": url, "archive_sha256": digest}, indent=2))
            break
        except Exception as exc:
            last_error = exc
            if SOURCE_DIR.exists():
                shutil.rmtree(SOURCE_DIR)
    else:
        raise RuntimeError(f"LKH indirilemedi: {last_error}")
assert hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest() == LKH_FINDTOUR_SHA256
if not SOURCE_INFO_PATH.exists():
    cached = next((p for p in (LKH_ROOT / "LKH-3.0.14.tar", LKH_ROOT / "LKH-3.0.14.tgz") if p.exists()), None)
    assert cached is not None, "LKH kaynak kayıt dosyası eksik; kaynak arşiv bulunamadı"
    SOURCE_INFO_PATH.write_text(json.dumps({"label": "recovered_cache", "url": "local_cache",
                                            "archive_sha256": hashlib.sha256(cached.read_bytes()).hexdigest()}, indent=2))
NATIVE_LKH = SOURCE_DIR / "LKH"
if not NATIVE_LKH.exists():
    with (LKH_ROOT / "native_build.log").open("w") as build_log:
        subprocess.run(["make", "-j", str(min(6, CPU_COUNT))], cwd=SOURCE_DIR, check=True,
                       stdout=build_log, stderr=subprocess.STDOUT)
PATCHED_DIR = LKH_ROOT / "LKH-3.0.14-BetaTour-v10"
PATCHED_LKH = PATCHED_DIR / "LKH"
if not PATCHED_LKH.exists():
    if PATCHED_DIR.exists():
        shutil.rmtree(PATCHED_DIR)
    build_repair_hook.build(SOURCE_DIR, PATCHED_DIR, jobs=min(6, CPU_COUNT))
BUILD_INFO = {"native_sha256": sha256_file(NATIVE_LKH),
              "patched_sha256": sha256_file(PATCHED_LKH),
              "upstream_findtour_sha256": LKH_FINDTOUR_SHA256}
origin_path = OUTPUT_ROOT / "code" / "lkh_source_info.json"
if not origin_path.exists():
    origin_path.write_text(SOURCE_INFO_PATH.read_text())
print("LKH 3.0.14 derlendi:", BUILD_INFO)


LKH 3.0.14 derlendi: {'native_sha256': '9211567cc25e47b0ef135367c6ce3162f6ff013451ec9891ddc146ef47157768', 'patched_sha256': 'de15ec9d99dcaede7e42be5d852ce1f967995e341f7722ce11febb8ef4139a00', 'upstream_findtour_sha256': '567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee'}


In [ ]:
def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

MANIFEST = {"schema": 2, "profile": "pilot_old_v8_instances",
            "experiment_name": EXPERIMENT_NAME, "arms": ARMS, "seeds": SEEDS,
            "instance_specs": INSTANCE_SPECS, "xy_hashes": EXPECTED_XY_SHA256,
            "repair_params": REPAIR_PARAMS, "repo_commit": REPO_COMMIT,
            "checkpoint_sha256": CHECKPOINT_SHA256, "hardware": HARDWARE,
            "code_sha256": CODE_SHA256, "build": BUILD_INFO,
            "analysis_plan": "Development pilot only. The manuscript stays unchanged until a separately frozen policy passes independent evaluation."}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in MANIFEST:
        if key != "hardware":
            assert saved[key] == MANIFEST[key], f"Drive'daki pilotun {key} ayarı değişti. Yeni EXPERIMENT_NAME kullanın."
    for key in ("gpu", "vcpus"):
        assert saved["hardware"][key] == HARDWARE[key], f"Donanım {key} değişti; eşli pilot için aynı türü seçin."
    if saved["hardware"] != HARDWARE:
        print("UYARI: CPU modeli veya kütüphane sürümü değişti; kol donanımları ayrı kaydedilecek.")
else:
    write_json_atomic(manifest_path, MANIFEST)

# Small exact compression/expansion check and patched-vs-native first-tour check.
warm_xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
warm_tour = np.arange(100, dtype=np.int32)
packed = compress_tour(warm_xy, warm_tour, free_windows(100, np.array([5, 35, 70]), 12))
assert T.tour_length(warm_xy, packed.expand(packed.initial_tour), "EUC_2D") == T.tour_length(warm_xy, warm_tour, "EUC_2D")
warm_succ = np.stack([np.roll(np.arange(100), -1)] * 6, axis=1).astype(np.int32)
M.predict_tourfold(MODEL, warm_xy, 16, DEVICE, ens_succ=warm_succ, ens_cost=np.ones(6))
warm_problem = WORK_ROOT / "warmup.tsp"
T.write_tsp(warm_problem, warm_xy)
warm_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_dir.parent.mkdir(exist_ok=True)
warm = run_repair_arm("sham", PATCHED_LKH, NATIVE_LKH, warm_problem, warm_xy,
                      warm_dir, SEEDS, 15.0, MODEL, DEVICE, T, M,
                      barrier_fraction=0.0)
def edge_set(tour):
    tour = np.asarray(tour, dtype=np.int32)
    return set(zip(np.minimum(tour, np.roll(tour, -1)),
                   np.maximum(tour, np.roll(tour, -1))))
for seed in SEEDS:
    native = T.run_lkh(str(NATIVE_LKH), str(warm_problem),
                       {"CANDIDATE_SET_TYPE": "POPMUSIC", "MAX_CANDIDATES": 6,
                        "SUBGRADIENT": "NO", "SEED": seed, "MAX_TRIALS": 1,
                        "TIME_LIMIT": 30, "STOP_AT_OPTIMUM": "NO"},
                       str(WORK_ROOT / "warmup" / f"native_{seed}_{uuid.uuid4().hex}"), timeout=30)
    patch_first = np.asarray(read_tour(warm_dir / f"seed_{seed}" / "barrier.tour")) - 1
    assert edge_set(native["tour"]) == edge_set(patch_first), f"İlk tur uyuşmuyor: {seed}"
print("GPU, yama, sıkıştırma ve ilk tur eşitliği doğrulandı.")


GPU, yama, sıkıştırma ve ilk tur eşitliği doğrulandı.


In [ ]:
INSTANCES = []
for group, n, count, distribution, generator_seed, budget_s in INSTANCE_SPECS:
    rng = np.random.default_rng(generator_seed)
    for index in range(count):
        xy = T.to_int_coords(T.gen_instance(n, distribution, rng))
        name = f"augment_{group}_{index:03d}"
        digest = hashlib.sha256(np.ascontiguousarray(xy).tobytes()).hexdigest()
        assert digest == EXPECTED_XY_SHA256[name], f"Eski v8 koordinatları uyuşmuyor: {name}"
        INSTANCES.append({"name": name, "group": group, "n": n,
                          "budget_s": budget_s, "xy": xy, "xy_sha256": digest})
INSTANCES.sort(key=lambda item: (item["budget_s"], item["name"]))
assert len(INSTANCES) == 8
def arm_order(name):
    return sorted(ARMS, key=lambda arm: hashlib.sha256((name + ":" + arm).encode()).hexdigest())
print("Pilot:", len(INSTANCES), "örnek,", len(ARMS), "kol; kesintisiz tek kol bütçesi",
      sum(item["budget_s"] for item in INSTANCES) / 3600, "saat")


Pilot: 8 örnek, 6 kol; kesintisiz tek kol bütçesi 0.4666666666666667 saat


In [ ]:
def arm_record_path(inst, arm):
    return OUTPUT_ROOT / "instances" / inst["name"] / arm / "result.json"

def validate_saved(inst, arm, record):
    assert record["complete"] and record["arm"] == arm and record["instance"] == inst["name"]
    assert record["xy_sha256"] == inst["xy_sha256"] and record["seeds"] == SEEDS
    assert record["budget_s"] == inst["budget_s"] and record["build"] == BUILD_INFO
    assert record["code_sha256"] == CODE_SHA256 and record["repair_params"] == REPAIR_PARAMS
    assert len(record["seed_costs"]) == 6 and min(record["seed_costs"]) == record["best_cost"]
    return record

def save_progress(state="running", last=None):
    completed = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
    write_json_atomic(OUTPUT_ROOT / "progress.json", {"state": state, "completed_arms": completed,
                      "total_arms": len(INSTANCES) * len(ARMS), "last": last,
                      "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})

new_arms = 0
save_progress()
for inst in INSTANCES:
    instance_dir = OUTPUT_ROOT / "instances" / inst["name"]
    instance_dir.mkdir(parents=True, exist_ok=True)
    coord_path = instance_dir / "xy.npz"
    if not coord_path.exists():
        temp = instance_dir / "xy.tmp.npz"
        np.savez_compressed(temp, xy=inst["xy"])
        os.replace(temp, coord_path)
    with np.load(coord_path, allow_pickle=False) as saved_xy:
        assert hashlib.sha256(np.ascontiguousarray(saved_xy["xy"]).tobytes()).hexdigest() == inst["xy_sha256"]
    local_problem = WORK_ROOT / "problems" / (inst["name"] + ".tsp")
    local_problem.parent.mkdir(exist_ok=True)
    T.write_tsp(local_problem, inst["xy"])
    for arm in arm_order(inst["name"]):
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            validate_saved(inst, arm, json.loads(record_path.read_text()))
            print("ATLA:", inst["name"], arm, flush=True)
            continue
        if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
            break
        attempt_id = uuid.uuid4().hex
        local_run = WORK_ROOT / "runs" / inst["name"] / arm / attempt_id
        local_run.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("BAŞLA:", inst["name"], arm, "bütçe", inst["budget_s"], "sn", flush=True)
            if arm == "native_pop6":
                raw = run_native_arm(arm, NATIVE_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"], T)
            else:
                raw = run_repair_arm(arm, PATCHED_LKH, NATIVE_LKH, local_problem,
                                     inst["xy"], local_run, SEEDS, inst["budget_s"],
                                     MODEL, DEVICE, T, M, **REPAIR_PARAMS)
            drive_arm = record_path.parent
            drive_arm.mkdir(parents=True, exist_ok=True)
            audit_rel = f"instances/{inst['name']}/{arm}/{attempt_id}"
            audit_dest = OUTPUT_ROOT / audit_rel
            shutil.copytree(local_run, audit_dest)
            record = {"schema": 2, "complete": True, "instance": inst["name"],
                      "group": inst["group"], "n": inst["n"], "xy_sha256": inst["xy_sha256"],
                      "arm": arm, "seeds": SEEDS, "budget_s": inst["budget_s"],
                      "seed_costs": raw["seed_costs"], "best_cost": raw["best_cost"],
                      "barrier_costs": raw.get("barrier_costs"),
                      "barrier_tour_hashes": raw.get("barrier_tour_hashes"),
                      "lkh_costs": raw.get("lkh_costs"),
                      "repair_costs": raw.get("repair_costs"),
                      "repair_wall_s": raw.get("repair_wall_s"),
                      "barrier_wall_from_start_s": raw.get("barrier_wall_from_start_s"),
                      "model_inference_s": raw.get("model_inference_s"),
                      "completed_attempts_per_seed": raw.get("completed_attempts_per_seed"),
                      "wall_elapsed_s": raw["wall_elapsed_s"], "build": BUILD_INFO,
                      "code_sha256": CODE_SHA256, "repair_params": REPAIR_PARAMS,
                      "hardware": HARDWARE, "audit_relpath": audit_rel,
                      "attempt_id": attempt_id,
                      "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            validate_saved(inst, arm, record)
            write_json_atomic(record_path, record)
            new_arms += 1
            save_progress(last=inst["name"] + "/" + arm)
            print("BİTTİ:", inst["name"], arm, "en iyi", raw["best_cost"], flush=True)
        except Exception as exc:
            write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + "_" + arm + "_" + attempt_id + ".json"),
                              {"instance": inst["name"], "arm": arm, "error": repr(exc),
                               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
            save_progress(state="error", last=inst["name"] + "/" + arm)
            raise
    if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
        break
completed_count = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
save_progress(state="complete" if completed_count == len(INSTANCES) * len(ARMS) else "partial")
print(f"Tamamlanan kol: {completed_count}/{len(INSTANCES) * len(ARMS)}. Bu oturumdaki yeni kol: {new_arms}")


ATLA: augment_clu10k_000 repair_longest
ATLA: augment_clu10k_000 native_pop6
ATLA: augment_clu10k_000 repair_random
ATLA: augment_clu10k_000 repair_betatour
ATLA: augment_clu10k_000 repair_disagreement
ATLA: augment_clu10k_000 sham
ATLA: augment_clu10k_001 repair_longest
ATLA: augment_clu10k_001 repair_random
ATLA: augment_clu10k_001 native_pop6
ATLA: augment_clu10k_001 repair_betatour
ATLA: augment_clu10k_001 repair_disagreement
ATLA: augment_clu10k_001 sham
ATLA: augment_uni10k_000 repair_random
ATLA: augment_uni10k_000 sham
ATLA: augment_uni10k_000 native_pop6
ATLA: augment_uni10k_000 repair_betatour
ATLA: augment_uni10k_000 repair_disagreement
ATLA: augment_uni10k_000 repair_longest
ATLA: augment_uni10k_001 repair_random
ATLA: augment_uni10k_001 repair_disagreement
ATLA: augment_uni10k_001 native_pop6
ATLA: augment_uni10k_001 sham
ATLA: augment_uni10k_001 repair_betatour
ATLA: augment_uni10k_001 repair_longest
ATLA: augment_clu20k_000 repair_disagreement
ATLA: augment_clu20k_000 re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""v10 Colab analysis cell: audit saved barriers before interpreting endpoints.

Paste this entire file into the original notebook's failed summary cell. It
reads the 48 completed Drive records and does not start any LKH run.
"""

from itertools import combinations


def canonical_edge_hash(tour):
    """A TSP cycle has the same hash after rotation or reversal."""
    tour = np.asarray(tour, dtype=np.int32)
    following = np.roll(tour, -1)
    edges = np.column_stack((np.minimum(tour, following),
                             np.maximum(tour, following)))
    edges = np.ascontiguousarray(edges[np.lexsort((edges[:, 1], edges[:, 0]))])
    return hashlib.sha256(edges.tobytes()).hexdigest()


def comparison_summary(frame, treatment, control, basis):
    if frame.empty:
        return None
    difference = 100.0 * (frame[treatment].to_numpy(dtype=np.float64) -
                          frame[control].to_numpy(dtype=np.float64)) / frame[control].to_numpy(dtype=np.float64)
    wins = int((difference < 0).sum())
    losses = int((difference > 0).sum())
    return {"treatment": treatment, "control": control, "basis": basis,
            "n_instances": len(difference), "wins": wins, "losses": losses,
            "ties": int((difference == 0).sum()),
            "mean_relative_pct": float(difference.mean()),
            "median_relative_pct": float(np.median(difference)),
            "exploratory_sign_p": float(binomtest(wins, wins + losses).pvalue)
            if wins + losses else 1.0}


patched_arms = [arm for arm in ARMS if arm != "native_pop6"]
summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)
rows, barrier_rows, pair_rows = [], [], []

for inst in INSTANCES:
    records = {}
    for arm in ARMS:
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            records[arm] = validate_saved(inst, arm, json.loads(record_path.read_text()))
    if len(records) != len(ARMS):
        continue

    states = {}
    for arm in patched_arms:
        record = records[arm]
        raw_hashes = record["barrier_tour_hashes"]
        saved_costs = record["barrier_costs"]
        if raw_hashes is None or saved_costs is None or len(raw_hashes) != len(SEEDS) or len(saved_costs) != len(SEEDS):
            raise AssertionError(f"Bariyer kaydı eksik: {inst['name']}/{arm}")
        for index, seed in enumerate(SEEDS):
            path = OUTPUT_ROOT / record["audit_relpath"] / f"seed_{seed}" / "barrier.tour"
            tour = np.asarray(read_tour(path), dtype=np.int32) - 1
            if not T.is_valid_tour(tour, inst["n"]):
                raise AssertionError(f"Geçersiz bariyer turu: {path}")
            raw_hash = hashlib.sha256(np.ascontiguousarray(tour).tobytes()).hexdigest()
            if raw_hash != raw_hashes[index]:
                raise AssertionError(f"Bariyer dosyası kayıtla uyuşmuyor: {path}")
            cost = int(T.tour_length(inst["xy"], tour, "EUC_2D"))
            if cost != int(saved_costs[index]):
                raise AssertionError(f"Bariyer maliyeti kayıtla uyuşmuyor: {path}")
            edge_hash = canonical_edge_hash(tour)
            states[arm, seed] = (raw_hash, edge_hash, cost)
            barrier_rows.append({"instance": inst["name"], "arm": arm, "seed": seed,
                                 "raw_tour_sha256": raw_hash,
                                 "undirected_edges_sha256": edge_hash,
                                 "barrier_cost": cost,
                                 "callback_wall_s": record["barrier_wall_from_start_s"],
                                 "audit_tour": str(path.relative_to(OUTPUT_ROOT))})

    for arm_a, arm_b in combinations(patched_arms, 2):
        raw_matches = [states[arm_a, seed][0] == states[arm_b, seed][0] for seed in SEEDS]
        edge_matches = [states[arm_a, seed][1] == states[arm_b, seed][1] for seed in SEEDS]
        pair_rows.append({"instance": inst["name"], "arm_a": arm_a, "arm_b": arm_b,
                          "matching_raw_seeds": sum(raw_matches),
                          "matching_edge_seeds": sum(edge_matches),
                          "all_six_edge_match": all(edge_matches),
                          "representation_only_mismatch_seeds": sum(e and not r for e, r in zip(edge_matches, raw_matches)),
                          "max_abs_barrier_cost_gap": max(abs(states[arm_a, seed][2] - states[arm_b, seed][2])
                                                      for seed in SEEDS)})

    row = {"instance": inst["name"], "group": inst["group"],
           "n": inst["n"], "budget_s": inst["budget_s"],
           "mixed_hardware": len({(records[a]["hardware"]["gpu"],
                                    records[a]["hardware"]["vcpus"],
                                    records[a]["hardware"]["cpu_model"]) for a in ARMS}) > 1,
           "barrier_tours_match": all(len({states[arm, seed][1] for arm in patched_arms}) == 1
                                      for seed in SEEDS)}
    row.update({arm: records[arm]["best_cost"] for arm in ARMS})
    rows.append(row)

table = pd.DataFrame(rows)
barrier_table = pd.DataFrame(barrier_rows)
pair_table = pd.DataFrame(pair_rows)
table.to_csv(summary_dir / "complete_instances.csv", index=False)
barrier_table.to_csv(summary_dir / "barrier_tour_audit.csv", index=False)
pair_table.to_csv(summary_dir / "barrier_pairwise.csv", index=False)

same_hardware = table.loc[~table["mixed_hardware"]].copy() if not table.empty else table
endpoint_rows = []
for treatment in ("repair_betatour", "repair_longest", "repair_disagreement", "repair_random", "sham"):
    for control in ("native_pop6", "sham", "repair_random", "repair_longest", "repair_disagreement"):
        if treatment == control:
            continue
        result = comparison_summary(same_hardware, treatment, control,
                                    "same-instance endpoint; barrier may differ")
        if result is not None:
            endpoint_rows.append(result)
endpoint_columns = ["treatment", "control", "basis", "n_instances", "wins", "losses", "ties",
                    "mean_relative_pct", "median_relative_pct", "exploratory_sign_p"]
endpoint_table = pd.DataFrame(endpoint_rows, columns=endpoint_columns)
endpoint_table.to_csv(summary_dir / "endpoint_comparisons.csv", index=False)

matched_rows = []
for treatment in ("repair_betatour", "repair_longest", "repair_disagreement", "repair_random", "sham"):
    for control in patched_arms:
        if treatment == control:
            continue
        if pair_table.empty:
            continue
        matching_names = set(pair_table.loc[
            ((pair_table["arm_a"] == treatment) & (pair_table["arm_b"] == control) |
             (pair_table["arm_a"] == control) & (pair_table["arm_b"] == treatment)) &
            pair_table["all_six_edge_match"], "instance"])
        matched = same_hardware.loc[same_hardware["instance"].isin(matching_names)]
        result = comparison_summary(matched, treatment, control,
                                    "all six saved barrier cycles match")
        if result is not None:
            matched_rows.append(result)
matched_table = pd.DataFrame(matched_rows, columns=endpoint_columns)
matched_table.to_csv(summary_dir / "matched_barrier_comparisons.csv", index=False)

raw_mismatch_count = int((pair_table["matching_raw_seeds"] < len(SEEDS)).sum()) if not pair_table.empty else 0
edge_mismatch_count = int((pair_table["matching_edge_seeds"] < len(SEEDS)).sum()) if not pair_table.empty else 0
note = {"analysis_revision": "canonical-barrier-audit-2026-10-29",
        "completed_instances": len(table), "planned_instances": len(INSTANCES),
        "same_hardware_instances": len(same_hardware),
        "all_arm_barrier_matched_instances": int(table["barrier_tours_match"].sum()) if not table.empty else 0,
        "arm_pairs_with_raw_mismatch": raw_mismatch_count,
        "arm_pairs_with_real_edge_mismatch": edge_mismatch_count,
        "interpretation": "Endpoint comparisons are descriptive. Policy-specific paired comparisons use only instances with all six identical undirected barrier cycles for that arm pair. This reused-case pilot does not justify a manuscript performance claim."}
write_json_atomic(summary_dir / "pilot_note.json", note)

print("Tamamlanan örnek:", len(table), "/", len(INSTANCES))
print("Tüm onarım kollarında aynı bariyer çevrimi:", note["all_arm_barrier_matched_instances"], "/", len(table))
print("Ham karma uyuşmazlığı olan kol çiftleri:", raw_mismatch_count,
      "; gerçek kenar uyuşmazlığı olan kol çiftleri:", edge_mismatch_count)
if not table.empty and table["mixed_hardware"].any():
    print("Farklı donanımlı örnekler uç sonuç karşılaştırmasından çıkarıldı:",
          table.loc[table["mixed_hardware"], "instance"].tolist())
print("Uç sonuçlar betimleyicidir; eşleşmeyen bariyerlerden politika üstünlüğü çıkarılmadı.")
display(table)
display(endpoint_table.loc[(endpoint_table["treatment"] == "repair_betatour") &
                           (endpoint_table["control"].isin(["native_pop6", "sham", "repair_random", "repair_longest"]))])
display(matched_table.loc[matched_table["treatment"] == "repair_betatour"])


Tamamlanan örnek: 8 / 8
Tüm onarım kollarında aynı bariyer çevrimi: 1 / 8
Ham karma uyuşmazlığı olan kol çiftleri: 50 ; gerçek kenar uyuşmazlığı olan kol çiftleri: 50
Uç sonuçlar betimleyicidir; eşleşmeyen bariyerlerden politika üstünlüğü çıkarılmadı.


,instance,group,n,budget_s,mixed_hardware,barrier_tours_match,native_pop6,sham,repair_random,repair_longest,repair_disagreement,repair_betatour
0,augment_clu10k_000,clu10k,10000,120.0,False,False,33760199,33760199,33760225,33760225,33760225,33760225
1,augment_clu10k_001,clu10k,10000,120.0,False,False,16201186,16201186,16201903,16201903,16201903,16201903
2,augment_uni10k_000,uni10k,10000,120.0,False,False,72087781,72087781,72090390,72090390,72090390,72090390
3,augment_uni10k_001,uni10k,10000,120.0,False,False,71648570,71648570,71654927,71654927,71654927,71654927
4,augment_clu20k_000,clu20k,20000,300.0,False,True,58587608,58587608,58587635,58587635,58587635,58587635
5,augment_clu20k_001,clu20k,20000,300.0,False,False,51650231,51650332,51650332,51650332,51650332,51650332
6,augment_uni20k_000,uni20k,20000,300.0,False,False,101470982,101470982,101470982,101470982,101470982,101470982
7,augment_uni20k_001,uni20k,20000,300.0,False,False,101207058,101207058,101207109,101207109,101208056,101207109


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct,exploratory_sign_p
0,repair_betatour,native_pop6,same-instance endpoint; barrier may differ,8,0,7,1,0.002161,0.000136,0.015625
1,repair_betatour,sham,same-instance endpoint; barrier may differ,8,0,6,2,0.002136,0.000064,0.031250
2,repair_betatour,repair_random,same-instance endpoint; barrier may differ,8,0,0,8,0.000000,0.000000,1.000000
3,repair_betatour,repair_longest,same-instance endpoint; barrier may differ,8,0,0,8,0.000000,0.000000,1.000000


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct,exploratory_sign_p
0,repair_betatour,sham,all six saved barrier cycles match,4,0,3,1,0.003336,0.002236,0.25
1,repair_betatour,repair_random,all six saved barrier cycles match,5,0,0,5,0.000000,0.000000,1.00
2,repair_betatour,repair_longest,all six saved barrier cycles match,3,0,0,3,0.000000,0.000000,1.00
3,repair_betatour,repair_disagreement,all six saved barrier cycles match,3,1,0,2,-0.000312,0.000000,1.00


In [ ]:
def checked_zip(name, include_audit):
    local = WORK_ROOT / name
    temp = WORK_ROOT / (name + ".tmp")
    files = []
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix == ".zip" or path.name.endswith(".tmp") or path.name.endswith(".tmp.npz"):
                continue
            rel = path.relative_to(OUTPUT_ROOT)
            if not include_audit and ("policy_audit" in rel.parts or path.name in
                                      {"xy.npz", "native.candidates", "live.candidates", "post.candidates", "first.tour", "barrier.tour", "best.tour", "repair.tour", "stdout.txt", "stderr.txt", "policy_arrays.npz"}):
                continue
            archive.write(path, str(rel), compress_type=zipfile.ZIP_STORED if path.suffix == ".npz" else zipfile.ZIP_DEFLATED)
            files.append(str(rel))
    with zipfile.ZipFile(temp) as check:
        assert check.testzip() is None and len(check.namelist()) == len(files)
    os.replace(temp, local)
    target = OUTPUT_ROOT / "archives" / name
    tmp_target = target.with_name(target.name + ".tmp")
    shutil.copy2(local, tmp_target)
    os.replace(tmp_target, target)
    assert sha256_file(local) == sha256_file(target)
    return {"local": str(local), "drive": str(target), "sha256": sha256_file(local),
            "bytes": local.stat().st_size, "files": len(files)}

complete = len(rows) == 8
audit_zip = checked_zip(EXPERIMENT_NAME + "_audit.zip", True) if complete else None
results_zip = checked_zip(EXPERIMENT_NAME + "_results.zip", False)
write_json_atomic(OUTPUT_ROOT / "archives" / "archive_info.json",
                  {"complete": complete, "results": results_zip, "audit": audit_zip})
print("Sonuç ZIP'i:", results_zip)
if complete:
    print("Tam denetim ZIP'i Drive'da:", audit_zip)
    from google.colab import files
    files.download(results_zip["local"])
else:
    print(f"Ara sonuçlar Drive'da; tamamlanan örnek {len(rows)}/8. Yeniden çalıştırınca devam eder.")


Sonuç ZIP'i: {'local': '/content/betatour_v10_repair_work/v10_repair_pilot_results.zip', 'drive': '/content/drive/MyDrive/BetaTour/v10_repair_pilot/archives/v10_repair_pilot_results.zip', 'sha256': '0bb5ca73a8543f952f49d7e636ce5a303eda9f7e81851001d0fcdc0dc9cd1ede', 'bytes': 9373505, 'files': 1205}
Tam denetim ZIP'i Drive'da: {'local': '/content/betatour_v10_repair_work/v10_repair_pilot_audit.zip', 'drive': '/content/drive/MyDrive/BetaTour/v10_repair_pilot/archives/v10_repair_pilot_audit.zip', 'sha256': '04d8795a866b8bbb463b69668e8ec4e2abeb36d8832a2feaa7db2c7d2d000415', 'bytes': 330663173, 'files': 3277}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>